# Bronze to Silver Transformations
This notebook is responsible for transforming the raw bronze data to become clean, standardized data. Such transformations include...

    1. Deduplicate every table by primary key
    2. Cast columns to correct types
    3. Trim/standardize strings
    4. Validate primary keys are not null
    5. Validate foreign keys exist
    6. Validate important numeric/date ranges
    7. Validate lookup/enumeration values
    8. Add is_valid_record and dq_error_reason
    9. Keep invalid rows in a separate quarantine table or flag them


### Future To-Do List:
- Study & double-check all programming
- Add better documentation
- Compartmentalize each data transformation phase into their own notebook for better debugging, then use Lakeflow to orchestrate altogether.
- Reimplement addinng record-level data quality tags and quaratine record operations

In [0]:
# import library
import re
from pyspark.sql import Row
from pyspark.sql.functions import col, when, count, sum as spark_sum
from pyspark.sql.types import StructType, StructField, StringType

primary_key_map = {
    "business": ["business_id"],
    "customer": ["customer_id"],
    "department_type": ["department_type_id"],
    "departments": ["department_id"],
    "email_queue": ["email_queue_id"],
    "employee": ["employee_id"],
    "facility": ["facility_id"],
    "facility_type": ["facility_type_id"],
    "facility_type_department_rule": ["facility_type_id", "department_type_id"],
    "incident": ["incident_id"],
    "incident_severity": ["incident_severity_id"],
    "incident_status": ["incident_status_id"],
    "incident_type": ["incident_type_id"],
    "lockerassignment": ["locker_assignment_id"],
    "lockerlocations": ["locker_location_id"],
    "notifications": ["notification_id"],
    "package": ["package_id"],
    "package_flow_event_rule": ["package_flow_type_id", "package_movement_event_type_id"],
    "package_flow_type": ["package_flow_type_id"],
    "package_movement": ["package_movement_id"],
    "package_movement_event_type": ["package_movement_event_type_id"],
    "package_movement_field_rule": ["package_movement_event_type_id"],
    "package_movement_staffing_rule": ["package_flow_type_id", "package_movement_event_type_id", "facility_type_id"],
    "package_route_plan": ["package_id"],
    "package_status": ["package_status_id"],
    "package_to_locker": ["package_id"],
    "refunds": ["refund_id"],
    "roles": ["role_id"],
    "service_type": ["service_type_id"],
    "shipping_cost": ["package_id"],
    "shippingdetails": ["package_id"],
    "smartlocker": ["locker_id"],
    "territory": ["territory_id"],
    "user_logins": ["user_id"],
    "user_roles": ["user_role_id"],
    "works_on": ["employee_id", "department_id"],
    "zip_geo": ["zip_code"],
}

# Keep the legacy alias for compatibility with existing validation cells.
pk_map = list(primary_key_map.items())


In [0]:
# ----------------------------
# Source / target schemas
# ----------------------------

catalog = "postal_databricks"

# IMPORTANT:
# This first transform must always read from bronze and write to silver.
# Do not use globals().get() here because Databricks keeps old notebook variables alive.
source_schema = "bronze"
target_schema = "silver"
quarantine_schema = "silver_quarantine"

silver_base_path = None
quarantine_base_path = None

write_quarantine_tables = True
write_empty_quarantine_tables = False

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{source_schema}`
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{target_schema}`
MANAGED LOCATION 'abfss://silver@postalsg.dfs.core.windows.net/postal_bi_system/silver'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{quarantine_schema}`
MANAGED LOCATION 'abfss://silver@postalsg.dfs.core.windows.net/postal_bi_system/silver_quarantine'
""")

DataFrame[]

### Deduplicate every table by primary key

In [0]:
# ============================================================
# Silver Transform #1: Deduplicate Bronze Tables By Primary Key
# Cell 1: Config + helper functions
#
# Purpose:
#   Read every bronze Delta table.
#   Keep exactly 1 row per primary key.
#   Write valid deduplicated rows to the silver schema.
#   Optionally quarantine duplicate losers and NULL-PK rows.
#
# Assumes this notebook already has either:
#   - primary_key_map from the PK validation cells, OR
#   - pk_map from the duplicate validation cell.
# ============================================================

from functools import reduce
from datetime import datetime, timezone

from pyspark import StorageLevel
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col,
    lit,
    row_number,
    count as spark_count,
    xxhash64,
    current_timestamp,
    desc,
    when,
    sum as spark_sum,
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)


# ----------------------------
# Reuse the canonical PK map
# ----------------------------

def get_primary_key_map_for_silver():
    """
    Reuses the notebook's canonical primary_key_map.
    A legacy pk_map alias is still supported for compatibility.
    """
    if "primary_key_map" in globals() and isinstance(primary_key_map, dict):
        return primary_key_map

    if "pk_map" in globals():
        if isinstance(pk_map, dict):
            return pk_map

        return {
            table_name: pk_columns
            for table_name, pk_columns in pk_map
        }

    raise ValueError(
        "No primary_key_map found. Run the canonical PK map cell first."
    )


dedupe_primary_key_map = get_primary_key_map_for_silver()


# ----------------------------
# General helpers
# ----------------------------

def quote_full_table(catalog_name: str, schema_name: str, table_name: str) -> str:
    return f"`{catalog_name}`.`{schema_name}`.`{table_name}`"


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


def resolve_column_name(df: DataFrame, requested_column_name: str):
    """
    Case-insensitive column resolver.
    This protects you if Databricks/Spark preserves casing differently than MySQL.
    """
    column_lookup = {
        existing_column.lower(): existing_column
        for existing_column in df.columns
    }

    return column_lookup.get(requested_column_name.lower())


def write_delta_table(
    df: DataFrame,
    catalog_name: str,
    schema_name: str,
    table_name: str,
    base_path: str = None,
    mode: str = "overwrite",
):
    """
    Writes a DataFrame as a Delta table.

    If base_path is provided, the table is written to:
        base_path/table_name

    If base_path is None, Databricks uses the schema's default storage location.
    """
    target_table = quote_full_table(catalog_name, schema_name, table_name)

    writer = (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
    )

    if base_path:
        table_path = f"{base_path.rstrip('/')}/{table_name}"
        writer = writer.option("path", table_path)

    writer.saveAsTable(target_table)


def and_all(conditions):
    """
    Safely ANDs a list of Spark Column conditions.
    """
    return reduce(lambda left, right: left & right, conditions)


In [0]:
print("catalog =", catalog)
print("source_schema =", source_schema)
print("target_schema =", target_schema)

if source_schema != "bronze":
    raise ValueError("STOP: source_schema must be bronze before writing canonical silver tables.")

catalog = postal_databricks
source_schema = bronze
target_schema = silver


In [0]:
# ============================================================
# Silver Transform #1
# Cell 2: Deduplication logic for one table
# ============================================================

# These columns are used only to choose the "best" row when duplicate PKs exist.
# The first matching columns found in a table will be used in this order.
# If none exist, the row hash tie-breaker is still used.
DEDUPLICATION_ORDER_PREFERENCE = [
    "updated_at",
    "created_at",
    "event_timestamp",
    "received_date",
    "incident_date",
    "refund_date",
    "reviewed_at",
    "paid_at",
    "charge_recorded_at",
    "selected_at",
    "assigned_at",
    "expires_at",
    "retrieved_at",
    "notification_date",
    "manager_start_date",
]


def deduplicate_table_by_primary_key(
    df: DataFrame,
    table_name: str,
    pk_columns: list,
):
    """
    Returns:
      deduped_df:
        One valid row per non-null primary key.

      duplicate_reject_df:
        Extra rows from duplicate PK groups.

      null_pk_reject_df:
        Rows where one or more PK columns are NULL.

      resolved_pk_columns:
        Actual DataFrame column names for the PK columns.
    """

    resolved_pk_columns = []
    missing_pk_columns = []

    for pk_column in pk_columns:
        actual_column = resolve_column_name(df, pk_column)

        if actual_column is None:
            missing_pk_columns.append(pk_column)
        else:
            resolved_pk_columns.append(actual_column)

    if missing_pk_columns:
        raise ValueError(
            f"{table_name} is missing expected PK column(s): {missing_pk_columns}"
        )

    source_columns = df.columns

    pk_not_null_condition = and_all([
        col(f"`{pk_column}`").isNotNull()
        for pk_column in resolved_pk_columns
    ])

    valid_pk_df = df.filter(pk_not_null_condition)
    null_pk_reject_df = (
        df
        .filter(~pk_not_null_condition)
        .withColumn("_quarantine_reason", lit("NULL_PRIMARY_KEY"))
        .withColumn("_source_table", lit(table_name))
        .withColumn("_primary_key_columns", lit(", ".join(resolved_pk_columns)))
        .withColumn("_quarantined_at", current_timestamp())
    )

    # Add row hash as a deterministic final tie-breaker.
    # If two rows are truly identical, either row is equivalent.
    valid_pk_with_hash_df = valid_pk_df.withColumn(
        "_silver_row_hash",
        xxhash64(*[
            col(f"`{column_name}`")
            for column_name in source_columns
        ])
    )

    order_columns = []

    for preferred_column in DEDUPLICATION_ORDER_PREFERENCE:
        actual_column = resolve_column_name(valid_pk_with_hash_df, preferred_column)

        if actual_column is not None:
            order_columns.append(col(f"`{actual_column}`").desc_nulls_last())

    order_columns.append(col("_silver_row_hash").desc_nulls_last())

    pk_partition_columns = [
        col(f"`{pk_column}`")
        for pk_column in resolved_pk_columns
    ]

    pk_window = Window.partitionBy(*pk_partition_columns)
    ranked_window = pk_window.orderBy(*order_columns)

    ranked_df = (
        valid_pk_with_hash_df
        .withColumn("_silver_duplicate_group_count", spark_count(lit(1)).over(pk_window))
        .withColumn("_silver_dedupe_rank", row_number().over(ranked_window))
    )

    deduped_df = (
        ranked_df
        .filter(col("_silver_dedupe_rank") == 1)
        .select(*[
            col(f"`{column_name}`")
            for column_name in source_columns
        ])
    )

    duplicate_reject_df = (
        ranked_df
        .filter(col("_silver_dedupe_rank") > 1)
        .withColumn("_quarantine_reason", lit("DUPLICATE_PRIMARY_KEY_LOSER"))
        .withColumn("_source_table", lit(table_name))
        .withColumn("_primary_key_columns", lit(", ".join(resolved_pk_columns)))
        .withColumn("_quarantined_at", current_timestamp())
    )

    return deduped_df, duplicate_reject_df, null_pk_reject_df, resolved_pk_columns

In [0]:
# ============================================================
# Silver Transform #1
# Cell 3: Deduplicate all bronze tables and write canonical silver tables
# ============================================================

dedupe_run_started_at = datetime.now(timezone.utc).isoformat()

dedupe_summary_rows = []

for table_name, pk_columns in dedupe_primary_key_map.items():
    source_table = quote_full_table(catalog, source_schema, table_name)
    target_table = quote_full_table(catalog, target_schema, table_name)

    if not table_exists(source_table):
        dedupe_summary_rows.append({
            "run_started_at": dedupe_run_started_at,
            "table_name": table_name,
            "source_table": source_table,
            "target_table": target_table,
            "primary_key_columns": ", ".join(pk_columns),
            "source_rows": None,
            "silver_rows": None,
            "null_pk_rows_quarantined": None,
            "duplicate_rows_quarantined": None,
            "rows_removed_from_silver": None,
            "status": "SOURCE_TABLE_MISSING",
            "message": f"Skipped because {source_table} does not exist.",
        })
        continue

    try:
        bronze_df = spark.table(source_table)

        deduped_df, duplicate_reject_df, null_pk_reject_df, resolved_pk_columns = (
            deduplicate_table_by_primary_key(
                df=bronze_df,
                table_name=table_name,
                pk_columns=pk_columns,
            )
        )

        # Note: .persist() is not supported on serverless compute.
        # Spark Connect automatically manages caching and optimization.

        source_rows = bronze_df.count()
        silver_rows = deduped_df.count()
        duplicate_rows_quarantined = duplicate_reject_df.count()
        null_pk_rows_quarantined = null_pk_reject_df.count()
        rows_removed_from_silver = source_rows - silver_rows

        write_delta_table(
            df=deduped_df,
            catalog_name=catalog,
            schema_name=target_schema,
            table_name=table_name,
            base_path=silver_base_path,
            mode="overwrite",
        )

        if write_quarantine_tables:
            if duplicate_rows_quarantined > 0 or write_empty_quarantine_tables:
                write_delta_table(
                    df=duplicate_reject_df,
                    catalog_name=catalog,
                    schema_name=quarantine_schema,
                    table_name=f"{table_name}__duplicate_pk_rows",
                    base_path=quarantine_base_path,
                    mode="overwrite",
                )

            if null_pk_rows_quarantined > 0 or write_empty_quarantine_tables:
                write_delta_table(
                    df=null_pk_reject_df,
                    catalog_name=catalog,
                    schema_name=quarantine_schema,
                    table_name=f"{table_name}__null_pk_rows",
                    base_path=quarantine_base_path,
                    mode="overwrite",
                )

        dedupe_summary_rows.append({
            "run_started_at": dedupe_run_started_at,
            "table_name": table_name,
            "source_table": source_table,
            "target_table": target_table,
            "primary_key_columns": ", ".join(resolved_pk_columns),
            "source_rows": source_rows,
            "silver_rows": silver_rows,
            "null_pk_rows_quarantined": null_pk_rows_quarantined,
            "duplicate_rows_quarantined": duplicate_rows_quarantined,
            "rows_removed_from_silver": rows_removed_from_silver,
            "status": "OK",
            "message": "Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.",
        })

    except Exception as error:
        dedupe_summary_rows.append({
            "run_started_at": dedupe_run_started_at,
            "table_name": table_name,
            "source_table": source_table,
            "target_table": target_table,
            "primary_key_columns": ", ".join(pk_columns),
            "source_rows": None,
            "silver_rows": None,
            "null_pk_rows_quarantined": None,
            "duplicate_rows_quarantined": None,
            "rows_removed_from_silver": None,
            "status": "FAILED",
            "message": str(error),
        })


dedupe_summary_schema = StructType([
    StructField("run_started_at", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("primary_key_columns", StringType(), True),
    StructField("source_rows", LongType(), True),
    StructField("silver_rows", LongType(), True),
    StructField("null_pk_rows_quarantined", LongType(), True),
    StructField("duplicate_rows_quarantined", LongType(), True),
    StructField("rows_removed_from_silver", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
])

silver_dedupe_summary_df = spark.createDataFrame(
    dedupe_summary_rows,
    schema=dedupe_summary_schema,
)

write_delta_table(
    df=silver_dedupe_summary_df,
    catalog_name=catalog,
    schema_name=target_schema,
    table_name="_silver_dedupe_run_summary",
    base_path=silver_base_path,
    mode="overwrite",
)

display(
    silver_dedupe_summary_df
    .orderBy(
        col("status").desc(),
        col("rows_removed_from_silver").desc_nulls_last(),
        col("table_name"),
    )
)


run_started_at,table_name,source_table,target_table,primary_key_columns,source_rows,silver_rows,null_pk_rows_quarantined,duplicate_rows_quarantined,rows_removed_from_silver,status,message
2026-06-22T04:28:50.161745+00:00,business,`postal_databricks`.`bronze`.`business`,`postal_databricks`.`silver`.`business`,business_id,3095,3095,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,customer,`postal_databricks`.`bronze`.`customer`,`postal_databricks`.`silver`.`customer`,customer_id,96096,96096,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,department_type,`postal_databricks`.`bronze`.`department_type`,`postal_databricks`.`silver`.`department_type`,department_type_id,6,6,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,departments,`postal_databricks`.`bronze`.`departments`,`postal_databricks`.`silver`.`departments`,department_id,1596,1596,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,email_queue,`postal_databricks`.`bronze`.`email_queue`,`postal_databricks`.`silver`.`email_queue`,email_queue_id,0,0,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,employee,`postal_databricks`.`bronze`.`employee`,`postal_databricks`.`silver`.`employee`,employee_id,5000,5000,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,facility,`postal_databricks`.`bronze`.`facility`,`postal_databricks`.`silver`.`facility`,facility_id,541,541,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,facility_type,`postal_databricks`.`bronze`.`facility_type`,`postal_databricks`.`silver`.`facility_type`,facility_type_id,5,5,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,facility_type_department_rule,`postal_databricks`.`bronze`.`facility_type_department_rule`,`postal_databricks`.`silver`.`facility_type_department_rule`,"facility_type_id, department_type_id",13,13,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.
2026-06-22T04:28:50.161745+00:00,incident,`postal_databricks`.`bronze`.`incident`,`postal_databricks`.`silver`.`incident`,incident_id,6286,6286,0,0,0,OK,Canonical silver written after dedupe. Duplicate-loser and NULL-PK rejects were quarantined instead of silently discarded.


In [0]:
# ============================================================
# Silver Transform #1
# Cell 4: Verify deduplicated silver tables
# ============================================================

silver_pk_verification_rows = []

for table_name, pk_columns in dedupe_primary_key_map.items():
    silver_table = quote_full_table(catalog, target_schema, table_name)

    if not table_exists(silver_table):
        silver_pk_verification_rows.append({
            "table_name": table_name,
            "primary_key_columns": ", ".join(pk_columns),
            "silver_rows": None,
            "null_pk_rows": None,
            "duplicate_key_groups": None,
            "duplicate_extra_rows": None,
            "status": "SILVER_TABLE_MISSING",
        })
        continue

    silver_df = spark.table(silver_table)

    resolved_pk_columns = []
    missing_pk_columns = []

    for pk_column in pk_columns:
        actual_column = resolve_column_name(silver_df, pk_column)

        if actual_column is None:
            missing_pk_columns.append(pk_column)
        else:
            resolved_pk_columns.append(actual_column)

    if missing_pk_columns:
        silver_pk_verification_rows.append({
            "table_name": table_name,
            "primary_key_columns": ", ".join(pk_columns),
            "silver_rows": silver_df.count(),
            "null_pk_rows": None,
            "duplicate_key_groups": None,
            "duplicate_extra_rows": None,
            "status": f"MISSING_PK_COLUMNS: {missing_pk_columns}",
        })
        continue

    pk_select = ", ".join([
        f"`{pk_column}`"
        for pk_column in resolved_pk_columns
    ])

    null_condition = " OR ".join([
        f"`{pk_column}` IS NULL"
        for pk_column in resolved_pk_columns
    ])

    verification_query = f"""
        SELECT
            '{table_name}' AS table_name,
            '{", ".join(resolved_pk_columns)}' AS primary_key_columns,
            COUNT(*) AS silver_rows,
            COALESCE(SUM(CASE WHEN {null_condition} THEN 1 ELSE 0 END), 0) AS null_pk_rows,
            COALESCE(SUM(CASE WHEN row_count > 1 THEN 1 ELSE 0 END), 0) AS duplicate_key_groups,
            COALESCE(SUM(CASE WHEN row_count > 1 THEN row_count - 1 ELSE 0 END), 0) AS duplicate_extra_rows,
            CASE
                WHEN COALESCE(SUM(CASE WHEN {null_condition} THEN 1 ELSE 0 END), 0) > 0
                    THEN 'FAILED_NULL_PK'
                WHEN COALESCE(SUM(CASE WHEN row_count > 1 THEN row_count - 1 ELSE 0 END), 0) > 0
                    THEN 'FAILED_DUPLICATE_PK'
                ELSE 'OK'
            END AS status
        FROM (
            SELECT
                {pk_select},
                COUNT(*) AS row_count
            FROM {silver_table}
            GROUP BY {pk_select}
        ) grouped_pk
    """

    silver_pk_verification_rows.append(
        spark.sql(verification_query).collect()[0].asDict()
    )


silver_pk_verification_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("primary_key_columns", StringType(), True),
    StructField("silver_rows", LongType(), True),
    StructField("null_pk_rows", LongType(), True),
    StructField("duplicate_key_groups", LongType(), True),
    StructField("duplicate_extra_rows", LongType(), True),
    StructField("status", StringType(), True),
])

silver_pk_verification_df = spark.createDataFrame(
    silver_pk_verification_rows,
    schema=silver_pk_verification_schema,
)

display(
    silver_pk_verification_df
    .orderBy(
        col("status").desc(),
        col("duplicate_extra_rows").desc_nulls_last(),
        col("null_pk_rows").desc_nulls_last(),
        col("table_name"),
    )
)


table_name,primary_key_columns,silver_rows,null_pk_rows,duplicate_key_groups,duplicate_extra_rows,status
business,business_id,3095,0,0,0,OK
customer,customer_id,96096,0,0,0,OK
department_type,department_type_id,6,0,0,0,OK
departments,department_id,1596,0,0,0,OK
email_queue,email_queue_id,0,0,0,0,OK
employee,employee_id,5000,0,0,0,OK
facility,facility_id,541,0,0,0,OK
facility_type,facility_type_id,5,0,0,0,OK
facility_type_department_rule,"facility_type_id, department_type_id",13,0,0,0,OK
incident,incident_id,6286,0,0,0,OK


In [0]:
%python
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, desc

catalog = "postal_databricks"
schema = "bronze"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

pk_map = list(primary_key_map.items())

results = []

for table_name, pk_cols in pk_map:
    pk_select = ", ".join([f"`{c}`" for c in pk_cols])
    null_condition = " OR ".join([f"`{c}` IS NULL" for c in pk_cols])
    pk_label = ", ".join(pk_cols)

    query = f"""
        SELECT
            '{table_name}' AS table_name,
            '{pk_label}' AS primary_key_columns,
            COALESCE(SUM(CASE WHEN {null_condition} THEN row_count ELSE 0 END), 0) AS null_pk_rows,
            COALESCE(SUM(CASE WHEN NOT ({null_condition}) AND row_count > 1 THEN 1 ELSE 0 END), 0) AS duplicate_key_groups,
            COALESCE(SUM(CASE WHEN NOT ({null_condition}) AND row_count > 1 THEN row_count - 1 ELSE 0 END), 0) AS duplicate_extra_rows
        FROM (
            SELECT {pk_select}, COUNT(*) AS row_count
            FROM `{catalog}`.`{schema}`.`{table_name}`
            GROUP BY {pk_select}
        ) g
    """

    results.append(spark.sql(query))

duplicate_summary_df = reduce(DataFrame.unionByName, results)

display(
    duplicate_summary_df
    .orderBy(desc("duplicate_extra_rows"), desc("duplicate_key_groups"), desc("null_pk_rows"), col("table_name"))
)


table_name,primary_key_columns,null_pk_rows,duplicate_key_groups,duplicate_extra_rows
business,business_id,0,0,0
customer,customer_id,0,0,0
department_type,department_type_id,0,0,0
departments,department_id,0,0,0
email_queue,email_queue_id,0,0,0
employee,employee_id,0,0,0
facility,facility_id,0,0,0
facility_type,facility_type_id,0,0,0
facility_type_department_rule,"facility_type_id, department_type_id",0,0,0
incident,incident_id,0,0,0


### Cast columns to correct type
In the actual Azure clloud ecosystem, Databricks can connect to external databases through JDBC or Unity Catalog connections, so you can read MySQL metadata directly from Databricks if the connection is configured.

For this local notebook we'll just use a dictionary.

In [0]:
# ============================================================
# Bronze EDA Check #2: Column Type Validation
# Cell 1: Config + expected MySQL schema dictionary
# ============================================================

catalog = "postal_databricks"
source_schema = "bronze"

expected_mysql_schema = {
    "business": {
        "business_id": "binary(16)",
        "business_name": "varchar(150)",
        "street_address": "varchar(150)",
        "county": "varchar(50)",
        "city": "varchar(50)",
        "state_code": "char(2)",
        "zip_code": "varchar(10)",
        "territory_id": "int",
        "phone_number": "varchar(20)",
        "email": "varchar(100)",
        "created_at": "timestamp",
        "updated_at": "datetime",
        "preferred_facility_id": "int",
    },

    "customer": {
        "customer_id": "binary(16)",
        "first_name": "varchar(50)",
        "middle_initial": "char(1)",
        "last_name": "varchar(50)",
        "street_address": "varchar(100)",
        "county": "varchar(50)",
        "city": "varchar(50)",
        "state_code": "char(2)",
        "zip_code": "varchar(10)",
        "territory_id": "int",
        "phone_number": "varchar(15)",
        "email": "varchar(100)",
        "created_at": "timestamp",
        "updated_at": "datetime",
        "user_id": "int",
        "preferred_facility_id": "int",
        "birth_date": "date",
        "marital_status": "char(1)",
        "gender": "char(1)",
        "email_address": "varchar(150)",
        "annual_income": "decimal(10,2)",
        "total_children": "tinyint unsigned",
        "education_level": "varchar(30)",
        "occupation": "varchar(30)",
        "home_owner": "char(1)",
    },

    "departments": {
        "department_name": "varchar(40)",
        "department_type_id": "int",
        "manager_employee_id": "int",
        "manager_start_date": "datetime",
        "facility_id": "int",
        "department_id": "int",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "department_type": {
        "department_type_id": "int",
        "department_type_code": "varchar(20)",
        "department_type_name": "varchar(50)",
        "description": "varchar(255)",
        "is_active": "tinyint(1)",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "email_queue": {
        "email_queue_id": "int",
        "recipient_email": "varchar(255)",
        "email_subject": "varchar(255)",
        "email_body": "text",
        "sent_status": "tinyint(1)",
        "created_at": "timestamp",
    },

    "employee": {
        "employee_id": "int",
        "department_id": "int",
        "full_name": "varchar(50)",
        "phone_number": "varchar(15)",
        "email": "varchar(100)",
        "street_address": "varchar(100)",
        "job_title": "varchar(50)",
        "salary": "decimal(10,2)",
        "hours_worked": "smallint",
        "manager_employee_id": "int",
        "created_at": "timestamp",
        "updated_at": "datetime",
        "user_id": "int",
    },

    "facility": {
        "facility_id": "int",
        "facility_type_id": "int",
        "manager_employee_id": "int",
        "facility_name": "varchar(100)",
        "street_address": "varchar(120)",
        "county": "varchar(45)",
        "city": "varchar(60)",
        "state_code": "char(2)",
        "zip_code": "varchar(10)",
        "facility_department_prefix": "varchar(30)",
        "territory_id": "int",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "facility_type": {
        "facility_type_id": "int",
        "facility_type_code": "varchar(10)",
        "facility_type_name": "varchar(80)",
        "description": "varchar(255)",
        "is_customer_facing": "tinyint(1)",
        "handles_retail": "tinyint(1)",
        "handles_processing": "tinyint(1)",
        "handles_distribution": "tinyint(1)",
        "handles_local_delivery": "tinyint(1)",
        "is_active": "tinyint(1)",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "facility_type_department_rule": {
        "facility_type_id": "int",
        "department_type_id": "int",
        "created_at": "timestamp",
    },

    "incident": {
        "incident_id": "int",
        "reported_by_employee_id": "int",
        "package_id": "binary(16)",
        "incident_type_id": "int",
        "incident_severity_id": "int",
        "incident_status_id": "int",
        "description": "varchar(255)",
        "incident_date": "datetime",
        "created_at": "timestamp",
        "updated_at": "datetime",
        "resolved_at": "datetime",
        "resolved_by_employee_id": "int",
        "resolution_note": "varchar(255)",
        "facility_id": "int",
        "package_movement_id": "int",
        "customer_id": "binary(16)",
    },

    "incident_severity": {
        "incident_severity_id": "int",
        "severity_name": "varchar(30)",
        "sort_order": "int",
        "is_active": "tinyint(1)",
    },

    "incident_status": {
        "incident_status_id": "int",
        "status_name": "varchar(30)",
        "sort_order": "int",
        "is_closed_status": "tinyint(1)",
        "is_active": "tinyint(1)",
    },

    "incident_type": {
        "incident_type_id": "int",
        "type_name": "varchar(50)",
        "type_category": "varchar(50)",
        "is_active": "tinyint(1)",
        "blocks_package_movement": "tinyint(1)",
    },

    "lockerassignment": {
        "locker_assignment_id": "int",
        "locker_id": "int",
        "assigned_at": "timestamp",
        "expires_at": "datetime",
        "retrieved_at": "datetime",
        "customer_id": "binary(16)",
    },

    "lockerlocations": {
        "locker_location_id": "int",
        "location_name": "varchar(15)",
        "facility_id": "int",
    },

    "notifications": {
        "notification_id": "int",
        "package_id": "binary(16)",
        "notification_message": "varchar(100)",
        "notification_date": "datetime",
        "customer_id": "binary(16)",
    },

    "package": {
        "package_id": "binary(16)",
        "package_status_id": "int",
        "service_type_id": "int",
        "received_date": "datetime",
        "contents": "varchar(30)",
        "weight_oz": "decimal(8,2)",
        "length_in": "decimal(8,2)",
        "width_in": "decimal(8,2)",
        "height_in": "decimal(8,2)",
        "created_at": "timestamp",
        "updated_at": "datetime",
        "recipient_customer_id": "binary(16)",
        "package_flow_type_id": "int",
        "sender_customer_id": "binary(16)",
        "sender_business_id": "binary(16)",
    },

    "package_flow_event_rule": {
        "package_flow_type_id": "int",
        "package_movement_event_type_id": "int",
        "is_allowed": "tinyint(1)",
        "created_at": "timestamp",
        "updated_at": "timestamp",
    },

    "package_flow_type": {
        "package_flow_type_id": "int",
        "package_flow_type_name": "varchar(30)",
        "is_active": "tinyint(1)",
    },

    "package_movement": {
        "package_movement_id": "int",
        "package_id": "binary(16)",
        "package_movement_event_type_id": "int",
        "package_status_id": "int",
        "facility_id": "int",
        "from_facility_id": "int",
        "to_facility_id": "int",
        "processed_by_employee_id": "int",
        "event_timestamp": "datetime",
        "movement_note": "varchar(500)",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "package_movement_event_type": {
        "package_movement_event_type_id": "int",
        "event_type_name": "varchar(80)",
        "description": "varchar(255)",
        "default_package_status_name": "varchar(30)",
        "is_entry_event": "tinyint(1)",
        "is_exit_event": "tinyint(1)",
        "is_processing_event": "tinyint(1)",
        "is_final_event": "tinyint(1)",
        "sort_order": "int",
        "is_active": "tinyint(1)",
    },

    "package_movement_field_rule": {
        "package_movement_event_type_id": "int",
        "requires_facility_id": "tinyint(1)",
        "requires_from_facility_id": "tinyint(1)",
        "requires_to_facility_id": "tinyint(1)",
        "allows_from_facility_id": "tinyint(1)",
        "allows_to_facility_id": "tinyint(1)",
        "facility_id_must_equal_from_facility_id": "tinyint(1)",
        "facility_id_must_equal_to_facility_id": "tinyint(1)",
        "created_at": "timestamp",
        "updated_at": "timestamp",
    },

    "package_movement_staffing_rule": {
        "package_flow_type_id": "int",
        "package_movement_event_type_id": "int",
        "facility_type_id": "int",
        "required_department_type_id": "int",
        "requires_employee": "tinyint(1)",
        "created_at": "timestamp",
        "updated_at": "timestamp",
    },

    "package_route_plan": {
        "package_id": "binary(16)",
        "planned_origin_facility_id": "int",
        "planned_destination_facility_id": "int",
        "selection_source": "varchar(80)",
        "selected_at": "datetime",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "package_status": {
        "package_status_id": "int",
        "status_name": "varchar(30)",
        "status_category": "varchar(30)",
        "sort_order": "int",
        "is_final_status": "tinyint(1)",
        "is_active": "tinyint(1)",
        "allowed_service_type_id": "int",
    },

    "package_to_locker": {
        "locker_assignment_id": "int",
        "package_id": "binary(16)",
        "assigned_at": "datetime",
        "customer_id": "binary(16)",
    },

    "refunds": {
        "refund_id": "int",
        "package_id": "binary(16)",
        "incident_id": "int",
        "refund_amount": "decimal(8,2)",
        "refund_reason": "enum('Late Delivery','Damaged Package','Lost Package','Returned Package','Cancelled Package','Service Failure','Goodwill Adjustment','Other')",
        "refund_date": "datetime",
        "refund_status": "enum('Pending','Approved','Rejected','Paid','Cancelled')",
        "created_at": "timestamp",
        "updated_at": "datetime",
        "reviewed_at": "datetime",
        "paid_at": "datetime",
        "refund_note": "varchar(255)",
        "customer_id": "binary(16)",
        "reviewed_by_employee_id": "int",
    },

    "roles": {
        "role_id": "int",
        "role_name": "varchar(15)",
    },

    "service_type": {
        "service_type_id": "int",
        "service_type_name": "varchar(30)",
        "is_active": "tinyint(1)",
    },

    "shippingdetails": {
        "package_id": "binary(16)",
        "recipient_address": "varchar(150)",
        "recipient_territory_id": "int",
        "sender_address": "varchar(150)",
        "sender_territory_id": "int",
        "estimated_delivery_distance": "decimal(10,2)",
        "recipient_first_name": "varchar(20)",
        "recipient_middle_initial": "char(1)",
        "recipient_last_name": "varchar(20)",
        "expected_delivery_date": "datetime",
        "delivered_date": "datetime",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "shipping_cost": {
        "package_id": "binary(16)",
        "actual_shipping_charge": "decimal(8,2)",
        "material_cost": "decimal(12,2)",
        "transportation_cost": "decimal(12,2)",
        "charge_source": "varchar(50)",
        "charge_recorded_at": "datetime",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "smartlocker": {
        "locker_id": "int",
        "locker_location_id": "int",
        "locker_status": "enum('Available','Occupied','Out of service')",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "territory": {
        "territory_id": "int",
        "state": "char(2)",
        "city": "varchar(60)",
        "county": "varchar(60)",
        "zip_code": "char(5)",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "user_logins": {
        "user_id": "int",
        "username": "varchar(45)",
        "password_hash": "varchar(256)",
        "email": "varchar(50)",
        "is_active": "tinyint",
        "first_name": "varchar(45)",
        "last_name": "varchar(45)",
    },

    "user_roles": {
        "user_role_id": "int",
        "role_id": "int",
        "user_id": "int",
    },

    "works_on": {
        "hours_worked": "int",
        "employee_id": "int",
        "department_id": "int",
        "created_at": "timestamp",
        "updated_at": "datetime",
    },

    "zip_geo": {
        "zip_code": "varchar(10)",
        "latitude": "decimal(10,6)",
        "longitude": "decimal(10,6)",
        "created_at": "timestamp",
        "updated_at": "timestamp",
    },
}

In [0]:
# ============================================================
# Silver Transform #2: Cast Columns To Correct Types
# Cell 1: Config + schema-source function
#
# Current notebook behavior:
#   Uses expected_mysql_schema as the reference schema.
#
# Future Azure Databricks behavior:
#   Replace get_schema_spec_for_casting() with logic that reads
#   MySQL metadata directly from INFORMATION_SCHEMA.COLUMNS.
#
# Recommended run order:
#   bronze -> deduplicate by PK -> typed silver
# ============================================================

from datetime import datetime, timezone
import re
import uuid

from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col,
    lit,
    trim,
    lower,
    length,
    regexp_replace,
    unhex,
    when,
    sum as spark_sum,
    count as spark_count,
    current_timestamp,
    expr,
    concat_ws,
)

from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
    StringType,
    DateType,
    TimestampType,
    StructType,
    StructField,
)


# After the previous dedupe step, your clean input should usually be silver.
# This intentionally defaults to target_schema from the dedupe cells.
type_cast_source_schema = globals().get(
    "type_cast_source_schema",
    globals().get("target_schema", "silver")
)

# Write typed tables back to silver by default.
# If you want a safer staged output first, set:
#   type_cast_target_schema = "silver_typed"
type_cast_target_schema = globals().get(
    "type_cast_target_schema",
    globals().get("target_schema", "silver")
)

type_cast_quarantine_schema = globals().get(
    "type_cast_quarantine_schema",
    "silver_quarantine"
)

# Optional external paths.
# Leave as None to use the schema/table default location.
type_cast_base_path = globals().get("type_cast_base_path", globals().get("silver_base_path", None))
type_cast_quarantine_base_path = globals().get(
    "type_cast_quarantine_base_path",
    globals().get("quarantine_base_path", None)
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{type_cast_target_schema}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{type_cast_quarantine_schema}`")


def get_schema_spec_for_casting():
    """
    Return the schema dictionary used for type casting.

    expected_mysql_schema should be loaded from MySQL INFORMATION_SCHEMA.COLUMNS
    in the previous metadata cell.
    """
    schema_spec = globals().get("expected_mysql_schema")

    if not schema_spec:
        raise ValueError(
            "expected_mysql_schema was not loaded. Run the MySQL INFORMATION_SCHEMA "
            "metadata cell before this casting cell."
        )

    return schema_spec


schema_spec_for_casting = get_schema_spec_for_casting()
DQ_CAST_ERROR_FIELD = "dq_cast_error_reason"


def quote_identifier(identifier: str) -> str:
    return "`" + identifier.replace("`", "``") + "`"


def quote_full_table(catalog_name: str, schema_name: str, table_name: str) -> str:
    return (
        f"{quote_identifier(catalog_name)}."
        f"{quote_identifier(schema_name)}."
        f"{quote_identifier(table_name)}"
    )


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


def resolve_column_name(df: DataFrame, requested_column_name: str):
    """
    Case-insensitive column resolver.
    """
    lookup = {existing_column.lower(): existing_column for existing_column in df.columns}
    return lookup.get(requested_column_name.lower())


def normalize_mysql_type(mysql_type: str) -> str:
    return mysql_type.lower().strip().replace("`", "")


In [0]:
# ============================================================
# Silver Transform #2
# Cell 2: Type mapping + safe cast expressions
# ============================================================

# MySQL TINYINT(1) is often used like a boolean, but it is still a numeric MySQL type.
# Keep this False unless you intentionally want tinyint(1) columns turned into booleans.
TINYINT_1_AS_BOOLEAN = False


def parse_decimal_precision_scale(mysql_type: str):
    match = re.search(r"decimal\s*\((\d+)\s*,\s*(\d+)\)", mysql_type.lower())

    if match:
        return int(match.group(1)), int(match.group(2))

    # Safe default if MySQL metadata only says DECIMAL.
    return 38, 18


def parse_binary_length(mysql_type: str):
    match = re.search(r"binary\s*\((\d+)\)", mysql_type.lower())

    if match:
        return int(match.group(1))

    return None


def spark_type_sql_for_mysql_type(mysql_type: str) -> str:
    """
    Returns a SQL type string usable inside try_cast(... AS <type>).
    """
    t = normalize_mysql_type(mysql_type)

    if t.startswith("binary"):
        return "BINARY"

    if t.startswith("char") or t.startswith("varchar") or t == "text" or t.startswith("enum"):
        return "STRING"

    if t.startswith("date") and not t.startswith("datetime"):
        return "DATE"

    if t.startswith("datetime") or t.startswith("timestamp"):
        return "TIMESTAMP"

    if t.startswith("tinyint"):
        if TINYINT_1_AS_BOOLEAN and re.search(r"tinyint\s*\(\s*1\s*\)", t):
            return "BOOLEAN"
        return "TINYINT"

    if t.startswith("smallint"):
        return "SMALLINT"

    if t.startswith("int"):
        return "INT"

    if t.startswith("bigint"):
        return "BIGINT"

    if t.startswith("decimal") or t.startswith("numeric"):
        precision, scale = parse_decimal_precision_scale(t)
        return f"DECIMAL({precision},{scale})"

    if t.startswith("float"):
        return "FLOAT"

    if t.startswith("double"):
        return "DOUBLE"

    # Spark does not have a normal standalone TIME type.
    # Keep MySQL TIME as STRING unless you later choose a duration convention.
    if t.startswith("time"):
        return "STRING"

    if t.startswith("year"):
        return "INT"

    return "STRING"


def clean_numeric_sql(column_name: str) -> str:
    """
    SQL expression that turns '$1,234.56' into '1234.56' before numeric casting.
    This is intentionally light cleaning only.
    """
    q_col = quote_identifier(column_name)
    return f"NULLIF(trim(regexp_replace(cast({q_col} AS STRING), '[$,]', '')), '')"


def safe_try_cast_sql(column_name: str, target_sql_type: str) -> str:
    q_col = quote_identifier(column_name)
    return f"try_cast({q_col} AS {target_sql_type})"


def build_binary_cast_expression(column_name: str, mysql_type: str):
    """
    Handles MySQL BINARY(n), especially BINARY(16) UUID-style keys.

    Accepted inputs:
      - already-binary values of correct length
      - 32-character hex strings for BINARY(16)
      - UUID strings like xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx
      - optional 0x prefix

    Invalid non-null values become NULL and are counted as cast failures.
    """
    expected_length = parse_binary_length(mysql_type)
    source_col = col(quote_identifier(column_name))

    source_as_string = trim(source_col.cast("string"))
    cleaned_hex = regexp_replace(
        regexp_replace(source_as_string, "(?i)^0x", ""),
        "-",
        ""
    )

    hex_is_valid = cleaned_hex.rlike("^[0-9A-Fa-f]+$")

    if expected_length is not None:
        valid_hex_length = length(cleaned_hex) == lit(expected_length * 2)
        valid_binary_length = length(source_col) == lit(expected_length)

        return (
            when(source_col.isNull(), lit(None).cast(BinaryType()))
            .when(valid_binary_length, source_col.cast(BinaryType()))
            .when(hex_is_valid & valid_hex_length, unhex(cleaned_hex))
            .otherwise(lit(None).cast(BinaryType()))
        )

    return (
        when(source_col.isNull(), lit(None).cast(BinaryType()))
        .when(hex_is_valid, unhex(cleaned_hex))
        .otherwise(source_col.cast(BinaryType()))
    )


def build_boolean_cast_expression(column_name: str):
    source_string = lower(trim(col(quote_identifier(column_name)).cast("string")))

    return (
        when(col(quote_identifier(column_name)).isNull(), lit(None).cast(BooleanType()))
        .when(source_string.isin("1", "true", "t", "yes", "y"), lit(True))
        .when(source_string.isin("0", "false", "f", "no", "n"), lit(False))
        .otherwise(expr(safe_try_cast_sql(column_name, "BOOLEAN")))
    )


def build_cast_expression(column_name: str, mysql_type: str):
    """
    Builds a safe Spark expression that casts a column to the Spark equivalent
    of its expected MySQL type.

    Invalid values become NULL instead of crashing the notebook.
    The next cell records how many non-null values failed to cast.
    """
    t = normalize_mysql_type(mysql_type)
    target_sql_type = spark_type_sql_for_mysql_type(mysql_type)

    if t.startswith("binary"):
        return build_binary_cast_expression(column_name, mysql_type), target_sql_type

    if target_sql_type == "BOOLEAN":
        return build_boolean_cast_expression(column_name), target_sql_type

    if target_sql_type.startswith("DECIMAL"):
        return expr(f"try_cast({clean_numeric_sql(column_name)} AS {target_sql_type})"), target_sql_type

    if target_sql_type in {"TINYINT", "SMALLINT", "INT", "BIGINT", "FLOAT", "DOUBLE"}:
        return expr(f"try_cast({clean_numeric_sql(column_name)} AS {target_sql_type})"), target_sql_type

    # DATE / TIMESTAMP / STRING are handled by Spark try_cast.
    return expr(safe_try_cast_sql(column_name, target_sql_type)), target_sql_type

In [0]:
# ============================================================
# Silver Transform #2
# Cell 3: Cast one table using expected MySQL schema
# ============================================================

def cast_table_to_expected_types(
    df: DataFrame,
    table_name: str,
    expected_columns: dict,
):
    """
    Returns:
      casted_df:
        Same table with expected columns cast to their correct Spark type.

      metric_rows:
        One audit row per expected column.
    """
    if DQ_CAST_ERROR_FIELD in df.columns:
        df = df.drop(DQ_CAST_ERROR_FIELD)

    source_rows = df.count()
    cast_plan = []
    metric_rows = []

    for expected_column_name, expected_mysql_type in expected_columns.items():
        actual_column_name = resolve_column_name(df, expected_column_name)

        if actual_column_name is None:
            metric_rows.append({
                "table_name": table_name,
                "column_name": expected_column_name,
                "expected_mysql_type": expected_mysql_type,
                "target_spark_type": spark_type_sql_for_mysql_type(expected_mysql_type),
                "source_spark_type": None,
                "source_rows": source_rows,
                "source_non_null_rows": None,
                "cast_failure_rows": None,
                "status": "MISSING_COLUMN",
                "message": "Expected column does not exist in source table.",
            })
            continue

        source_field = df.schema[actual_column_name]
        cast_expression, target_spark_type = build_cast_expression(
            actual_column_name,
            expected_mysql_type,
        )

        cast_plan.append({
            "expected_column_name": expected_column_name,
            "actual_column_name": actual_column_name,
            "expected_mysql_type": expected_mysql_type,
            "source_spark_type": source_field.dataType.simpleString(),
            "target_spark_type": target_spark_type,
            "cast_expression": cast_expression,
            "cast_error_reason": f"CAST_FAILURE:{table_name}.{actual_column_name}",
        })

    agg_exprs = [spark_count(lit(1)).alias("_source_rows")]

    for index, plan_item in enumerate(cast_plan):
        actual_column_name = plan_item["actual_column_name"]
        cast_expression = plan_item["cast_expression"]
        source_col = col(quote_identifier(actual_column_name))

        agg_exprs.append(
            spark_sum(
                when(source_col.isNotNull(), lit(1)).otherwise(lit(0))
            ).alias(f"_c{index}_source_non_null_rows")
        )

        agg_exprs.append(
            spark_sum(
                when(source_col.isNotNull() & cast_expression.isNull(), lit(1)).otherwise(lit(0))
            ).alias(f"_c{index}_cast_failure_rows")
        )

    stats = df.agg(*agg_exprs).collect()[0].asDict()
    casted_df = df
    cast_error_reason_exprs = []

    for index, plan_item in enumerate(cast_plan):
        actual_column_name = plan_item["actual_column_name"]
        source_non_null_rows = stats[f"_c{index}_source_non_null_rows"]
        cast_failure_rows = stats[f"_c{index}_cast_failure_rows"]

        casted_df = casted_df.withColumn(
            actual_column_name,
            plan_item["cast_expression"],
        )

        source_col = col(quote_identifier(actual_column_name))
        casted_col = col(quote_identifier(actual_column_name))
        cast_error_reason_exprs.append(
            when(
                source_col.isNotNull() & casted_col.isNull(),
                lit(plan_item["cast_error_reason"])
            ).otherwise(lit(None))
        )

        if cast_failure_rows and cast_failure_rows > 0:
            status = "CAST_FAILURES_FOUND"
            message = "One or more non-null source values became NULL after casting."
        else:
            status = "OK"
            message = "Column cast successfully."

        metric_rows.append({
            "table_name": table_name,
            "column_name": actual_column_name,
            "expected_mysql_type": plan_item["expected_mysql_type"],
            "target_spark_type": plan_item["target_spark_type"],
            "source_spark_type": plan_item["source_spark_type"],
            "source_rows": stats["_source_rows"],
            "source_non_null_rows": source_non_null_rows,
            "cast_failure_rows": cast_failure_rows,
            "status": status,
            "message": message,
        })

    if cast_error_reason_exprs:
        combined_cast_reason = concat_ws("; ", *cast_error_reason_exprs)
        casted_df = casted_df.withColumn(
            DQ_CAST_ERROR_FIELD,
            when(length(combined_cast_reason) > 0, combined_cast_reason).otherwise(lit(None).cast(StringType()))
        )
    else:
        casted_df = casted_df.withColumn(DQ_CAST_ERROR_FIELD, lit(None).cast(StringType()))

    return casted_df, metric_rows


In [0]:
# ============================================================
# Silver Transform #2
# Cell 4: Run casts for every table and write typed Delta tables
# ============================================================

def write_delta_table(
    df: DataFrame,
    catalog_name: str,
    schema_name: str,
    table_name: str,
    base_path: str = None,
    mode: str = "overwrite",
):
    target_table = quote_full_table(catalog_name, schema_name, table_name)

    writer = (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
    )

    if base_path:
        table_path = f"{base_path.rstrip('/')}/{table_name}"
        writer = writer.option("path", table_path)

    writer.saveAsTable(target_table)


def prepare_df_for_same_table_overwrite(df: DataFrame) -> DataFrame:
    """
    Materializes the DataFrame before overwrite so we do not depend on a
    drop/rename swap that could break stable external locations.
    """
    try:
        return df.localCheckpoint(eager=True)
    except Exception:
        materialized_df = df.cache()
        materialized_df.count()
        return materialized_df


def safe_write_casted_table(
    df: DataFrame,
    catalog_name: str,
    source_schema_name: str,
    target_schema_name: str,
    table_name: str,
    base_path: str = None,
):
    source_table = quote_full_table(catalog_name, source_schema_name, table_name)
    target_table = quote_full_table(catalog_name, target_schema_name, table_name)

    same_table = source_table == target_table

    prepared_df = prepare_df_for_same_table_overwrite(df) if same_table else df

    write_delta_table(
        df=prepared_df,
        catalog_name=catalog_name,
        schema_name=target_schema_name,
        table_name=table_name,
        base_path=base_path,
        mode="overwrite",
    )

    if same_table:
        try:
            prepared_df.unpersist()
        except Exception:
            pass


type_cast_run_started_at = datetime.now(timezone.utc).isoformat()

all_type_cast_metric_rows = []

for table_name, expected_columns in schema_spec_for_casting.items():
    source_table = quote_full_table(catalog, type_cast_source_schema, table_name)

    if not table_exists(source_table):
        for column_name, expected_mysql_type in expected_columns.items():
            all_type_cast_metric_rows.append({
                "run_started_at": type_cast_run_started_at,
                "table_name": table_name,
                "column_name": column_name,
                "expected_mysql_type": expected_mysql_type,
                "target_spark_type": spark_type_sql_for_mysql_type(expected_mysql_type),
                "source_spark_type": None,
                "source_rows": None,
                "source_non_null_rows": None,
                "cast_failure_rows": None,
                "status": "MISSING_TABLE",
                "message": f"Source table does not exist: {source_table}",
            })
        continue

    try:
        source_df = spark.table(source_table)

        casted_df, metric_rows = cast_table_to_expected_types(
            df=source_df,
            table_name=table_name,
            expected_columns=expected_columns,
        )

        safe_write_casted_table(
            df=casted_df,
            catalog_name=catalog,
            source_schema_name=type_cast_source_schema,
            target_schema_name=type_cast_target_schema,
            table_name=table_name,
            base_path=type_cast_base_path,
        )

        for metric_row in metric_rows:
            metric_row["run_started_at"] = type_cast_run_started_at
            all_type_cast_metric_rows.append(metric_row)

    except Exception as error:
        for column_name, expected_mysql_type in expected_columns.items():
            all_type_cast_metric_rows.append({
                "run_started_at": type_cast_run_started_at,
                "table_name": table_name,
                "column_name": column_name,
                "expected_mysql_type": expected_mysql_type,
                "target_spark_type": spark_type_sql_for_mysql_type(expected_mysql_type),
                "source_spark_type": None,
                "source_rows": None,
                "source_non_null_rows": None,
                "cast_failure_rows": None,
                "status": "FAILED_TABLE_CAST",
                "message": str(error),
            })


type_cast_metric_schema = StructType([
    StructField("run_started_at", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("expected_mysql_type", StringType(), True),
    StructField("target_spark_type", StringType(), True),
    StructField("source_spark_type", StringType(), True),
    StructField("source_rows", LongType(), True),
    StructField("source_non_null_rows", LongType(), True),
    StructField("cast_failure_rows", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
])

silver_type_cast_summary_df = spark.createDataFrame(
    all_type_cast_metric_rows,
    schema=type_cast_metric_schema,
)

write_delta_table(
    df=silver_type_cast_summary_df,
    catalog_name=catalog,
    schema_name=type_cast_target_schema,
    table_name="_silver_type_cast_run_summary",
    base_path=type_cast_base_path,
    mode="overwrite",
)

display(
    silver_type_cast_summary_df
    .orderBy(
        col("status").desc(),
        col("cast_failure_rows").desc_nulls_last(),
        col("table_name"),
        col("column_name"),
    )
)


run_started_at,table_name,column_name,expected_mysql_type,target_spark_type,source_spark_type,source_rows,source_non_null_rows,cast_failure_rows,status,message
2026-06-22T04:51:46.272562+00:00,business,business_id,binary(16),BINARY,binary,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,business_name,varchar(150),STRING,string,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,city,varchar(50),STRING,string,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,county,varchar(50),STRING,string,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,created_at,timestamp,TIMESTAMP,timestamp,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,email,varchar(100),STRING,string,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,phone_number,varchar(20),STRING,string,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,preferred_facility_id,int,INT,int,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,state_code,char(2),STRING,string,3095,3095,0,OK,Column cast successfully.
2026-06-22T04:51:46.272562+00:00,business,street_address,varchar(150),STRING,string,3095,3095,0,OK,Column cast successfully.


In [0]:
# ============================================================
# Silver Transform #2
# Cell 5: Show only type-casting issues
# ============================================================

valid_type_cast_statuses = ["OK"]

display(
    silver_type_cast_summary_df
    .filter(~col("status").isin(valid_type_cast_statuses))
    .orderBy(
        col("status"),
        col("cast_failure_rows").desc_nulls_last(),
        col("table_name"),
        col("column_name"),
    )
)

run_started_at,table_name,column_name,expected_mysql_type,target_spark_type,source_spark_type,source_rows,source_non_null_rows,cast_failure_rows,status,message
2026-06-22T04:51:46.272562+00:00,package_movement_staffing_rule,requires_employee,tinyint(1),TINYINT,boolean,36,36,36,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_flow_event_rule,is_allowed,tinyint(1),TINYINT,boolean,24,24,24,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_status,is_active,tinyint(1),TINYINT,boolean,15,15,15,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_status,is_final_status,tinyint(1),TINYINT,boolean,15,15,15,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_movement_event_type,is_active,tinyint(1),TINYINT,boolean,12,12,12,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_movement_event_type,is_entry_event,tinyint(1),TINYINT,boolean,12,12,12,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_movement_event_type,is_exit_event,tinyint(1),TINYINT,boolean,12,12,12,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_movement_event_type,is_final_event,tinyint(1),TINYINT,boolean,12,12,12,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_movement_event_type,is_processing_event,tinyint(1),TINYINT,boolean,12,12,12,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.
2026-06-22T04:51:46.272562+00:00,package_movement_field_rule,allows_from_facility_id,tinyint(1),TINYINT,boolean,12,12,12,CAST_FAILURES_FOUND,One or more non-null source values became NULL after casting.


In [0]:
# Type compatibility helper functions
# Functions that map the MySQL expected types into Databricks/Spark types.

def normalize_type_name(type_name: str) -> str:
    return type_name.lower().strip().replace("`", "")


def expected_spark_type_family(mysql_type: str) -> set:
    """
    Converts a MySQL type into acceptable Spark/Databricks type names.

    This is intentionally compatibility-based instead of overly strict,
    because ADF/MySQL/Parquet ingestion may represent small integer and
    boolean-like fields differently.
    """
    t = normalize_type_name(mysql_type)

    if t.startswith("binary"):
        return {"binary"}

    if t.startswith("char") or t.startswith("varchar") or t == "text" or t.startswith("enum"):
        return {"string"}

    if t.startswith("date") and not t.startswith("datetime"):
        return {"date"}

    if t.startswith("datetime") or t.startswith("timestamp"):
        return {"timestamp", "timestamp_ntz"}

    if t.startswith("tinyint"):
        # MySQL tinyint(1) often behaves like a boolean, but may ingest as byte/int.
        return {"boolean", "tinyint", "smallint", "int", "bigint"}

    if t.startswith("smallint"):
        return {"smallint", "int", "bigint"}

    if t.startswith("int"):
        return {"int", "bigint"}

    if t.startswith("bigint"):
        return {"bigint"}

    if t.startswith("decimal"):
        # Decimal is checked more precisely in is_type_compatible().
        return {"decimal"}

    if t.startswith("float"):
        return {"float", "double"}

    if t.startswith("double"):
        return {"double"}

    return {"unknown"}


def parse_decimal_type(type_name: str):
    """
    Returns (precision, scale) for decimal(p,s), otherwise None.
    """
    match = re.search(r"decimal\((\d+),\s*(\d+)\)", normalize_type_name(type_name))
    if not match:
        return None

    return int(match.group(1)), int(match.group(2))


def is_type_compatible(expected_mysql_type: str, actual_spark_type: str) -> bool:
    expected = normalize_type_name(expected_mysql_type)
    actual = normalize_type_name(actual_spark_type)

    # Decimal check: require actual decimal precision/scale to be at least as wide.
    if expected.startswith("decimal"):
        expected_decimal = parse_decimal_type(expected)
        actual_decimal = parse_decimal_type(actual)

        if expected_decimal is None:
            return actual.startswith("decimal")

        if actual_decimal is None:
            return False

        expected_precision, expected_scale = expected_decimal
        actual_precision, actual_scale = actual_decimal

        return actual_precision >= expected_precision and actual_scale >= expected_scale

    accepted_types = expected_spark_type_family(expected)
    return actual in accepted_types


def expected_type_display(mysql_type: str) -> str:
    expected = normalize_type_name(mysql_type)

    if expected.startswith("decimal"):
        return expected

    return ", ".join(sorted(expected_spark_type_family(expected)))

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


results = []

for table_name, expected_columns in expected_mysql_schema.items():
    full_table_name = f"`{catalog}`.`{source_schema}`.`{table_name}`"

    if not table_exists(full_table_name):
        for column_name, expected_mysql_type in expected_columns.items():
            results.append({
                "table_name": table_name,
                "column_name": column_name,
                "expected_mysql_type": expected_mysql_type,
                "expected_spark_type": expected_type_display(expected_mysql_type),
                "actual_spark_type": None,
                "status": "MISSING_TABLE",
                "detail": f"Table {full_table_name} does not exist."
            })
        continue

    actual_df = spark.table(full_table_name)
    actual_fields = {field.name.lower(): field for field in actual_df.schema.fields}
    expected_column_lookup = {column_name.lower(): column_name for column_name in expected_columns.keys()}

    # Check expected columns.
    for column_name, expected_mysql_type in expected_columns.items():
        actual_field = actual_fields.get(column_name.lower())

        if actual_field is None:
            results.append({
                "table_name": table_name,
                "column_name": column_name,
                "expected_mysql_type": expected_mysql_type,
                "expected_spark_type": expected_type_display(expected_mysql_type),
                "actual_spark_type": None,
                "status": "MISSING_COLUMN",
                "detail": "Expected column is missing from the bronze table."
            })
            continue

        actual_spark_type = actual_field.dataType.simpleString()
        compatible = is_type_compatible(expected_mysql_type, actual_spark_type)

        results.append({
            "table_name": table_name,
            "column_name": actual_field.name,
            "expected_mysql_type": expected_mysql_type,
            "expected_spark_type": expected_type_display(expected_mysql_type),
            "actual_spark_type": actual_spark_type,
            "status": "OK" if compatible else "TYPE_MISMATCH",
            "detail": None if compatible else f"Expected compatible with {expected_type_display(expected_mysql_type)}, found {actual_spark_type}."
        })

    # Check extra columns that exist in bronze but not expected.
    for actual_column_name, actual_field in actual_fields.items():
        if actual_column_name not in expected_column_lookup:
            results.append({
                "table_name": table_name,
                "column_name": actual_field.name,
                "expected_mysql_type": None,
                "expected_spark_type": None,
                "actual_spark_type": actual_field.dataType.simpleString(),
                "status": "EXTRA_COLUMN",
                "detail": "Column exists in bronze but is not listed in expected_mysql_schema."
            })


type_check_result_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("expected_mysql_type", StringType(), True),
    StructField("expected_spark_type", StringType(), True),
    StructField("actual_spark_type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("detail", StringType(), True),
])

type_check_df = spark.createDataFrame(results, schema=type_check_result_schema)

display(
    type_check_df
    .orderBy("table_name", "column_name")
)

table_name,column_name,expected_mysql_type,expected_spark_type,actual_spark_type,status,detail
business,business_id,binary(16),binary,binary,OK,null
business,business_name,varchar(150),string,string,OK,null
business,city,varchar(50),string,string,OK,null
business,county,varchar(50),string,string,OK,null
business,created_at,timestamp,"timestamp, timestamp_ntz",timestamp,OK,null
business,email,varchar(100),string,string,OK,null
business,phone_number,varchar(20),string,string,OK,null
business,preferred_facility_id,int,"bigint, int",int,OK,null
business,state_code,char(2),string,string,OK,null
business,street_address,varchar(150),string,string,OK,null


In [0]:
display(
    type_check_df
    .filter(col("status") != "OK")
    .orderBy("table_name", "status", "column_name")
)

table_name,column_name,expected_mysql_type,expected_spark_type,actual_spark_type,status,detail
customer,birth_date,date,date,timestamp,TYPE_MISMATCH,"Expected compatible with date, found timestamp."


In [0]:
# Table summary
type_check_summary_df = (
    type_check_df
    .groupBy("table_name")
    .agg(
        count("*").alias("checked_columns"),
        spark_sum(when(col("status") == "OK", 1).otherwise(0)).alias("ok_columns"),
        spark_sum(when(col("status") == "TYPE_MISMATCH", 1).otherwise(0)).alias("type_mismatches"),
        spark_sum(when(col("status") == "MISSING_COLUMN", 1).otherwise(0)).alias("missing_columns"),
        spark_sum(when(col("status") == "EXTRA_COLUMN", 1).otherwise(0)).alias("extra_columns"),
        spark_sum(when(col("status") == "MISSING_TABLE", 1).otherwise(0)).alias("missing_table_columns")
    )
    .orderBy(
        col("type_mismatches").desc(),
        col("missing_columns").desc(),
        col("extra_columns").desc(),
        col("table_name")
    )
)

display(type_check_summary_df)

table_name,checked_columns,ok_columns,type_mismatches,missing_columns,extra_columns,missing_table_columns
customer,25,24,1,0,0,0
business,13,13,0,0,0,0
department_type,7,7,0,0,0,0
departments,8,8,0,0,0,0
email_queue,6,6,0,0,0,0
employee,13,13,0,0,0,0
facility,13,13,0,0,0,0
facility_type,12,12,0,0,0,0
facility_type_department_rule,3,3,0,0,0,0
incident,16,16,0,0,0,0


### Trim/standardize strings

In [0]:
# ============================================================
# Silver Transform #3: Trim / Standardize String Fields
# Cell 1: Config
#
# Recommended run order:
#   bronze -> dedupe by PK -> cast types -> trim/standardize strings
# ============================================================

from datetime import datetime, timezone
import uuid
import re

from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col,
    lit,
    trim,
    lower,
    upper,
    length,
    regexp_replace,
    when,
    expr,
    count as spark_count,
    sum as spark_sum,
    current_timestamp,
    create_map,
)
from pyspark.sql.types import (
    StringType,
    StructType,
    StructField,
    LongType,
)


# catalog = globals().get("catalog", "workspace")

string_clean_source_schema = globals().get(
    "string_clean_source_schema",
    globals().get("target_schema", "silver")
)

string_clean_target_schema = globals().get(
    "string_clean_target_schema",
    globals().get("target_schema", "silver")
)

string_clean_base_path = globals().get(
    "string_clean_base_path",
    globals().get("silver_base_path", None)
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{string_clean_target_schema}`")

if "expected_mysql_schema" not in globals():
    raise ValueError("expected_mysql_schema was not found. Run your schema dictionary cell first.")

string_clean_schema_spec = expected_mysql_schema
DQ_CAST_ERROR_FIELD = globals().get("DQ_CAST_ERROR_FIELD", "dq_cast_error_reason")

CONVERT_BLANK_STRINGS_TO_NULL = True
REMOVE_CONTROL_CHARACTERS = True
COLLAPSE_INTERNAL_WHITESPACE = True
APPLY_KNOWN_DOMAIN_STANDARDIZATION = True
NORMALIZE_PHONE_NUMBERS = False
INITCAP_NAME_COLUMNS = False


In [0]:
# ============================================================
# Silver Transform #3
# Cell 2: Helpers
# ============================================================

def quote_identifier(identifier: str) -> str:
    return "`" + identifier.replace("`", "``") + "`"


def quote_full_table(catalog_name: str, schema_name: str, table_name: str) -> str:
    return (
        f"{quote_identifier(catalog_name)}."
        f"{quote_identifier(schema_name)}."
        f"{quote_identifier(table_name)}"
    )


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


def resolve_column_name(df: DataFrame, requested_column_name: str):
    lookup = {
        existing_column.lower(): existing_column
        for existing_column in df.columns
    }

    return lookup.get(requested_column_name.lower())


def write_delta_table(
    df: DataFrame,
    catalog_name: str,
    schema_name: str,
    table_name: str,
    base_path: str = None,
    mode: str = "overwrite",
):
    target_table = quote_full_table(catalog_name, schema_name, table_name)

    writer = (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
    )

    if base_path:
        table_path = f"{base_path.rstrip('/')}/{table_name}"
        writer = writer.option("path", table_path)

    writer.saveAsTable(target_table)


def prepare_df_for_same_table_overwrite(df: DataFrame) -> DataFrame:
    try:
        return df.localCheckpoint(eager=True)
    except Exception:
        materialized_df = df.cache()
        materialized_df.count()
        return materialized_df


def safe_write_string_cleaned_table(
    df: DataFrame,
    catalog_name: str,
    source_schema_name: str,
    target_schema_name: str,
    table_name: str,
    base_path: str = None,
):
    source_table = quote_full_table(catalog_name, source_schema_name, table_name)
    target_table = quote_full_table(catalog_name, target_schema_name, table_name)

    same_table = source_table == target_table
    prepared_df = prepare_df_for_same_table_overwrite(df) if same_table else df

    write_delta_table(
        df=prepared_df,
        catalog_name=catalog_name,
        schema_name=target_schema_name,
        table_name=table_name,
        base_path=base_path,
        mode="overwrite",
    )

    if same_table:
        try:
            prepared_df.unpersist()
        except Exception:
            pass


def is_string_column(df: DataFrame, column_name: str) -> bool:
    return isinstance(df.schema[column_name].dataType, StringType)


def get_string_columns(df: DataFrame):
    return [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
    ]


In [0]:
# ============================================================
# Silver Transform #3
# Cell 3: Known domain standardization rules
#
# One-letter OLTP code mappings are intentionally column-specific.
# ============================================================

def normalize_domain_key(value: str) -> str:
    if value is None:
        return None

    value = str(value).strip().lower()
    value = re.sub(r"[_\-\/]+", " ", value)
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


GENERAL_DOMAIN_STANDARDIZATION_PAIRS = {
    "tx": "TX",
    "texas": "TX",
    "b2c": "B2C",
    "business to customer": "B2C",
    "business customer": "B2C",
    "p2p": "P2P",
    "peer to peer": "P2P",
    "peer peer": "P2P",
    "partial high school": "Partial High School",
    "high school": "High School",
    "graduate degree": "Graduate Degree",
    "partial college": "Partial College",
    "bachelors": "Bachelors",
    "bachelor": "Bachelors",
    "bachelor s": "Bachelors",
    "bachelors degree": "Bachelors",
    "bachelor degree": "Bachelors",
    "skilled manual": "Skilled Manual",
    "professional": "Professional",
    "management": "Management",
    "clerical": "Clerical",
    "manual": "Manual",
    "post office": "Post Office",
    "vehicle maintenance": "Vehicle Maintenance",
    "administrative office": "Administrative Office",
    "network facilities": "Network Facilities",
    "network facility": "Network Facilities",
    "mail processing": "Mail Processing",
    "mail processing facility": "Mail Processing",
    "retail services": "Retail Services",
    "delivery": "Delivery",
    "logistics": "Logistics",
    "operations": "Operations",
    "maintenance": "Maintenance",
    "administrative": "Administrative",
    "damaged package": "Damaged Package",
    "lost package": "Lost Package",
    "delayed delivery": "Delayed Delivery",
    "customer complaint": "Customer Complaint",
    "employee accident": "Employee Accident",
    "tracking error": "Tracking Error",
    "facility issue": "Facility Issue",
    "cancelled package": "Cancelled Package",
    "canceled package": "Cancelled Package",
    "returned package": "Returned Package",
    "other": "Other",
    "low": "Low",
    "medium": "Medium",
    "high": "High",
    "critical": "Critical",
    "open": "Open",
    "investigating": "Investigating",
    "resolved": "Resolved",
    "closed": "Closed",
    "created": "Created",
    "received": "Received",
    "sorted": "Sorted",
    "departed": "Departed",
    "in transit": "In Transit",
    "out for delivery": "Out for Delivery",
    "delivered": "Delivered",
    "delayed": "Delayed",
    "lost": "Lost",
    "damaged": "Damaged",
    "returned": "Returned",
    "cancelled": "Cancelled",
    "canceled": "Cancelled",
    "requested": "Requested",
    "approved": "Approved",
    "accepted": "Accepted",
    "rejected": "Rejected",
    "denied": "Rejected",
    "paid": "Paid",
    "disbursed": "Disbursed",
    "pending": "Pending",
}

GENERAL_DOMAIN_STANDARDIZATION_MAP = {
    normalize_domain_key(raw_value): canonical_value
    for raw_value, canonical_value in GENERAL_DOMAIN_STANDARDIZATION_PAIRS.items()
}

COLUMN_SPECIFIC_STANDARDIZATION_PAIRS = {
    ("customer", "gender"): {
        "male": "M",
        "m": "M",
        "female": "F",
        "f": "F",
        "n a": "N",
        "na": "N",
        "unknown": "N",
        "not applicable": "N",
        "none": "N",
        "n": "N",
    },
    ("customer", "marital_status"): {
        "married": "M",
        "m": "M",
        "single": "S",
        "s": "S",
    },
    ("customer", "home_owner"): {
        "yes": "Y",
        "y": "Y",
        "true": "Y",
        "1": "Y",
        "homeowner": "Y",
        "no": "N",
        "n": "N",
        "false": "N",
        "0": "N",
        "renter": "N",
    },
}

COLUMN_NAME_STANDARDIZATION_PAIRS = {
    "state": {
        "tx": "TX",
        "texas": "TX",
    },
    "state_code": {
        "tx": "TX",
        "texas": "TX",
    },
}


def get_column_specific_standardization_map(table_name: str, column_name: str):
    pairs = {}
    pairs.update(COLUMN_NAME_STANDARDIZATION_PAIRS.get(column_name.lower(), {}))
    pairs.update(COLUMN_SPECIFIC_STANDARDIZATION_PAIRS.get((table_name.lower(), column_name.lower()), {}))

    return {
        normalize_domain_key(raw_value): canonical_value
        for raw_value, canonical_value in pairs.items()
    }


def should_lowercase_column(column_name: str) -> bool:
    name = column_name.lower()
    return (
        name == "email"
        or name.endswith("_email")
        or "email_address" in name
    )


def should_uppercase_column(column_name: str) -> bool:
    name = column_name.lower()

    return (
        name in {"state", "state_code", "country_code"}
        or name.endswith("_state_code")
        or name.endswith("_code")
    )


def should_normalize_phone_column(column_name: str) -> bool:
    return "phone" in column_name.lower()


def should_initcap_column(column_name: str) -> bool:
    if not INITCAP_NAME_COLUMNS:
        return False

    name = column_name.lower()
    return (
        name.endswith("_name")
        or name in {"city", "county"}
    )


In [0]:
# ============================================================
# Silver Transform #3
# Cell 4: Build per-column standardization expressions
# ============================================================

def build_general_domain_standardization_expression(cleaned_col):
    if not GENERAL_DOMAIN_STANDARDIZATION_MAP:
        return cleaned_col

    mapping_expr_items = []

    for normalized_key, canonical_value in GENERAL_DOMAIN_STANDARDIZATION_MAP.items():
        mapping_expr_items.extend([
            lit(normalized_key),
            lit(canonical_value),
        ])

    mapping_expr = create_map(*mapping_expr_items)

    normalized_key_col = regexp_replace(
        regexp_replace(
            regexp_replace(
                lower(trim(cleaned_col)),
                r"[_\-\/]+",
                " "
            ),
            r"[^a-z0-9]+",
            " "
        ),
        r"\s+",
        " "
    )

    normalized_key_col = trim(normalized_key_col)

    return when(
        mapping_expr[normalized_key_col].isNotNull(),
        mapping_expr[normalized_key_col]
    ).otherwise(cleaned_col)


def build_column_specific_standardization_expression(table_name: str, column_name: str, cleaned_col):
    mapping = get_column_specific_standardization_map(table_name, column_name)

    if not mapping:
        return cleaned_col

    mapping_expr_items = []

    for normalized_key, canonical_value in mapping.items():
        mapping_expr_items.extend([
            lit(normalized_key),
            lit(canonical_value),
        ])

    mapping_expr = create_map(*mapping_expr_items)

    normalized_key_col = regexp_replace(
        regexp_replace(
            regexp_replace(
                lower(trim(cleaned_col)),
                r"[_\-\/]+",
                " "
            ),
            r"[^a-z0-9]+",
            " "
        ),
        r"\s+",
        " "
    )

    normalized_key_col = trim(normalized_key_col)

    return when(
        mapping_expr[normalized_key_col].isNotNull(),
        mapping_expr[normalized_key_col]
    ).otherwise(cleaned_col)


def build_string_clean_expression(table_name: str, column_name: str):
    q_col = quote_identifier(column_name)
    cleaned_col = col(q_col).cast("string")

    if REMOVE_CONTROL_CHARACTERS:
        cleaned_col = regexp_replace(cleaned_col, r"[\x00-\x1F\x7F]", " ")

    cleaned_col = regexp_replace(cleaned_col, r"[\t\r\n]+", " ")

    if COLLAPSE_INTERNAL_WHITESPACE:
        cleaned_col = regexp_replace(cleaned_col, r"\s+", " ")

    cleaned_col = trim(cleaned_col)

    if CONVERT_BLANK_STRINGS_TO_NULL:
        cleaned_col = when(length(cleaned_col) == 0, lit(None)).otherwise(cleaned_col)

    if should_lowercase_column(column_name):
        cleaned_col = when(cleaned_col.isNotNull(), lower(cleaned_col)).otherwise(cleaned_col)

    if should_uppercase_column(column_name):
        cleaned_col = when(cleaned_col.isNotNull(), upper(cleaned_col)).otherwise(cleaned_col)

    if NORMALIZE_PHONE_NUMBERS and should_normalize_phone_column(column_name):
        digits_only = regexp_replace(cleaned_col, r"[^0-9]", "")

        cleaned_col = (
            when(cleaned_col.isNull(), lit(None))
            .when((length(digits_only) == 11) & digits_only.startswith("1"), expr(f"substring(regexp_replace({q_col}, '[^0-9]', ''), 2, 10)"))
            .otherwise(digits_only)
        )

        if CONVERT_BLANK_STRINGS_TO_NULL:
            cleaned_col = when(length(cleaned_col) == 0, lit(None)).otherwise(cleaned_col)

    if should_initcap_column(column_name):
        cleaned_col = expr(f"initcap({q_col})")

    if APPLY_KNOWN_DOMAIN_STANDARDIZATION:
        cleaned_col = build_general_domain_standardization_expression(cleaned_col)
        cleaned_col = build_column_specific_standardization_expression(table_name, column_name, cleaned_col)

    return cleaned_col


In [0]:
# ============================================================
# Silver Transform #3
# Cell 5: Clean one table and collect metrics
# ============================================================

def standardize_string_columns_for_table(
    df: DataFrame,
    table_name: str,
):
    string_columns = get_string_columns(df)

    if not string_columns:
        return df, [{
            "table_name": table_name,
            "column_name": None,
            "source_rows": df.count(),
            "non_null_before_rows": None,
            "non_null_after_rows": None,
            "blank_to_null_rows": None,
            "changed_rows": None,
            "status": "NO_STRING_COLUMNS",
            "message": "Table has no StringType columns to standardize.",
        }]

    cleaned_df = df

    clean_expr_by_column = {
        column_name: build_string_clean_expression(table_name, column_name)
        for column_name in string_columns
    }

    metric_exprs = [
        spark_count(lit(1)).alias("_source_rows")
    ]

    for index, column_name in enumerate(string_columns):
        original_col = col(quote_identifier(column_name))
        cleaned_col = clean_expr_by_column[column_name]
        original_as_string = original_col.cast("string")

        metric_exprs.append(
            spark_sum(
                when(original_col.isNotNull(), lit(1)).otherwise(lit(0))
            ).alias(f"_c{index}_non_null_before_rows")
        )

        metric_exprs.append(
            spark_sum(
                when(cleaned_col.isNotNull(), lit(1)).otherwise(lit(0))
            ).alias(f"_c{index}_non_null_after_rows")
        )

        metric_exprs.append(
            spark_sum(
                when(
                    original_col.isNotNull()
                    & cleaned_col.isNull()
                    & (length(trim(original_as_string)) == 0),
                    lit(1)
                ).otherwise(lit(0))
            ).alias(f"_c{index}_blank_to_null_rows")
        )

        metric_exprs.append(
            spark_sum(
                when(
                    original_col.eqNullSafe(cleaned_col),
                    lit(0)
                ).otherwise(lit(1))
            ).alias(f"_c{index}_changed_rows")
        )

    stats = df.agg(*metric_exprs).collect()[0].asDict()
    source_rows = stats["_source_rows"]
    metric_rows = []

    for index, column_name in enumerate(string_columns):
        cleaned_df = cleaned_df.withColumn(
            column_name,
            clean_expr_by_column[column_name],
        )

        changed_rows = stats[f"_c{index}_changed_rows"]
        blank_to_null_rows = stats[f"_c{index}_blank_to_null_rows"]

        if changed_rows and changed_rows > 0:
            status = "STANDARDIZED"
            message = "One or more values changed after string standardization."
        else:
            status = "OK"
            message = "No values required string standardization."

        metric_rows.append({
            "table_name": table_name,
            "column_name": column_name,
            "source_rows": source_rows,
            "non_null_before_rows": stats[f"_c{index}_non_null_before_rows"],
            "non_null_after_rows": stats[f"_c{index}_non_null_after_rows"],
            "blank_to_null_rows": blank_to_null_rows,
            "changed_rows": changed_rows,
            "status": status,
            "message": message,
        })

    return cleaned_df, metric_rows


In [0]:
# ============================================================
# Silver Transform #3
# Cell 6: Run string standardization for every table
# ============================================================

string_clean_run_started_at = datetime.now(timezone.utc).isoformat()

all_string_clean_metric_rows = []

for table_name in string_clean_schema_spec.keys():
    source_table = quote_full_table(catalog, string_clean_source_schema, table_name)

    if not table_exists(source_table):
        all_string_clean_metric_rows.append({
            "run_started_at": string_clean_run_started_at,
            "table_name": table_name,
            "column_name": None,
            "source_rows": None,
            "non_null_before_rows": None,
            "non_null_after_rows": None,
            "blank_to_null_rows": None,
            "changed_rows": None,
            "status": "MISSING_TABLE",
            "message": f"Source table does not exist: {source_table}",
        })
        continue

    try:
        source_df = spark.table(source_table)

        cleaned_df, metric_rows = standardize_string_columns_for_table(
            df=source_df,
            table_name=table_name,
        )

        safe_write_string_cleaned_table(
            df=cleaned_df,
            catalog_name=catalog,
            source_schema_name=string_clean_source_schema,
            target_schema_name=string_clean_target_schema,
            table_name=table_name,
            base_path=string_clean_base_path,
        )

        for metric_row in metric_rows:
            metric_row["run_started_at"] = string_clean_run_started_at
            all_string_clean_metric_rows.append(metric_row)

    except Exception as error:
        all_string_clean_metric_rows.append({
            "run_started_at": string_clean_run_started_at,
            "table_name": table_name,
            "column_name": None,
            "source_rows": None,
            "non_null_before_rows": None,
            "non_null_after_rows": None,
            "blank_to_null_rows": None,
            "changed_rows": None,
            "status": "FAILED_TABLE_STRING_STANDARDIZATION",
            "message": str(error),
        })


string_clean_metric_schema = StructType([
    StructField("run_started_at", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("source_rows", LongType(), True),
    StructField("non_null_before_rows", LongType(), True),
    StructField("non_null_after_rows", LongType(), True),
    StructField("blank_to_null_rows", LongType(), True),
    StructField("changed_rows", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
])

silver_string_standardization_summary_df = spark.createDataFrame(
    all_string_clean_metric_rows,
    schema=string_clean_metric_schema,
)

write_delta_table(
    df=silver_string_standardization_summary_df,
    catalog_name=catalog,
    schema_name=string_clean_target_schema,
    table_name="_silver_string_standardization_summary",
    base_path=string_clean_base_path,
    mode="overwrite",
)

display(
    silver_string_standardization_summary_df
    .orderBy(
        col("status").desc(),
        col("changed_rows").desc_nulls_last(),
        col("blank_to_null_rows").desc_nulls_last(),
        col("table_name"),
        col("column_name"),
    )
)


run_started_at,table_name,column_name,source_rows,non_null_before_rows,non_null_after_rows,blank_to_null_rows,changed_rows,status,message
2026-06-22T04:53:45.298187+00:00,department_type,department_type_code,6,6,6,0,4,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,facility,county,541,541,540,1,1,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,package_movement_event_type,default_package_status_name,12,12,12,0,1,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,package_movement_event_type,event_type_name,12,12,12,0,1,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,package_status,status_name,15,15,15,0,1,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,business,business_name,3095,3095,3095,0,0,OK,No values required string standardization.
2026-06-22T04:53:45.298187+00:00,business,city,3095,3095,3095,0,0,OK,No values required string standardization.
2026-06-22T04:53:45.298187+00:00,business,county,3095,3095,3095,0,0,OK,No values required string standardization.
2026-06-22T04:53:45.298187+00:00,business,dq_cast_error_reason,3095,0,0,0,0,OK,No values required string standardization.
2026-06-22T04:53:45.298187+00:00,business,email,3095,3095,3095,0,0,OK,No values required string standardization.


In [0]:
# ============================================================
# Silver Transform #3
# Cell 7: Problem/change-only string standardization summary
# ============================================================

display(
    silver_string_standardization_summary_df
    .filter(
        (col("status") != "OK")
        & (col("status") != "NO_STRING_COLUMNS")
    )
    .orderBy(
        col("status"),
        col("changed_rows").desc_nulls_last(),
        col("blank_to_null_rows").desc_nulls_last(),
        col("table_name"),
        col("column_name"),
    )
)

run_started_at,table_name,column_name,source_rows,non_null_before_rows,non_null_after_rows,blank_to_null_rows,changed_rows,status,message
2026-06-22T04:53:45.298187+00:00,department_type,department_type_code,6,6,6,0,4,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,facility,county,541,541,540,1,1,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,package_movement_event_type,default_package_status_name,12,12,12,0,1,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,package_movement_event_type,event_type_name,12,12,12,0,1,STANDARDIZED,One or more values changed after string standardization.
2026-06-22T04:53:45.298187+00:00,package_status,status_name,15,15,15,0,1,STANDARDIZED,One or more values changed after string standardization.


In [0]:
# ============================================================
# Silver Validation Check #3: Trim / Standardize String Validation
#
# Checks:
#   - Leading / trailing spaces
#   - Double or repeated spaces
#   - Blank strings
#   - Case/capitalization variants
#   - Tabs, newlines, and control characters
#   - Low-cardinality domain variants
#   - Acronym/full-word variants
#   - Possible typos in domain fields
#
# Assumes:
#   - catalog and source_schema are already defined
#   - expected_mysql_schema is already defined from your schema ZIP
# ============================================================

from functools import reduce
from difflib import get_close_matches

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col,
    lit,
    trim,
    lower,
    regexp_replace,
    length,
    when,
    sum as spark_sum,
    count,
    countDistinct,
    approx_count_distinct,
    collect_set,
    concat_ws,
    desc,
)

from pyspark.sql.types import StringType, StructType, StructField, LongType


# Use approximate distinct counts for faster EDA.
# Set this to False if you want exact distinct counts.
use_approx_distinct = True

# A column is considered low-cardinality if its approximate distinct count is <= this.
low_cardinality_threshold = 100

# For value-frequency displays, cap how many raw values we pull per column.
max_values_per_domain_column = 250


def standardize_string_expr(column_name: str):
    """
    Standard silver-style text normalization expression:
      - trim leading/trailing whitespace
      - collapse repeated whitespace into one space
      - lowercase
    """
    return lower(regexp_replace(trim(col(column_name)), r"\s+", " "))


def distinct_count_expr(column_expr):
    if use_approx_distinct:
        return approx_count_distinct(column_expr)
    return countDistinct(column_expr)


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


In [0]:
# ============================================================
# Cell 2: Generic String Quality Summary
# ============================================================

string_quality_results = []

for table_name in expected_mysql_schema.keys():
    full_table_name = f"`{catalog}`.`{source_schema}`.`{table_name}`"

    if not table_exists(full_table_name):
        string_quality_results.append({
            "table_name": table_name,
            "column_name": None,
            "total_rows": None,
            "non_null_rows": None,
            "null_rows": None,
            "blank_string_rows": None,
            "leading_trailing_space_rows": None,
            "repeated_space_rows": None,
            "tab_newline_control_char_rows": None,
            "raw_distinct_values": None,
            "standardized_distinct_values": None,
            "possible_case_spacing_variants": None,
            "status": "MISSING_TABLE",
        })
        continue

    df = spark.table(full_table_name)

    string_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
    ]

    if not string_columns:
        continue

    agg_exprs = []

    for c in string_columns:
        safe_alias = c.replace("`", "")

        raw_col = col(c)
        std_col = standardize_string_expr(c)

        agg_exprs.extend([
            count(lit(1)).alias(f"{safe_alias}__total_rows"),

            count(raw_col).alias(f"{safe_alias}__non_null_rows"),

            spark_sum(
                when(raw_col.isNull(), 1).otherwise(0)
            ).alias(f"{safe_alias}__null_rows"),

            spark_sum(
                when(raw_col.isNotNull() & (length(trim(raw_col)) == 0), 1).otherwise(0)
            ).alias(f"{safe_alias}__blank_string_rows"),

            spark_sum(
                when(raw_col.isNotNull() & (raw_col != trim(raw_col)), 1).otherwise(0)
            ).alias(f"{safe_alias}__leading_trailing_space_rows"),

            spark_sum(
                when(raw_col.isNotNull() & raw_col.rlike(r" {2,}"), 1).otherwise(0)
            ).alias(f"{safe_alias}__repeated_space_rows"),

            spark_sum(
                when(raw_col.isNotNull() & raw_col.rlike(r"[\t\r\n\x00-\x1F\x7F]"), 1).otherwise(0)
            ).alias(f"{safe_alias}__tab_newline_control_char_rows"),

            distinct_count_expr(raw_col).alias(f"{safe_alias}__raw_distinct_values"),

            distinct_count_expr(std_col).alias(f"{safe_alias}__standardized_distinct_values"),
        ])

    agg_row = df.agg(*agg_exprs).collect()[0].asDict()

    for c in string_columns:
        safe_alias = c.replace("`", "")

        raw_distinct = agg_row[f"{safe_alias}__raw_distinct_values"]
        standardized_distinct = agg_row[f"{safe_alias}__standardized_distinct_values"]

        possible_case_spacing_variants = (
            raw_distinct is not None
            and standardized_distinct is not None
            and raw_distinct > standardized_distinct
        )

        issue_count = sum([
            agg_row[f"{safe_alias}__blank_string_rows"] or 0,
            agg_row[f"{safe_alias}__leading_trailing_space_rows"] or 0,
            agg_row[f"{safe_alias}__repeated_space_rows"] or 0,
            agg_row[f"{safe_alias}__tab_newline_control_char_rows"] or 0,
            1 if possible_case_spacing_variants else 0,
        ])

        string_quality_results.append({
            "table_name": table_name,
            "column_name": c,
            "total_rows": agg_row[f"{safe_alias}__total_rows"],
            "non_null_rows": agg_row[f"{safe_alias}__non_null_rows"],
            "null_rows": agg_row[f"{safe_alias}__null_rows"],
            "blank_string_rows": agg_row[f"{safe_alias}__blank_string_rows"],
            "leading_trailing_space_rows": agg_row[f"{safe_alias}__leading_trailing_space_rows"],
            "repeated_space_rows": agg_row[f"{safe_alias}__repeated_space_rows"],
            "tab_newline_control_char_rows": agg_row[f"{safe_alias}__tab_newline_control_char_rows"],
            "raw_distinct_values": raw_distinct,
            "standardized_distinct_values": standardized_distinct,
            "possible_case_spacing_variants": possible_case_spacing_variants,
            "status": "HAS_STRING_QUALITY_ISSUE" if issue_count > 0 else "OK",
        })


string_quality_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("total_rows", LongType(), True),
    StructField("non_null_rows", LongType(), True),
    StructField("null_rows", LongType(), True),
    StructField("blank_string_rows", LongType(), True),
    StructField("leading_trailing_space_rows", LongType(), True),
    StructField("repeated_space_rows", LongType(), True),
    StructField("tab_newline_control_char_rows", LongType(), True),
    StructField("raw_distinct_values", LongType(), True),
    StructField("standardized_distinct_values", LongType(), True),
    StructField("possible_case_spacing_variants", StringType(), True),
    StructField("status", StringType(), True),
])

string_quality_df = spark.createDataFrame(string_quality_results, schema=string_quality_schema)

display(
    string_quality_df
    .orderBy(
        desc("blank_string_rows"),
        desc("leading_trailing_space_rows"),
        desc("repeated_space_rows"),
        desc("tab_newline_control_char_rows"),
        desc("raw_distinct_values"),
        "table_name",
        "column_name"
    )
)

table_name,column_name,total_rows,non_null_rows,null_rows,blank_string_rows,leading_trailing_space_rows,repeated_space_rows,tab_newline_control_char_rows,raw_distinct_values,standardized_distinct_values,possible_case_spacing_variants,status
facility,county,541,541,0,1,0,0,0,169,165,true,HAS_STRING_QUALITY_ISSUE
customer,email,96096,96096,0,0,0,0,0,101629,101629,false,OK
customer,email_address,96096,96096,0,0,0,0,0,101629,101629,false,OK
shippingdetails,recipient_address,105454,105454,0,0,0,0,0,96484,93327,true,HAS_STRING_QUALITY_ISSUE
customer,street_address,96096,96096,0,0,0,0,0,94160,90818,true,HAS_STRING_QUALITY_ISSUE
customer,phone_number,96096,96096,0,0,0,0,0,75551,75551,false,OK
package,contents,105454,105454,0,0,0,0,0,6239,6257,false,OK
employee,email,5000,5000,0,0,0,0,0,5338,5338,false,OK
employee,phone_number,5000,5000,0,0,0,0,0,5029,5029,false,OK
employee,full_name,5000,5000,0,0,0,0,0,4804,5014,false,OK


In [0]:
# ============================================================
# Cell 3: Show only string-quality problems
# ============================================================

display(
    string_quality_df
    .filter(col("status") != "OK")
    .orderBy(
        desc("blank_string_rows"),
        desc("leading_trailing_space_rows"),
        desc("repeated_space_rows"),
        desc("tab_newline_control_char_rows"),
        "table_name",
        "column_name"
    )
)

table_name,column_name,total_rows,non_null_rows,null_rows,blank_string_rows,leading_trailing_space_rows,repeated_space_rows,tab_newline_control_char_rows,raw_distinct_values,standardized_distinct_values,possible_case_spacing_variants,status
facility,county,541,541,0,1,0,0,0,169,165,true,HAS_STRING_QUALITY_ISSUE
business,business_name,3095,3095,0,0,0,0,0,3057,2795,true,HAS_STRING_QUALITY_ISSUE
business,city,3095,3095,0,0,0,0,0,423,410,true,HAS_STRING_QUALITY_ISSUE
customer,city,96096,96096,0,0,0,0,0,1065,1019,true,HAS_STRING_QUALITY_ISSUE
customer,first_name,96096,96096,0,0,0,0,0,42,41,true,HAS_STRING_QUALITY_ISSUE
customer,street_address,96096,96096,0,0,0,0,0,94160,90818,true,HAS_STRING_QUALITY_ISSUE
departments,department_name,1596,1596,0,0,0,0,0,1648,1521,true,HAS_STRING_QUALITY_ISSUE
facility,city,541,541,0,0,0,0,0,371,367,true,HAS_STRING_QUALITY_ISSUE
facility,facility_name,541,541,0,0,0,0,0,541,521,true,HAS_STRING_QUALITY_ISSUE
facility,street_address,541,541,0,0,0,0,0,561,505,true,HAS_STRING_QUALITY_ISSUE


In [0]:
# ============================================================
# Cell 4: Low-Cardinality Case / Spacing Variant Details
# ============================================================

variant_results = []

candidate_columns = (
    string_quality_df
    .filter(
        (col("raw_distinct_values") <= low_cardinality_threshold)
        & (col("possible_case_spacing_variants") == "true")
    )
    .select("table_name", "column_name")
    .collect()
)

for row in candidate_columns:
    table_name = row["table_name"]
    column_name = row["column_name"]

    full_table_name = f"`{catalog}`.`{source_schema}`.`{table_name}`"

    df = spark.table(full_table_name)

    grouped = (
        df
        .filter(col(column_name).isNotNull())
        .withColumn("standardized_value", standardize_string_expr(column_name))
        .groupBy("standardized_value")
        .agg(
            count("*").alias("row_count"),
            countDistinct(col(column_name)).alias("raw_variant_count"),
            concat_ws(" | ", collect_set(col(column_name))).alias("raw_variants")
        )
        .filter(col("raw_variant_count") > 1)
        .orderBy(desc("row_count"))
        .limit(100)
    )

    for r in grouped.collect():
        variant_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "standardized_value": r["standardized_value"],
            "raw_variant_count": r["raw_variant_count"],
            "row_count": r["row_count"],
            "raw_variants": r["raw_variants"],
        })


variant_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("standardized_value", StringType(), True),
    StructField("raw_variant_count", LongType(), True),
    StructField("row_count", LongType(), True),
    StructField("raw_variants", StringType(), True),
])

case_spacing_variant_df = spark.createDataFrame(variant_results, schema=variant_schema)

display(
    case_spacing_variant_df
    .orderBy(desc("row_count"), "table_name", "column_name")
)

table_name,column_name,standardized_value,raw_variant_count,row_count,raw_variants


In [0]:
# ============================================================
# Cell 5: Domain Rules for Low-Cardinality Fields
# ============================================================

domain_rules = {
    ("business", "state_code"): {
        "allowed": ["TX"],
        "aliases": {
            "texas": "TX",
            "tx.": "TX",
        },
    },

    ("customer", "state_code"): {
        "allowed": ["TX"],
        "aliases": {
            "texas": "TX",
            "tx.": "TX",
        },
    },

    ("facility", "state_code"): {
        "allowed": ["TX"],
        "aliases": {
            "texas": "TX",
            "tx.": "TX",
        },
    },

    ("territory", "state"): {
        "allowed": ["TX"],
        "aliases": {
            "texas": "TX",
            "tx.": "TX",
        },
    },

    ("customer", "marital_status"): {
        "allowed": ["M", "S"],
        "aliases": {
            "married": "M",
            "single": "S",
        },
    },

    ("customer", "gender"): {
        "allowed": ["M", "F", "N"],
        "aliases": {
            "male": "M",
            "female": "F",
            "n/a": "N",
            "na": "N",
            "not applicable": "N",
        },
    },

    ("customer", "home_owner"): {
        "allowed": ["Y", "N"],
        "aliases": {
            "yes": "Y",
            "no": "N",
            "true": "Y",
            "false": "N",
        },
    },

    ("facility_type", "facility_type_name"): {
        "allowed": [
            "Post Office",
            "Vehicle Maintenance",
            "Administrative Office",
            "Network Facilities",
            "Mail Processing",
        ],
        "aliases": {
            "po": "Post Office",
            "p.o.": "Post Office",
            "post offices": "Post Office",
            "vehicle maintenance facility": "Vehicle Maintenance",
            "admin office": "Administrative Office",
            "administrative": "Administrative Office",
            "network facility": "Network Facilities",
            "network facilities": "Network Facilities",
            "nf": "Network Facilities",
            "mail processing facility": "Mail Processing",
            "mp": "Mail Processing",
        },
    },

    ("department_type", "department_type_name"): {
        "allowed": [
            "Retail Services",
            "Delivery",
            "Logistics",
            "Operations",
            "Maintenance",
            "Administrative",
        ],
        "aliases": {
            "retail": "Retail Services",
            "retail service": "Retail Services",
            "retail services": "Retail Services",
            "delivery services": "Delivery",
            "logistic": "Logistics",
            "logistics": "Logistics",
            "ops": "Operations",
            "operation": "Operations",
            "operations": "Operations",
            "maint": "Maintenance",
            "admin": "Administrative",
            "administration": "Administrative",
        },
    },

    ("incident_severity", "severity_name"): {
        "allowed": ["Low", "Medium", "High", "Critical"],
        "aliases": {
            "med": "Medium",
            "crit": "Critical",
        },
    },

    ("incident_status", "status_name"): {
        "allowed": ["Open", "Investigating", "Resolved", "Closed"],
        "aliases": {
            "investigation": "Investigating",
            "in investigation": "Investigating",
            "resolve": "Resolved",
            "complete": "Closed",
            "completed": "Closed",
        },
    },

    ("incident_type", "type_name"): {
        "allowed": [
            "Damaged Package",
            "Lost Package",
            "Delayed Delivery",
            "Customer Complaint",
            "Employee Accident",
            "Tracking Error",
            "Facility Issue",
            "Cancelled Package",
            "Returned Package",
            "Other",
        ],
        "aliases": {
            "damage": "Damaged Package",
            "damaged": "Damaged Package",
            "lost": "Lost Package",
            "delay": "Delayed Delivery",
            "delayed": "Delayed Delivery",
            "late delivery": "Delayed Delivery",
            "complaint": "Customer Complaint",
            "customer issue": "Customer Complaint",
            "accident": "Employee Accident",
            "tracking": "Tracking Error",
            "facility problem": "Facility Issue",
            "canceled package": "Cancelled Package",
            "cancelled": "Cancelled Package",
            "canceled": "Cancelled Package",
            "returned": "Returned Package",
        },
    },

    ("package_flow_type", "package_flow_type_name"): {
        "allowed": ["B2C", "P2P"],
        "aliases": {
            "business to customer": "B2C",
            "business-to-customer": "B2C",
            "customer order": "B2C",
            "peer to peer": "P2P",
            "peer-to-peer": "P2P",
            "customer to customer": "P2P",
            "customer-to-customer": "P2P",
        },
    },

    ("smartlocker", "locker_status"): {
        "allowed": ["Available", "Occupied", "Out of service"],
        "aliases": {
            "open": "Available",
            "free": "Available",
            "in use": "Occupied",
            "out-of-service": "Out of service",
            "out of service": "Out of service",
            "broken": "Out of service",
            "maintenance": "Out of service",
        },
    },

    ("refunds", "refund_status"): {
        "allowed": ["Pending", "Approved", "Rejected", "Paid", "Cancelled"],
        "aliases": {
            "approve": "Approved",
            "denied": "Rejected",
            "deny": "Rejected",
            "complete": "Paid",
            "completed": "Paid",
            "canceled": "Cancelled",
        },
    },

    ("refunds", "refund_reason"): {
        "allowed": [
            "Late Delivery",
            "Damaged Package",
            "Lost Package",
            "Returned Package",
            "Cancelled Package",
            "Service Failure",
            "Goodwill Adjustment",
            "Other",
        ],
        "aliases": {
            "delayed delivery": "Late Delivery",
            "late": "Late Delivery",
            "damaged": "Damaged Package",
            "lost": "Lost Package",
            "returned": "Returned Package",
            "canceled package": "Cancelled Package",
            "canceled": "Cancelled Package",
            "service issue": "Service Failure",
            "goodwill": "Goodwill Adjustment",
        },
    },
}


def normalize_python_string(value):
    if value is None:
        return None

    return " ".join(str(value).strip().lower().split())


def classify_domain_value(raw_value, allowed_values, aliases):
    if raw_value is None:
        return ("NULL", None)

    raw_as_text = str(raw_value)
    normalized_raw = normalize_python_string(raw_as_text)

    if normalized_raw == "":
        return ("BLANK_STRING", None)

    allowed_lookup = {
        normalize_python_string(v): v
        for v in allowed_values
    }

    alias_lookup = {
        normalize_python_string(k): v
        for k, v in aliases.items()
    }

    if normalized_raw in allowed_lookup:
        canonical_value = allowed_lookup[normalized_raw]

        if raw_as_text == canonical_value:
            return ("OK", canonical_value)

        return ("CASE_OR_SPACING_VARIANT", canonical_value)

    if normalized_raw in alias_lookup:
        return ("ACRONYM_OR_ALIAS_VARIANT", alias_lookup[normalized_raw])

    close_matches = get_close_matches(
        normalized_raw,
        list(allowed_lookup.keys()),
        n=1,
        cutoff=0.82
    )

    if close_matches:
        return ("POSSIBLE_TYPO", allowed_lookup[close_matches[0]])

    return ("OUTSIDE_ALLOWED_DOMAIN", None)

In [0]:
# ============================================================
# Cell 6: Domain Value Validation
# ============================================================

domain_results = []

for (table_name, column_name), rule in domain_rules.items():
    full_table_name = f"`{catalog}`.`{source_schema}`.`{table_name}`"

    if not table_exists(full_table_name):
        domain_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "raw_value": None,
            "row_count": None,
            "standardized_value": None,
            "status": "MISSING_TABLE",
            "suggested_canonical_value": None,
        })
        continue

    df = spark.table(full_table_name)

    if column_name not in df.columns:
        domain_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "raw_value": None,
            "row_count": None,
            "standardized_value": None,
            "status": "MISSING_COLUMN",
            "suggested_canonical_value": None,
        })
        continue

    value_counts = (
        df
        .groupBy(col(column_name).alias("raw_value"))
        .agg(count("*").alias("row_count"))
        .orderBy(desc("row_count"))
        .limit(max_values_per_domain_column)
        .collect()
    )

    for r in value_counts:
        raw_value = r["raw_value"]
        status, suggested_value = classify_domain_value(
            raw_value,
            rule["allowed"],
            rule.get("aliases", {})
        )

        domain_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "raw_value": None if raw_value is None else str(raw_value),
            "row_count": r["row_count"],
            "standardized_value": normalize_python_string(raw_value),
            "status": status,
            "suggested_canonical_value": suggested_value,
        })


domain_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("raw_value", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("standardized_value", StringType(), True),
    StructField("status", StringType(), True),
    StructField("suggested_canonical_value", StringType(), True),
])

domain_validation_df = spark.createDataFrame(domain_results, schema=domain_schema)

display(
    domain_validation_df
    .orderBy("table_name", "column_name", desc("row_count"))
)

table_name,column_name,raw_value,row_count,standardized_value,status,suggested_canonical_value
business,state_code,TX,3095,tx,OK,TX
customer,gender,F,48996,f,OK,F
customer,gender,M,47100,m,OK,M
customer,home_owner,Y,58472,y,OK,Y
customer,home_owner,N,37624,n,OK,N
customer,marital_status,S,49419,s,OK,S
customer,marital_status,M,46677,m,OK,M
customer,state_code,TX,96096,tx,OK,TX
department_type,department_type_name,Administrative,1,administrative,OK,Administrative
department_type,department_type_name,Delivery,1,delivery,OK,Delivery


In [0]:
# ============================================================
# Cell 7: Domain Problems Only
# ============================================================

display(
    domain_validation_df
    .filter(col("status") != "OK")
    .orderBy("table_name", "column_name", "status", desc("row_count"))
)

table_name,column_name,raw_value,row_count,standardized_value,status,suggested_canonical_value


In [0]:
# ============================================================
# Cell 8: String Quality + Domain Summary
# ============================================================

string_quality_problem_summary_df = (
    string_quality_df
    .groupBy("table_name")
    .agg(
        spark_sum("blank_string_rows").alias("blank_string_rows"),
        spark_sum("leading_trailing_space_rows").alias("leading_trailing_space_rows"),
        spark_sum("repeated_space_rows").alias("repeated_space_rows"),
        spark_sum("tab_newline_control_char_rows").alias("tab_newline_control_char_rows"),
        spark_sum(
            when(col("possible_case_spacing_variants") == "true", 1).otherwise(0)
        ).alias("columns_with_possible_case_spacing_variants"),
    )
)

domain_problem_summary_df = (
    domain_validation_df
    .filter(col("status") != "OK")
    .groupBy("table_name")
    .agg(
        count("*").alias("domain_problem_value_count"),
        spark_sum("row_count").alias("domain_problem_row_count"),
    )
)

string_domain_summary_df = (
    string_quality_problem_summary_df
    .join(domain_problem_summary_df, on="table_name", how="full")
    .fillna(0)
    .orderBy(
        desc("blank_string_rows"),
        desc("leading_trailing_space_rows"),
        desc("repeated_space_rows"),
        desc("tab_newline_control_char_rows"),
        desc("domain_problem_row_count"),
        "table_name"
    )
)

display(string_domain_summary_df)

table_name,blank_string_rows,leading_trailing_space_rows,repeated_space_rows,tab_newline_control_char_rows,columns_with_possible_case_spacing_variants,domain_problem_value_count,domain_problem_row_count
facility,1,0,0,0,4,0,0
business,0,0,0,0,2,0,0
customer,0,0,0,0,3,0,0
department_type,0,0,0,0,0,0,0
departments,0,0,0,0,1,0,0
email_queue,0,0,0,0,0,0,0
employee,0,0,0,0,0,0,0
facility_type,0,0,0,0,0,0,0
incident,0,0,0,0,0,0,0
incident_severity,0,0,0,0,0,0,0


### Validate primary keys are not null

In [0]:
# ============================================================
# Silver Validation Check #4: Primary Keys Are Not NULL
# Cell 1: Primary key map
#
# Assumes:
#   - catalog = "postal_databricks"
#   - source_schema = "bronze"
# ============================================================

primary_key_map = {
    "business": ["business_id"],
    "customer": ["customer_id"],
    "departments": ["department_id"],
    "department_type": ["department_type_id"],
    "email_queue": ["email_queue_id"],
    "employee": ["employee_id"],
    "facility": ["facility_id"],
    "facility_type": ["facility_type_id"],
    "facility_type_department_rule": ["facility_type_id", "department_type_id"],
    "incident": ["incident_id"],
    "incident_severity": ["incident_severity_id"],
    "incident_status": ["incident_status_id"],
    "incident_type": ["incident_type_id"],
    "lockerassignment": ["locker_assignment_id"],
    "lockerlocations": ["locker_location_id"],
    "notifications": ["notification_id"],
    "package": ["package_id"],
    "package_flow_event_rule": ["package_flow_type_id", "package_movement_event_type_id"],
    "package_flow_type": ["package_flow_type_id"],
    "package_movement": ["package_movement_id"],
    "package_movement_event_type": ["package_movement_event_type_id"],
    "package_movement_field_rule": ["package_movement_event_type_id"],
    "package_movement_staffing_rule": [
        "package_flow_type_id",
        "package_movement_event_type_id",
        "facility_type_id",
    ],
    "package_route_plan": ["package_id"],
    "package_status": ["package_status_id"],
    "package_to_locker": ["package_id"],
    "refunds": ["refund_id"],
    "roles": ["role_id"],
    "service_type": ["service_type_id"],
    "shippingdetails": ["package_id"],
    "shipping_cost": ["package_id"],
    "smartlocker": ["locker_id"],
    "territory": ["territory_id"],
    "user_logins": ["user_id"],
    "user_roles": ["user_role_id"],
    "works_on": ["employee_id", "department_id"],
    "zip_geo": ["zip_code"],
}


In [0]:
# ============================================================
# Cell 2: Validate that primary keys are not NULL
# ============================================================

from pyspark.sql.functions import col, lit, count, sum as spark_sum, when
from pyspark.sql.types import StructType, StructField, StringType, LongType


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


pk_null_results = []

for table_name, pk_columns in primary_key_map.items():
    full_table_name = f"`{catalog}`.`{source_schema}`.`{table_name}`"

    if not table_exists(full_table_name):
        pk_null_results.append({
            "table_name": table_name,
            "primary_key_columns": ", ".join(pk_columns),
            "total_rows": None,
            "null_pk_rows": None,
            "missing_pk_columns": ", ".join(pk_columns),
            "pk_component_null_counts": None,
            "status": "MISSING_TABLE",
        })
        continue

    df = spark.table(full_table_name)
    actual_columns_lower = {c.lower(): c for c in df.columns}

    missing_pk_columns = [
        pk_col
        for pk_col in pk_columns
        if pk_col.lower() not in actual_columns_lower
    ]

    if missing_pk_columns:
        pk_null_results.append({
            "table_name": table_name,
            "primary_key_columns": ", ".join(pk_columns),
            "total_rows": df.count(),
            "null_pk_rows": None,
            "missing_pk_columns": ", ".join(missing_pk_columns),
            "pk_component_null_counts": None,
            "status": "MISSING_PK_COLUMN",
        })
        continue

    # Use actual column casing from the table.
    resolved_pk_columns = [
        actual_columns_lower[pk_col.lower()]
        for pk_col in pk_columns
    ]

    any_pk_null_condition = None

    for pk_col in resolved_pk_columns:
        condition = col(pk_col).isNull()
        any_pk_null_condition = (
            condition
            if any_pk_null_condition is None
            else any_pk_null_condition | condition
        )

    agg_exprs = [
        count(lit(1)).alias("total_rows"),
        spark_sum(
            when(any_pk_null_condition, 1).otherwise(0)
        ).alias("null_pk_rows"),
    ]

    for pk_col in resolved_pk_columns:
        agg_exprs.append(
            spark_sum(
                when(col(pk_col).isNull(), 1).otherwise(0)
            ).alias(f"{pk_col}__null_count")
        )

    agg_row = df.agg(*agg_exprs).collect()[0].asDict()

    component_null_counts = "; ".join([
        f"{pk_col}={agg_row[f'{pk_col}__null_count']}"
        for pk_col in resolved_pk_columns
    ])

    null_pk_rows = agg_row["null_pk_rows"] or 0

    pk_null_results.append({
        "table_name": table_name,
        "primary_key_columns": ", ".join(resolved_pk_columns),
        "total_rows": agg_row["total_rows"],
        "null_pk_rows": null_pk_rows,
        "missing_pk_columns": None,
        "pk_component_null_counts": component_null_counts,
        "status": "OK" if null_pk_rows == 0 else "HAS_NULL_PRIMARY_KEY",
    })


pk_null_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("primary_key_columns", StringType(), True),
    StructField("total_rows", LongType(), True),
    StructField("null_pk_rows", LongType(), True),
    StructField("missing_pk_columns", StringType(), True),
    StructField("pk_component_null_counts", StringType(), True),
    StructField("status", StringType(), True),
])

pk_null_df = spark.createDataFrame(pk_null_results, schema=pk_null_schema)

display(
    pk_null_df
    .orderBy(
        col("null_pk_rows").desc_nulls_last(),
        "status",
        "table_name"
    )
)

table_name,primary_key_columns,total_rows,null_pk_rows,missing_pk_columns,pk_component_null_counts,status
business,business_id,3095,0,null,business_id=0,OK
customer,customer_id,96096,0,null,customer_id=0,OK
department_type,department_type_id,6,0,null,department_type_id=0,OK
departments,department_id,1596,0,null,department_id=0,OK
email_queue,email_queue_id,0,0,null,email_queue_id=None,OK
employee,employee_id,5000,0,null,employee_id=0,OK
facility,facility_id,541,0,null,facility_id=0,OK
facility_type,facility_type_id,5,0,null,facility_type_id=0,OK
facility_type_department_rule,"facility_type_id, department_type_id",13,0,null,facility_type_id=0; department_type_id=0,OK
incident,incident_id,6286,0,null,incident_id=0,OK


In [0]:
# ============================================================
# Cell 3: Problem-only primary-key NULL results
# ============================================================

display(
    pk_null_df
    .filter(col("status") != "OK")
    .orderBy(
        col("null_pk_rows").desc_nulls_last(),
        "table_name"
    )
)

table_name,primary_key_columns,total_rows,null_pk_rows,missing_pk_columns,pk_component_null_counts,status


In [0]:
# ============================================================
# Cell 4: Sample rows where a selected table has NULL primary keys
# ============================================================

table_to_inspect = "package_movement"  # change this

pk_columns = primary_key_map[table_to_inspect]
full_table_name = f"`{catalog}`.`{source_schema}`.`{table_to_inspect}`"

df = spark.table(full_table_name)

actual_columns_lower = {c.lower(): c for c in df.columns}
resolved_pk_columns = [
    actual_columns_lower[pk_col.lower()]
    for pk_col in pk_columns
]

any_pk_null_condition = None

for pk_col in resolved_pk_columns:
    condition = col(pk_col).isNull()
    any_pk_null_condition = (
        condition
        if any_pk_null_condition is None
        else any_pk_null_condition | condition
    )

display(
    df
    .filter(any_pk_null_condition)
    .limit(100)
)

package_movement_id,package_id,package_movement_event_type_id,package_status_id,facility_id,from_facility_id,to_facility_id,processed_by_employee_id,event_timestamp,movement_note,created_at,updated_at


### Validate foreign keys exist

In [0]:
# ============================================================
# Silver Validation Check #5: Foreign Keys Exist
# Cell 1: Foreign key map
#
# Assumes:
#   - catalog = "postal_databricks"
#   - source_schema = "bronze"
#
# Notes:
#   - This FK map is based on SchemaOnly.zip.
#   - Each FK is checked only when the child FK value is NOT NULL.
#   - NULL FK values are reported separately as null_fk_rows.
# ============================================================

foreign_key_map = [
    {
        "constraint_name": "fk_business_preferred_facility",
        "child_table": "business",
        "child_column": "preferred_facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },
    {
        "constraint_name": "fk_business_territory",
        "child_table": "business",
        "child_column": "territory_id",
        "parent_table": "territory",
        "parent_column": "territory_id",
    },

    {
        "constraint_name": "customer_ibfk_2",
        "child_table": "customer",
        "child_column": "user_id",
        "parent_table": "user_logins",
        "parent_column": "user_id",
    },
    {
        "constraint_name": "fk_customer_preferred_facility",
        "child_table": "customer",
        "child_column": "preferred_facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },
    {
        "constraint_name": "fk_customer_territory",
        "child_table": "customer",
        "child_column": "territory_id",
        "parent_table": "territory",
        "parent_column": "territory_id",
    },

    {
        "constraint_name": "fk_departments_department_type",
        "child_table": "departments",
        "child_column": "department_type_id",
        "parent_table": "department_type",
        "parent_column": "department_type_id",
    },
    {
        "constraint_name": "fk_departments_facility",
        "child_table": "departments",
        "child_column": "facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },
    {
        "constraint_name": "fk_departments_manager",
        "child_table": "departments",
        "child_column": "manager_employee_id",
        "parent_table": "employee",
        "parent_column": "employee_id",
    },

    {
        "constraint_name": "fk_employee_department",
        "child_table": "employee",
        "child_column": "department_id",
        "parent_table": "departments",
        "parent_column": "department_id",
    },
    {
        "constraint_name": "fk_employee_manager",
        "child_table": "employee",
        "child_column": "manager_employee_id",
        "parent_table": "employee",
        "parent_column": "employee_id",
    },
    {
        "constraint_name": "fk_employeeLogin",
        "child_table": "employee",
        "child_column": "user_id",
        "parent_table": "user_logins",
        "parent_column": "user_id",
    },

    {
        "constraint_name": "fk_facility_facility_type",
        "child_table": "facility",
        "child_column": "facility_type_id",
        "parent_table": "facility_type",
        "parent_column": "facility_type_id",
    },
    {
        "constraint_name": "fk_facility_manager_employee",
        "child_table": "facility",
        "child_column": "manager_employee_id",
        "parent_table": "employee",
        "parent_column": "employee_id",
    },
    {
        "constraint_name": "fk_facility_territory",
        "child_table": "facility",
        "child_column": "territory_id",
        "parent_table": "territory",
        "parent_column": "territory_id",
    },

    {
        "constraint_name": "fk_ftdr_department_type",
        "child_table": "facility_type_department_rule",
        "child_column": "department_type_id",
        "parent_table": "department_type",
        "parent_column": "department_type_id",
    },
    {
        "constraint_name": "fk_ftdr_facility_type",
        "child_table": "facility_type_department_rule",
        "child_column": "facility_type_id",
        "parent_table": "facility_type",
        "parent_column": "facility_type_id",
    },

    {
        "constraint_name": "fk_incident_customer",
        "child_table": "incident",
        "child_column": "customer_id",
        "parent_table": "customer",
        "parent_column": "customer_id",
    },
    {
        "constraint_name": "fk_incident_facility",
        "child_table": "incident",
        "child_column": "facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },
    {
        "constraint_name": "fk_incident_package",
        "child_table": "incident",
        "child_column": "package_id",
        "parent_table": "package",
        "parent_column": "package_id",
    },
    {
        "constraint_name": "fk_incident_package_movement",
        "child_table": "incident",
        "child_column": "package_movement_id",
        "parent_table": "package_movement",
        "parent_column": "package_movement_id",
    },
    {
        "constraint_name": "fk_incident_reported_by_employee",
        "child_table": "incident",
        "child_column": "reported_by_employee_id",
        "parent_table": "employee",
        "parent_column": "employee_id",
    },
    {
        "constraint_name": "fk_incident_resolved_by_employee",
        "child_table": "incident",
        "child_column": "resolved_by_employee_id",
        "parent_table": "employee",
        "parent_column": "employee_id",
    },
    {
        "constraint_name": "fk_incident_severity",
        "child_table": "incident",
        "child_column": "incident_severity_id",
        "parent_table": "incident_severity",
        "parent_column": "incident_severity_id",
    },
    {
        "constraint_name": "fk_incident_status",
        "child_table": "incident",
        "child_column": "incident_status_id",
        "parent_table": "incident_status",
        "parent_column": "incident_status_id",
    },
    {
        "constraint_name": "fk_incident_type",
        "child_table": "incident",
        "child_column": "incident_type_id",
        "parent_table": "incident_type",
        "parent_column": "incident_type_id",
    },

    {
        "constraint_name": "customerID_fk",
        "child_table": "lockerassignment",
        "child_column": "customer_id",
        "parent_table": "customer",
        "parent_column": "customer_id",
    },
    {
        "constraint_name": "lockerID_fk",
        "child_table": "lockerassignment",
        "child_column": "locker_id",
        "parent_table": "smartlocker",
        "parent_column": "locker_id",
    },

    {
        "constraint_name": "fk_lockerlocations_facility",
        "child_table": "lockerlocations",
        "child_column": "facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },

    {
        "constraint_name": "customer_id",
        "child_table": "notifications",
        "child_column": "customer_id",
        "parent_table": "customer",
        "parent_column": "customer_id",
    },
    {
        "constraint_name": "notifications_ibfk_1",
        "child_table": "notifications",
        "child_column": "package_id",
        "parent_table": "package",
        "parent_column": "package_id",
    },

    {
        "constraint_name": "fk_package_package_flow_type",
        "child_table": "package",
        "child_column": "package_flow_type_id",
        "parent_table": "package_flow_type",
        "parent_column": "package_flow_type_id",
    },
    {
        "constraint_name": "fk_package_recipient_customer",
        "child_table": "package",
        "child_column": "recipient_customer_id",
        "parent_table": "customer",
        "parent_column": "customer_id",
    },
    {
        "constraint_name": "fk_package_sender_business",
        "child_table": "package",
        "child_column": "sender_business_id",
        "parent_table": "business",
        "parent_column": "business_id",
    },
    {
        "constraint_name": "fk_package_sender_customer",
        "child_table": "package",
        "child_column": "sender_customer_id",
        "parent_table": "customer",
        "parent_column": "customer_id",
    },
    {
        "constraint_name": "fk_package_service_type",
        "child_table": "package",
        "child_column": "service_type_id",
        "parent_table": "service_type",
        "parent_column": "service_type_id",
    },
    {
        "constraint_name": "fk_package_status",
        "child_table": "package",
        "child_column": "package_status_id",
        "parent_table": "package_status",
        "parent_column": "package_status_id",
    },

    {
        "constraint_name": "fk_pfer_event_type",
        "child_table": "package_flow_event_rule",
        "child_column": "package_movement_event_type_id",
        "parent_table": "package_movement_event_type",
        "parent_column": "package_movement_event_type_id",
    },
    {
        "constraint_name": "fk_pfer_package_flow_type",
        "child_table": "package_flow_event_rule",
        "child_column": "package_flow_type_id",
        "parent_table": "package_flow_type",
        "parent_column": "package_flow_type_id",
    },

    {
        "constraint_name": "fk_package_movement_employee",
        "child_table": "package_movement",
        "child_column": "processed_by_employee_id",
        "parent_table": "employee",
        "parent_column": "employee_id",
    },
    {
        "constraint_name": "fk_package_movement_event_type",
        "child_table": "package_movement",
        "child_column": "package_movement_event_type_id",
        "parent_table": "package_movement_event_type",
        "parent_column": "package_movement_event_type_id",
    },
    {
        "constraint_name": "fk_package_movement_facility",
        "child_table": "package_movement",
        "child_column": "facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },
    {
        "constraint_name": "fk_package_movement_from_facility",
        "child_table": "package_movement",
        "child_column": "from_facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },
    {
        "constraint_name": "fk_package_movement_package",
        "child_table": "package_movement",
        "child_column": "package_id",
        "parent_table": "package",
        "parent_column": "package_id",
    },
    {
        "constraint_name": "fk_package_movement_status",
        "child_table": "package_movement",
        "child_column": "package_status_id",
        "parent_table": "package_status",
        "parent_column": "package_status_id",
    },
    {
        "constraint_name": "fk_package_movement_to_facility",
        "child_table": "package_movement",
        "child_column": "to_facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },

    {
        "constraint_name": "fk_pmfr_event_type",
        "child_table": "package_movement_field_rule",
        "child_column": "package_movement_event_type_id",
        "parent_table": "package_movement_event_type",
        "parent_column": "package_movement_event_type_id",
    },

    {
        "constraint_name": "fk_pmsr_event_type",
        "child_table": "package_movement_staffing_rule",
        "child_column": "package_movement_event_type_id",
        "parent_table": "package_movement_event_type",
        "parent_column": "package_movement_event_type_id",
    },
    {
        "constraint_name": "fk_pmsr_facility_type",
        "child_table": "package_movement_staffing_rule",
        "child_column": "facility_type_id",
        "parent_table": "facility_type",
        "parent_column": "facility_type_id",
    },
    {
        "constraint_name": "fk_pmsr_package_flow_type",
        "child_table": "package_movement_staffing_rule",
        "child_column": "package_flow_type_id",
        "parent_table": "package_flow_type",
        "parent_column": "package_flow_type_id",
    },
    {
        "constraint_name": "fk_pmsr_required_department_type",
        "child_table": "package_movement_staffing_rule",
        "child_column": "required_department_type_id",
        "parent_table": "department_type",
        "parent_column": "department_type_id",
    },

    {
        "constraint_name": "fk_package_route_plan_destination_facility",
        "child_table": "package_route_plan",
        "child_column": "planned_destination_facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },
    {
        "constraint_name": "fk_package_route_plan_origin_facility",
        "child_table": "package_route_plan",
        "child_column": "planned_origin_facility_id",
        "parent_table": "facility",
        "parent_column": "facility_id",
    },
    {
        "constraint_name": "fk_package_route_plan_package",
        "child_table": "package_route_plan",
        "child_column": "package_id",
        "parent_table": "package",
        "parent_column": "package_id",
    },

    {
        "constraint_name": "fk_package_status_allowed_service_type",
        "child_table": "package_status",
        "child_column": "allowed_service_type_id",
        "parent_table": "service_type",
        "parent_column": "service_type_id",
    },

    {
        "constraint_name": "fk_associatedpackage_locker",
        "child_table": "package_to_locker",
        "child_column": "locker_assignment_id",
        "parent_table": "lockerassignment",
        "parent_column": "locker_assignment_id",
    },
    {
        "constraint_name": "fk_package_to_locker_customer",
        "child_table": "package_to_locker",
        "child_column": "customer_id",
        "parent_table": "customer",
        "parent_column": "customer_id",
    },
    {
        "constraint_name": "fk_PackageGoingIntoLocker",
        "child_table": "package_to_locker",
        "child_column": "package_id",
        "parent_table": "package",
        "parent_column": "package_id",
    },

    {
        "constraint_name": "fk_CustomerRefund",
        "child_table": "refunds",
        "child_column": "customer_id",
        "parent_table": "customer",
        "parent_column": "customer_id",
    },
    {
        "constraint_name": "fk_refunds_incident",
        "child_table": "refunds",
        "child_column": "incident_id",
        "parent_table": "incident",
        "parent_column": "incident_id",
    },
    {
        "constraint_name": "fk_refunds_reviewed_by_employee",
        "child_table": "refunds",
        "child_column": "reviewed_by_employee_id",
        "parent_table": "employee",
        "parent_column": "employee_id",
    },
    {
        "constraint_name": "refunds_ibfk_1",
        "child_table": "refunds",
        "child_column": "package_id",
        "parent_table": "package",
        "parent_column": "package_id",
    },

    {
        "constraint_name": "fk_shipping_cost_package",
        "child_table": "shipping_cost",
        "child_column": "package_id",
        "parent_table": "package",
        "parent_column": "package_id",
    },

    {
        "constraint_name": "fk_AssociatedPackage",
        "child_table": "shippingdetails",
        "child_column": "package_id",
        "parent_table": "package",
        "parent_column": "package_id",
    },
    {
        "constraint_name": "fk_shippingdetails_recipient_territory",
        "child_table": "shippingdetails",
        "child_column": "recipient_territory_id",
        "parent_table": "territory",
        "parent_column": "territory_id",
    },
    {
        "constraint_name": "fk_shippingdetails_sender_territory",
        "child_table": "shippingdetails",
        "child_column": "sender_territory_id",
        "parent_table": "territory",
        "parent_column": "territory_id",
    },

    {
        "constraint_name": "locationID_fk",
        "child_table": "smartlocker",
        "child_column": "locker_location_id",
        "parent_table": "lockerlocations",
        "parent_column": "locker_location_id",
    },

    {
        "constraint_name": "fk_territory_zip_geo",
        "child_table": "territory",
        "child_column": "zip_code",
        "parent_table": "zip_geo",
        "parent_column": "zip_code",
    },

    {
        "constraint_name": "roleID_fk",
        "child_table": "user_roles",
        "child_column": "role_id",
        "parent_table": "roles",
        "parent_column": "role_id",
    },
    {
        "constraint_name": "userID_fk",
        "child_table": "user_roles",
        "child_column": "user_id",
        "parent_table": "user_logins",
        "parent_column": "user_id",
    },

    {
        "constraint_name": "fk_works_on_department",
        "child_table": "works_on",
        "child_column": "department_id",
        "parent_table": "departments",
        "parent_column": "department_id",
    },
    {
        "constraint_name": "fk_works_on_employee",
        "child_table": "works_on",
        "child_column": "employee_id",
        "parent_table": "employee",
        "parent_column": "employee_id",
    },
]


In [0]:
# ============================================================
# Cell 2: Validate that populated foreign keys exist
# ============================================================

from pyspark.sql.functions import col, lit, count, sum as spark_sum, hex as spark_hex
from pyspark.sql.types import StructType, StructField, StringType, LongType, BinaryType


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


def resolve_column_name(df, requested_column_name: str):
    """
    Return the actual column name from a DataFrame, preserving original casing.
    """
    lookup = {c.lower(): c for c in df.columns}
    return lookup.get(requested_column_name.lower())


def readable_value_expr(df, column_name: str, alias_name: str):
    """
    Display binary IDs as hex strings; display other values as strings.
    """
    field = df.schema[column_name]
    if isinstance(field.dataType, BinaryType):
        return spark_hex(col(column_name)).alias(alias_name)

    return col(column_name).cast("string").alias(alias_name)


fk_results = []

for fk in foreign_key_map:
    constraint_name = fk["constraint_name"]
    child_table = fk["child_table"]
    child_column = fk["child_column"]
    parent_table = fk["parent_table"]
    parent_column = fk["parent_column"]

    child_full_name = f"`{catalog}`.`{source_schema}`.`{child_table}`"
    parent_full_name = f"`{catalog}`.`{source_schema}`.`{parent_table}`"

    if not table_exists(child_full_name):
        fk_results.append({
            "constraint_name": constraint_name,
            "child_table": child_table,
            "child_column": child_column,
            "parent_table": parent_table,
            "parent_column": parent_column,
            "child_total_rows": None,
            "child_null_fk_rows": None,
            "child_populated_fk_rows": None,
            "orphan_distinct_fk_values": None,
            "orphan_child_rows": None,
            "status": "MISSING_CHILD_TABLE",
            "detail": f"Child table does not exist: {child_full_name}",
        })
        continue

    if not table_exists(parent_full_name):
        fk_results.append({
            "constraint_name": constraint_name,
            "child_table": child_table,
            "child_column": child_column,
            "parent_table": parent_table,
            "parent_column": parent_column,
            "child_total_rows": None,
            "child_null_fk_rows": None,
            "child_populated_fk_rows": None,
            "orphan_distinct_fk_values": None,
            "orphan_child_rows": None,
            "status": "MISSING_PARENT_TABLE",
            "detail": f"Parent table does not exist: {parent_full_name}",
        })
        continue

    child_df = spark.table(child_full_name)
    parent_df = spark.table(parent_full_name)

    resolved_child_column = resolve_column_name(child_df, child_column)
    resolved_parent_column = resolve_column_name(parent_df, parent_column)

    if resolved_child_column is None:
        fk_results.append({
            "constraint_name": constraint_name,
            "child_table": child_table,
            "child_column": child_column,
            "parent_table": parent_table,
            "parent_column": parent_column,
            "child_total_rows": child_df.count(),
            "child_null_fk_rows": None,
            "child_populated_fk_rows": None,
            "orphan_distinct_fk_values": None,
            "orphan_child_rows": None,
            "status": "MISSING_CHILD_COLUMN",
            "detail": f"Child FK column does not exist: {child_table}.{child_column}",
        })
        continue

    if resolved_parent_column is None:
        fk_results.append({
            "constraint_name": constraint_name,
            "child_table": child_table,
            "child_column": child_column,
            "parent_table": parent_table,
            "parent_column": parent_column,
            "child_total_rows": child_df.count(),
            "child_null_fk_rows": None,
            "child_populated_fk_rows": None,
            "orphan_distinct_fk_values": None,
            "orphan_child_rows": None,
            "status": "MISSING_PARENT_COLUMN",
            "detail": f"Parent PK column does not exist: {parent_table}.{parent_column}",
        })
        continue

    child_summary = (
        child_df
        .agg(
            count(lit(1)).alias("child_total_rows"),
            spark_sum(col(resolved_child_column).isNull().cast("long")).alias("child_null_fk_rows"),
            spark_sum(col(resolved_child_column).isNotNull().cast("long")).alias("child_populated_fk_rows"),
        )
        .collect()[0]
        .asDict()
    )

    child_fk_counts = (
        child_df
        .filter(col(resolved_child_column).isNotNull())
        .groupBy(col(resolved_child_column).alias("fk_value"))
        .agg(count(lit(1)).alias("child_row_count"))
    )

    parent_keys = (
        parent_df
        .filter(col(resolved_parent_column).isNotNull())
        .select(col(resolved_parent_column).alias("parent_key"))
        .dropDuplicates()
    )

    orphan_fk_counts = (
        child_fk_counts
        .join(
            parent_keys,
            child_fk_counts["fk_value"] == parent_keys["parent_key"],
            "left_anti"
        )
    )

    orphan_summary = (
        orphan_fk_counts
        .agg(
            count(lit(1)).alias("orphan_distinct_fk_values"),
            spark_sum(col("child_row_count")).alias("orphan_child_rows"),
        )
        .collect()[0]
        .asDict()
    )

    orphan_distinct_fk_values = orphan_summary["orphan_distinct_fk_values"] or 0
    orphan_child_rows = orphan_summary["orphan_child_rows"] or 0

    fk_results.append({
        "constraint_name": constraint_name,
        "child_table": child_table,
        "child_column": resolved_child_column,
        "parent_table": parent_table,
        "parent_column": resolved_parent_column,
        "child_total_rows": child_summary["child_total_rows"],
        "child_null_fk_rows": child_summary["child_null_fk_rows"] or 0,
        "child_populated_fk_rows": child_summary["child_populated_fk_rows"] or 0,
        "orphan_distinct_fk_values": orphan_distinct_fk_values,
        "orphan_child_rows": orphan_child_rows,
        "status": "OK" if orphan_child_rows == 0 else "HAS_ORPHAN_FOREIGN_KEYS",
        "detail": None if orphan_child_rows == 0 else "Populated child FK values exist with no matching parent key.",
    })


fk_validation_schema = StructType([
    StructField("constraint_name", StringType(), True),
    StructField("child_table", StringType(), True),
    StructField("child_column", StringType(), True),
    StructField("parent_table", StringType(), True),
    StructField("parent_column", StringType(), True),
    StructField("child_total_rows", LongType(), True),
    StructField("child_null_fk_rows", LongType(), True),
    StructField("child_populated_fk_rows", LongType(), True),
    StructField("orphan_distinct_fk_values", LongType(), True),
    StructField("orphan_child_rows", LongType(), True),
    StructField("status", StringType(), True),
    StructField("detail", StringType(), True),
])

fk_validation_df = spark.createDataFrame(fk_results, schema=fk_validation_schema)

display(
    fk_validation_df
    .orderBy(
        col("orphan_child_rows").desc_nulls_last(),
        col("orphan_distinct_fk_values").desc_nulls_last(),
        "child_table",
        "child_column",
    )
)

constraint_name,child_table,child_column,parent_table,parent_column,child_total_rows,child_null_fk_rows,child_populated_fk_rows,orphan_distinct_fk_values,orphan_child_rows,status,detail
fk_business_preferred_facility,business,preferred_facility_id,facility,facility_id,3095,0,3095,0,0,OK,null
fk_business_territory,business,territory_id,territory,territory_id,3095,0,3095,0,0,OK,null
fk_customer_preferred_facility,customer,preferred_facility_id,facility,facility_id,96096,96096,0,0,0,OK,null
fk_customer_territory,customer,territory_id,territory,territory_id,96096,0,96096,0,0,OK,null
customer_ibfk_2,customer,user_id,user_logins,user_id,96096,96096,0,0,0,OK,null
fk_departments_department_type,departments,department_type_id,department_type,department_type_id,1596,0,1596,0,0,OK,null
fk_departments_facility,departments,facility_id,facility,facility_id,1596,0,1596,0,0,OK,null
fk_departments_manager,departments,manager_employee_id,employee,employee_id,1596,0,1596,0,0,OK,null
fk_employee_department,employee,department_id,departments,department_id,5000,0,5000,0,0,OK,null
fk_employee_manager,employee,manager_employee_id,employee,employee_id,5000,1596,3404,0,0,OK,null


In [0]:
# ============================================================
# Cell 3: Problem-only FK results
# ============================================================

display(
    fk_validation_df
    .filter(col("status") != "OK")
    .orderBy(
        col("orphan_child_rows").desc_nulls_last(),
        col("orphan_distinct_fk_values").desc_nulls_last(),
        "child_table",
        "child_column",
    )
)

constraint_name,child_table,child_column,parent_table,parent_column,child_total_rows,child_null_fk_rows,child_populated_fk_rows,orphan_distinct_fk_values,orphan_child_rows,status,detail


In [0]:
# ============================================================
# Cell 4: FK validation summary by child table
# ============================================================

from pyspark.sql.functions import when

fk_table_summary_df = (
    fk_validation_df
    .groupBy("child_table")
    .agg(
        count(lit(1)).alias("foreign_keys_checked"),
        spark_sum(when(col("status") == "OK", 1).otherwise(0)).alias("ok_foreign_keys"),
        spark_sum(when(col("status") == "HAS_ORPHAN_FOREIGN_KEYS", 1).otherwise(0)).alias("foreign_keys_with_orphans"),
        spark_sum(col("child_null_fk_rows")).alias("total_null_fk_rows"),
        spark_sum(col("child_populated_fk_rows")).alias("total_populated_fk_rows"),
        spark_sum(col("orphan_distinct_fk_values")).alias("total_orphan_distinct_fk_values"),
        spark_sum(col("orphan_child_rows")).alias("total_orphan_child_rows"),
    )
    .orderBy(
        col("total_orphan_child_rows").desc_nulls_last(),
        col("foreign_keys_with_orphans").desc_nulls_last(),
        "child_table",
    )
)

display(fk_table_summary_df)

child_table,foreign_keys_checked,ok_foreign_keys,foreign_keys_with_orphans,total_null_fk_rows,total_populated_fk_rows,total_orphan_distinct_fk_values,total_orphan_child_rows
business,2,2,0,0,6190,0,0
customer,3,3,0,192192,96096,0,0
departments,3,3,0,0,4788,0,0
employee,3,3,0,6594,8406,0,0
facility,3,3,0,1,1622,0,0
facility_type_department_rule,2,2,0,0,26,0,0
incident,9,9,0,3729,52845,0,0
lockerassignment,2,2,0,0,16234,0,0
lockerlocations,1,1,0,0,6,0,0
notifications,2,2,0,0,0,0,0


In [0]:
# ============================================================
# Cell 5: Sample orphan FK values for one FK relationship
# ============================================================

fk_to_inspect = {
    "child_table": "package_movement",
    "child_column": "package_id",
    "parent_table": "package",
    "parent_column": "package_id",
}

child_table = fk_to_inspect["child_table"]
child_column = fk_to_inspect["child_column"]
parent_table = fk_to_inspect["parent_table"]
parent_column = fk_to_inspect["parent_column"]

child_df = spark.table(f"`{catalog}`.`{source_schema}`.`{child_table}`")
parent_df = spark.table(f"`{catalog}`.`{source_schema}`.`{parent_table}`")

resolved_child_column = resolve_column_name(child_df, child_column)
resolved_parent_column = resolve_column_name(parent_df, parent_column)

child_fk_counts = (
    child_df
    .filter(col(resolved_child_column).isNotNull())
    .groupBy(col(resolved_child_column).alias("fk_value"))
    .agg(count(lit(1)).alias("child_row_count"))
)

parent_keys = (
    parent_df
    .filter(col(resolved_parent_column).isNotNull())
    .select(col(resolved_parent_column).alias("parent_key"))
    .dropDuplicates()
)

orphan_fk_values = (
    child_fk_counts
    .join(
        parent_keys,
        child_fk_counts["fk_value"] == parent_keys["parent_key"],
        "left_anti"
    )
    .orderBy(col("child_row_count").desc())
)

# Display binary values as hex when needed.
child_field = child_df.schema[resolved_child_column]

if isinstance(child_field.dataType, BinaryType):
    display(
        orphan_fk_values
        .select(
            spark_hex(col("fk_value")).alias("orphan_fk_value_hex"),
            col("child_row_count"),
        )
        .limit(100)
    )
else:
    display(
        orphan_fk_values
        .select(
            col("fk_value").cast("string").alias("orphan_fk_value"),
            col("child_row_count"),
        )
        .limit(100)
    )

orphan_fk_value_hex,child_row_count


In [0]:
# ============================================================
# Cell 6: Sample full child rows with orphan FK values
# ============================================================

fk_to_inspect = {
    "child_table": "package_movement",
    "child_column": "package_id",
    "parent_table": "package",
    "parent_column": "package_id",
}

child_table = fk_to_inspect["child_table"]
child_column = fk_to_inspect["child_column"]
parent_table = fk_to_inspect["parent_table"]
parent_column = fk_to_inspect["parent_column"]

child_df = spark.table(f"`{catalog}`.`{source_schema}`.`{child_table}`")
parent_df = spark.table(f"`{catalog}`.`{source_schema}`.`{parent_table}`")

resolved_child_column = resolve_column_name(child_df, child_column)
resolved_parent_column = resolve_column_name(parent_df, parent_column)

parent_keys = (
    parent_df
    .filter(col(resolved_parent_column).isNotNull())
    .select(col(resolved_parent_column).alias("parent_key"))
    .dropDuplicates()
)

orphan_child_rows = (
    child_df
    .filter(col(resolved_child_column).isNotNull())
    .join(
        parent_keys,
        child_df[resolved_child_column] == parent_keys["parent_key"],
        "left_anti"
    )
)

display(orphan_child_rows.limit(100))

package_movement_id,package_id,package_movement_event_type_id,package_status_id,facility_id,from_facility_id,to_facility_id,processed_by_employee_id,event_timestamp,movement_note,created_at,updated_at


### Validate important numeric/date ranges

In [0]:
# ============================================================
# Silver Validation Check #6: Important Numeric / Date Range Validation
# Cell 1: Rule configuration
#
# Assumes:
#   - catalog = "postal_databricks"
#   - source_schema = "bronze"
#   - expected_mysql_schema already exists
#
# Purpose:
#   Validate important numeric and date/timestamp ranges before silver.
# ============================================================

from pyspark.sql.functions import (
    col,
    lit,
    count,
    sum as spark_sum,
    expr,
    when,
    min as spark_min,
    max as spark_max,
)
from pyspark.sql.types import StructType, StructField, StringType, LongType


# Adjust these thresholds if your demo data intentionally uses wider ranges.
MIN_DEMO_TIMESTAMP = "2000-01-01"
MAX_PACKAGE_WEIGHT_OZ = 1120        # 70 lbs * 16 oz
MAX_PACKAGE_DIMENSION_IN = 130      # loose demo threshold
MAX_SHIPPING_DISTANCE_MILES = 3000  # loose US/Texas demo threshold
MAX_EMPLOYEE_HOURS = 80             # weekly/overtime-friendly threshold


numeric_date_range_rules = [
    # ----------------------------
    # customer
    # ----------------------------
    {
        "table_name": "customer",
        "rule_name": "annual_income_non_negative",
        "columns": ["annual_income"],
        "invalid_condition": "`annual_income` IS NOT NULL AND `annual_income` < 0",
        "severity": "HIGH",
        "description": "Customer annual_income should not be negative.",
    },
    {
        "table_name": "customer",
        "rule_name": "total_children_between_0_and_5",
        "columns": ["total_children"],
        "invalid_condition": "`total_children` IS NOT NULL AND (`total_children` < 0 OR `total_children` > 5)",
        "severity": "MEDIUM",
        "description": "Customer total_children should be between 0 and 5 for this demo.",
    },
    {
        "table_name": "customer",
        "rule_name": "birth_date_reasonable",
        "columns": ["birth_date"],
        "invalid_condition": "`birth_date` IS NOT NULL AND (`birth_date` < DATE '1900-01-01' OR `birth_date` > current_date())",
        "severity": "HIGH",
        "description": "Customer birth_date should be realistic and not in the future.",
    },

    # ----------------------------
    # employee
    # ----------------------------
    {
        "table_name": "employee",
        "rule_name": "salary_non_negative",
        "columns": ["salary"],
        "invalid_condition": "`salary` IS NOT NULL AND `salary` < 0",
        "severity": "HIGH",
        "description": "Employee salary should not be negative.",
    },
    {
        "table_name": "employee",
        "rule_name": "hours_worked_reasonable",
        "columns": ["hours_worked"],
        "invalid_condition": f"`hours_worked` IS NOT NULL AND (`hours_worked` < 0 OR `hours_worked` > {MAX_EMPLOYEE_HOURS})",
        "severity": "MEDIUM",
        "description": "Employee hours_worked should be within a reasonable weekly range.",
    },

    # ----------------------------
    # package
    # ----------------------------
    {
        "table_name": "package",
        "rule_name": "weight_positive_and_reasonable",
        "columns": ["weight_oz"],
        "invalid_condition": f"`weight_oz` IS NOT NULL AND (`weight_oz` <= 0 OR `weight_oz` > {MAX_PACKAGE_WEIGHT_OZ})",
        "severity": "HIGH",
        "description": "Package weight_oz should be positive and not above the demo maximum.",
    },
    {
        "table_name": "package",
        "rule_name": "length_positive_and_reasonable",
        "columns": ["length_in"],
        "invalid_condition": f"`length_in` IS NOT NULL AND (`length_in` <= 0 OR `length_in` > {MAX_PACKAGE_DIMENSION_IN})",
        "severity": "HIGH",
        "description": "Package length_in should be positive and reasonable.",
    },
    {
        "table_name": "package",
        "rule_name": "width_positive_and_reasonable",
        "columns": ["width_in"],
        "invalid_condition": f"`width_in` IS NOT NULL AND (`width_in` <= 0 OR `width_in` > {MAX_PACKAGE_DIMENSION_IN})",
        "severity": "HIGH",
        "description": "Package width_in should be positive and reasonable.",
    },
    {
        "table_name": "package",
        "rule_name": "height_positive_and_reasonable",
        "columns": ["height_in"],
        "invalid_condition": f"`height_in` IS NOT NULL AND (`height_in` <= 0 OR `height_in` > {MAX_PACKAGE_DIMENSION_IN})",
        "severity": "HIGH",
        "description": "Package height_in should be positive and reasonable.",
    },
    {
        "table_name": "package",
        "rule_name": "received_date_reasonable",
        "columns": ["received_date"],
        "invalid_condition": f"`received_date` IS NOT NULL AND (`received_date` < TIMESTAMP '{MIN_DEMO_TIMESTAMP}' OR `received_date` > current_timestamp() + INTERVAL 1 DAY)",
        "severity": "HIGH",
        "description": "Package received_date should be within the expected demo date range.",
    },

    # ----------------------------
    # package_movement
    # ----------------------------
    {
        "table_name": "package_movement",
        "rule_name": "event_timestamp_reasonable",
        "columns": ["event_timestamp"],
        "invalid_condition": f"`event_timestamp` IS NOT NULL AND (`event_timestamp` < TIMESTAMP '{MIN_DEMO_TIMESTAMP}' OR `event_timestamp` > current_timestamp() + INTERVAL 1 DAY)",
        "severity": "HIGH",
        "description": "Package movement event_timestamp should be within the expected demo date range.",
    },

    # ----------------------------
    # shippingdetails
    # ----------------------------
    {
        "table_name": "shippingdetails",
        "rule_name": "estimated_delivery_distance_non_negative_reasonable",
        "columns": ["estimated_delivery_distance"],
        "invalid_condition": f"`estimated_delivery_distance` IS NOT NULL AND (`estimated_delivery_distance` < 0 OR `estimated_delivery_distance` > {MAX_SHIPPING_DISTANCE_MILES})",
        "severity": "MEDIUM",
        "description": "Estimated delivery distance should not be negative or unreasonably large.",
    },
    {
        "table_name": "shippingdetails",
        "rule_name": "expected_delivery_date_reasonable",
        "columns": ["expected_delivery_date"],
        "invalid_condition": f"`expected_delivery_date` IS NOT NULL AND (`expected_delivery_date` < TIMESTAMP '{MIN_DEMO_TIMESTAMP}' OR `expected_delivery_date` > current_timestamp() + INTERVAL 365 DAY)",
        "severity": "MEDIUM",
        "description": "Expected delivery date should be within a reasonable demo range.",
    },
    {
        "table_name": "shippingdetails",
        "rule_name": "delivered_date_reasonable",
        "columns": ["delivered_date"],
        "invalid_condition": f"`delivered_date` IS NOT NULL AND (`delivered_date` < TIMESTAMP '{MIN_DEMO_TIMESTAMP}' OR `delivered_date` > current_timestamp() + INTERVAL 1 DAY)",
        "severity": "HIGH",
        "description": "Delivered date should not be unrealistically old or in the future.",
    },
    {
        "table_name": "shippingdetails",
        "rule_name": "delivered_date_not_before_expected_date",
        "columns": ["delivered_date", "expected_delivery_date"],
        "invalid_condition": "`delivered_date` IS NOT NULL AND `expected_delivery_date` IS NOT NULL AND `delivered_date` < `expected_delivery_date`",
        "severity": "LOW",
        "description": "Delivered date before expected date may be valid early delivery, but should be reviewed.",
    },

    # ----------------------------
    # shipping_cost
    # ----------------------------
    {
        "table_name": "shipping_cost",
        "rule_name": "actual_shipping_charge_non_negative",
        "columns": ["actual_shipping_charge"],
        "invalid_condition": "`actual_shipping_charge` IS NOT NULL AND `actual_shipping_charge` < 0",
        "severity": "HIGH",
        "description": "Actual shipping charge should not be negative.",
    },
    {
        "table_name": "shipping_cost",
        "rule_name": "material_cost_non_negative",
        "columns": ["material_cost"],
        "invalid_condition": "`material_cost` IS NOT NULL AND `material_cost` < 0",
        "severity": "HIGH",
        "description": "Material cost should not be negative.",
    },
    {
        "table_name": "shipping_cost",
        "rule_name": "transportation_cost_non_negative",
        "columns": ["transportation_cost"],
        "invalid_condition": "`transportation_cost` IS NOT NULL AND `transportation_cost` < 0",
        "severity": "HIGH",
        "description": "Transportation cost should not be negative.",
    },
    {
        "table_name": "shipping_cost",
        "rule_name": "charge_recorded_at_reasonable",
        "columns": ["charge_recorded_at"],
        "invalid_condition": f"`charge_recorded_at` IS NOT NULL AND (`charge_recorded_at` < TIMESTAMP '{MIN_DEMO_TIMESTAMP}' OR `charge_recorded_at` > current_timestamp() + INTERVAL 1 DAY)",
        "severity": "MEDIUM",
        "description": "Shipping charge recorded timestamp should be within the expected demo date range.",
    },

    # ----------------------------
    # refunds
    # ----------------------------
    {
        "table_name": "refunds",
        "rule_name": "refund_amount_non_negative",
        "columns": ["refund_amount"],
        "invalid_condition": "`refund_amount` IS NOT NULL AND `refund_amount` < 0",
        "severity": "HIGH",
        "description": "Refund amount should not be negative.",
    },
    {
        "table_name": "refunds",
        "rule_name": "refund_date_reasonable",
        "columns": ["refund_date"],
        "invalid_condition": f"`refund_date` IS NOT NULL AND (`refund_date` < TIMESTAMP '{MIN_DEMO_TIMESTAMP}' OR `refund_date` > current_timestamp() + INTERVAL 1 DAY)",
        "severity": "HIGH",
        "description": "Refund date should be within the expected demo date range.",
    },
    {
        "table_name": "refunds",
        "rule_name": "reviewed_at_not_before_refund_date",
        "columns": ["reviewed_at", "refund_date"],
        "invalid_condition": "`reviewed_at` IS NOT NULL AND `refund_date` IS NOT NULL AND `reviewed_at` < `refund_date`",
        "severity": "MEDIUM",
        "description": "Refund reviewed_at should not be before refund_date.",
    },
    {
        "table_name": "refunds",
        "rule_name": "paid_at_not_before_reviewed_at",
        "columns": ["paid_at", "reviewed_at"],
        "invalid_condition": "`paid_at` IS NOT NULL AND `reviewed_at` IS NOT NULL AND `paid_at` < `reviewed_at`",
        "severity": "HIGH",
        "description": "Refund paid_at should not be before reviewed_at.",
    },

    # ----------------------------
    # incident
    # ----------------------------
    {
        "table_name": "incident",
        "rule_name": "incident_date_reasonable",
        "columns": ["incident_date"],
        "invalid_condition": f"`incident_date` IS NOT NULL AND (`incident_date` < TIMESTAMP '{MIN_DEMO_TIMESTAMP}' OR `incident_date` > current_timestamp() + INTERVAL 1 DAY)",
        "severity": "HIGH",
        "description": "Incident date should be within the expected demo date range.",
    },
    {
        "table_name": "incident",
        "rule_name": "resolved_at_not_before_incident_date",
        "columns": ["resolved_at", "incident_date"],
        "invalid_condition": "`resolved_at` IS NOT NULL AND `incident_date` IS NOT NULL AND `resolved_at` < `incident_date`",
        "severity": "HIGH",
        "description": "Incident resolved_at should not be before incident_date.",
    },

    # ----------------------------
    # lockerassignment
    # ----------------------------
    {
        "table_name": "lockerassignment",
        "rule_name": "expires_at_not_before_assigned_at",
        "columns": ["expires_at", "assigned_at"],
        "invalid_condition": "`expires_at` IS NOT NULL AND `assigned_at` IS NOT NULL AND `expires_at` < `assigned_at`",
        "severity": "HIGH",
        "description": "Locker assignment expires_at should not be before assigned_at.",
    },
    {
        "table_name": "lockerassignment",
        "rule_name": "retrieved_at_not_before_assigned_at",
        "columns": ["retrieved_at", "assigned_at"],
        "invalid_condition": "`retrieved_at` IS NOT NULL AND `assigned_at` IS NOT NULL AND `retrieved_at` < `assigned_at`",
        "severity": "HIGH",
        "description": "Locker assignment retrieved_at should not be before assigned_at.",
    },

    # ----------------------------
    # zip_geo
    # ----------------------------
    {
        "table_name": "zip_geo",
        "rule_name": "latitude_texas_reasonable",
        "columns": ["latitude"],
        "invalid_condition": "`latitude` IS NOT NULL AND (`latitude` < 25 OR `latitude` > 37)",
        "severity": "MEDIUM",
        "description": "Latitude should be within a loose Texas bounding range.",
    },
    {
        "table_name": "zip_geo",
        "rule_name": "longitude_texas_reasonable",
        "columns": ["longitude"],
        "invalid_condition": "`longitude` IS NOT NULL AND (`longitude` < -107 OR `longitude` > -93)",
        "severity": "MEDIUM",
        "description": "Longitude should be within a loose Texas bounding range.",
    },

    # ----------------------------
    # sort_order fields
    # ----------------------------
    {
        "table_name": "incident_severity",
        "rule_name": "sort_order_positive",
        "columns": ["sort_order"],
        "invalid_condition": "`sort_order` IS NOT NULL AND `sort_order` < 0",
        "severity": "LOW",
        "description": "Sort order should not be negative.",
    },
    {
        "table_name": "incident_status",
        "rule_name": "sort_order_positive",
        "columns": ["sort_order"],
        "invalid_condition": "`sort_order` IS NOT NULL AND `sort_order` < 0",
        "severity": "LOW",
        "description": "Sort order should not be negative.",
    },
    {
        "table_name": "package_status",
        "rule_name": "sort_order_positive",
        "columns": ["sort_order"],
        "invalid_condition": "`sort_order` IS NOT NULL AND `sort_order` < 0",
        "severity": "LOW",
        "description": "Sort order should not be negative.",
    },
    {
        "table_name": "package_movement_event_type",
        "rule_name": "sort_order_positive",
        "columns": ["sort_order"],
        "invalid_condition": "`sort_order` IS NOT NULL AND `sort_order` < 0",
        "severity": "LOW",
        "description": "Sort order should not be negative.",
    },
]


In [0]:
# ============================================================
# Cell 2: Add generated rules for audit timestamps and tinyint flags
# ============================================================

generated_range_rules = []

for table_name, columns in expected_mysql_schema.items():
    # created_at / updated_at date bounds
    for timestamp_col in ["created_at", "updated_at"]:
        if timestamp_col in columns:
            generated_range_rules.append({
                "table_name": table_name,
                "rule_name": f"{timestamp_col}_reasonable",
                "columns": [timestamp_col],
                "invalid_condition": f"`{timestamp_col}` IS NOT NULL AND (`{timestamp_col}` < TIMESTAMP '{MIN_DEMO_TIMESTAMP}' OR `{timestamp_col}` > current_timestamp() + INTERVAL 1 DAY)",
                "severity": "MEDIUM",
                "description": f"{timestamp_col} should be within the expected demo timestamp range.",
            })

    # created_at <= updated_at
    if "created_at" in columns and "updated_at" in columns:
        generated_range_rules.append({
            "table_name": table_name,
            "rule_name": "updated_at_not_before_created_at",
            "columns": ["created_at", "updated_at"],
            "invalid_condition": "`created_at` IS NOT NULL AND `updated_at` IS NOT NULL AND `updated_at` < `created_at`",
            "severity": "MEDIUM",
            "description": "updated_at should not be before created_at.",
        })

    # Boolean-like tinyint fields should be 0/1 when stored numerically.
    for column_name, mysql_type in columns.items():
        mysql_type_lower = mysql_type.lower()

        if mysql_type_lower.startswith("tinyint(1)") or column_name.startswith("is_") or column_name.startswith("requires_") or column_name.startswith("allows_") or column_name.startswith("handles_") or column_name.startswith("blocks_"):
            generated_range_rules.append({
                "table_name": table_name,
                "rule_name": f"{column_name}_boolean_like_0_or_1",
                "columns": [column_name],
                "invalid_condition": f"`{column_name}` IS NOT NULL AND CAST(`{column_name}` AS INT) NOT IN (0, 1)",
                "severity": "MEDIUM",
                "description": f"{column_name} should behave like a boolean flag with values 0/1 or true/false.",
            })


all_numeric_date_range_rules = numeric_date_range_rules + generated_range_rules

print(f"Manual rules: {len(numeric_date_range_rules)}")
print(f"Generated rules: {len(generated_range_rules)}")
print(f"Total rules: {len(all_numeric_date_range_rules)}")

Manual rules: 33
Generated rules: 97
Total rules: 130


In [0]:
# ============================================================
# Cell 3: Run numeric/date range validation
# ============================================================

def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


def resolve_column_name(df, requested_column_name: str):
    """
    Return the actual column name from a DataFrame, preserving original casing.
    """
    lookup = {c.lower(): c for c in df.columns}
    return lookup.get(requested_column_name.lower())


def remap_condition_to_actual_columns(condition_sql: str, requested_to_actual: dict) -> str:
    """
    Replace backticked requested column names with backticked actual column names.
    Useful if actual table columns have different casing.
    """
    remapped = condition_sql

    for requested_col, actual_col in requested_to_actual.items():
        remapped = remapped.replace(f"`{requested_col}`", f"`{actual_col}`")

    return remapped


range_validation_results = []

# Group rules by table so each table is scanned once.
rules_by_table = {}

for rule in all_numeric_date_range_rules:
    rules_by_table.setdefault(rule["table_name"], []).append(rule)


for table_name, table_rules in rules_by_table.items():
    full_table_name = f"`{catalog}`.`{source_schema}`.`{table_name}`"

    if not table_exists(full_table_name):
        for rule in table_rules:
            range_validation_results.append({
                "table_name": table_name,
                "rule_name": rule["rule_name"],
                "severity": rule["severity"],
                "checked_columns": ", ".join(rule["columns"]),
                "total_rows": None,
                "invalid_rows": None,
                "status": "MISSING_TABLE",
                "description": rule["description"],
                "invalid_condition": rule["invalid_condition"],
            })
        continue

    df = spark.table(full_table_name)
    actual_columns_lower = {c.lower(): c for c in df.columns}

    valid_rules_for_table = []
    agg_exprs = [count(lit(1)).alias("__total_rows")]

    for idx, rule in enumerate(table_rules):
        missing_columns = [
            c for c in rule["columns"]
            if c.lower() not in actual_columns_lower
        ]

        if missing_columns:
            range_validation_results.append({
                "table_name": table_name,
                "rule_name": rule["rule_name"],
                "severity": rule["severity"],
                "checked_columns": ", ".join(rule["columns"]),
                "total_rows": df.count(),
                "invalid_rows": None,
                "status": "MISSING_COLUMN",
                "description": rule["description"],
                "invalid_condition": rule["invalid_condition"],
            })
            continue

        requested_to_actual = {
            requested_col: actual_columns_lower[requested_col.lower()]
            for requested_col in rule["columns"]
        }

        remapped_condition = remap_condition_to_actual_columns(
            rule["invalid_condition"],
            requested_to_actual
        )

        alias_name = f"rule_{idx}__invalid_rows"

        agg_exprs.append(
            spark_sum(
                when(expr(remapped_condition), 1).otherwise(0)
            ).alias(alias_name)
        )

        valid_rules_for_table.append({
            **rule,
            "alias_name": alias_name,
            "remapped_condition": remapped_condition,
        })

    if not valid_rules_for_table:
        continue

    agg_row = df.agg(*agg_exprs).collect()[0].asDict()
    total_rows = agg_row["__total_rows"]

    for rule in valid_rules_for_table:
        invalid_rows = agg_row[rule["alias_name"]] or 0

        range_validation_results.append({
            "table_name": table_name,
            "rule_name": rule["rule_name"],
            "severity": rule["severity"],
            "checked_columns": ", ".join(rule["columns"]),
            "total_rows": total_rows,
            "invalid_rows": invalid_rows,
            "status": "OK" if invalid_rows == 0 else "HAS_RANGE_ISSUE",
            "description": rule["description"],
            "invalid_condition": rule["remapped_condition"],
        })


range_validation_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("rule_name", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("checked_columns", StringType(), True),
    StructField("total_rows", LongType(), True),
    StructField("invalid_rows", LongType(), True),
    StructField("status", StringType(), True),
    StructField("description", StringType(), True),
    StructField("invalid_condition", StringType(), True),
])

range_validation_df = spark.createDataFrame(
    range_validation_results,
    schema=range_validation_schema
)

display(
    range_validation_df
    .orderBy(
        col("invalid_rows").desc_nulls_last(),
        "severity",
        "table_name",
        "rule_name"
    )
)

table_name,rule_name,severity,checked_columns,total_rows,invalid_rows,status,description,invalid_condition
employee,hours_worked_reasonable,MEDIUM,hours_worked,5000,5000,HAS_RANGE_ISSUE,Employee hours_worked should be within a reasonable weekly range.,`hours_worked` IS NOT NULL AND (`hours_worked` < 0 OR `hours_worked` > 80)
package,weight_positive_and_reasonable,HIGH,weight_oz,105454,260,HAS_RANGE_ISSUE,Package weight_oz should be positive and not above the demo maximum.,`weight_oz` IS NOT NULL AND (`weight_oz` <= 0 OR `weight_oz` > 1120)
package,height_positive_and_reasonable,HIGH,height_in,105454,24,HAS_RANGE_ISSUE,Package height_in should be positive and reasonable.,`height_in` IS NOT NULL AND (`height_in` <= 0 OR `height_in` > 130)
customer,annual_income_non_negative,HIGH,annual_income,96096,0,OK,Customer annual_income should not be negative.,`annual_income` IS NOT NULL AND `annual_income` < 0
customer,birth_date_reasonable,HIGH,birth_date,96096,0,OK,Customer birth_date should be realistic and not in the future.,`birth_date` IS NOT NULL AND (`birth_date` < DATE '1900-01-01' OR `birth_date` > current_date())
employee,salary_non_negative,HIGH,salary,5000,0,OK,Employee salary should not be negative.,`salary` IS NOT NULL AND `salary` < 0
incident,incident_date_reasonable,HIGH,incident_date,6286,0,OK,Incident date should be within the expected demo date range.,`incident_date` IS NOT NULL AND (`incident_date` < TIMESTAMP '2000-01-01' OR `incident_date` > current_timestamp() + INTERVAL 1 DAY)
incident,resolved_at_not_before_incident_date,HIGH,"resolved_at, incident_date",6286,0,OK,Incident resolved_at should not be before incident_date.,`resolved_at` IS NOT NULL AND `incident_date` IS NOT NULL AND `resolved_at` < `incident_date`
lockerassignment,expires_at_not_before_assigned_at,HIGH,"expires_at, assigned_at",8117,0,OK,Locker assignment expires_at should not be before assigned_at.,`expires_at` IS NOT NULL AND `assigned_at` IS NOT NULL AND `expires_at` < `assigned_at`
lockerassignment,retrieved_at_not_before_assigned_at,HIGH,"retrieved_at, assigned_at",8117,0,OK,Locker assignment retrieved_at should not be before assigned_at.,`retrieved_at` IS NOT NULL AND `assigned_at` IS NOT NULL AND `retrieved_at` < `assigned_at`


In [0]:
# ============================================================
# Cell 4: Problem-only range validation results
# ============================================================

display(
    range_validation_df
    .filter(col("status") != "OK")
    .orderBy(
        col("invalid_rows").desc_nulls_last(),
        "severity",
        "table_name",
        "rule_name"
    )
)

table_name,rule_name,severity,checked_columns,total_rows,invalid_rows,status,description,invalid_condition
employee,hours_worked_reasonable,MEDIUM,hours_worked,5000,5000,HAS_RANGE_ISSUE,Employee hours_worked should be within a reasonable weekly range.,`hours_worked` IS NOT NULL AND (`hours_worked` < 0 OR `hours_worked` > 80)
package,weight_positive_and_reasonable,HIGH,weight_oz,105454,260,HAS_RANGE_ISSUE,Package weight_oz should be positive and not above the demo maximum.,`weight_oz` IS NOT NULL AND (`weight_oz` <= 0 OR `weight_oz` > 1120)
package,height_positive_and_reasonable,HIGH,height_in,105454,24,HAS_RANGE_ISSUE,Package height_in should be positive and reasonable.,`height_in` IS NOT NULL AND (`height_in` <= 0 OR `height_in` > 130)


In [0]:
# ============================================================
# Cell 5: Numeric/date range summary by table
# ============================================================

range_summary_df = (
    range_validation_df
    .groupBy("table_name")
    .agg(
        count(lit(1)).alias("rules_checked"),
        spark_sum(when(col("status") == "OK", 1).otherwise(0)).alias("rules_passed"),
        spark_sum(when(col("status") == "HAS_RANGE_ISSUE", 1).otherwise(0)).alias("rules_failed"),
        spark_sum(when(col("severity") == "HIGH", col("invalid_rows")).otherwise(0)).alias("high_severity_invalid_rows"),
        spark_sum(when(col("severity") == "MEDIUM", col("invalid_rows")).otherwise(0)).alias("medium_severity_invalid_rows"),
        spark_sum(when(col("severity") == "LOW", col("invalid_rows")).otherwise(0)).alias("low_severity_invalid_rows"),
        spark_sum(col("invalid_rows")).alias("total_invalid_rows"),
    )
    .orderBy(
        col("total_invalid_rows").desc_nulls_last(),
        col("rules_failed").desc_nulls_last(),
        "table_name"
    )
)

display(range_summary_df)

table_name,rules_checked,rules_passed,rules_failed,high_severity_invalid_rows,medium_severity_invalid_rows,low_severity_invalid_rows,total_invalid_rows
employee,5,4,1,0,5000,0,5000
package,8,6,2,284,0,0,284
business,3,3,0,0,0,0,0
customer,6,6,0,0,0,0,0
department_type,4,4,0,0,0,0,0
departments,3,3,0,0,0,0,0
email_queue,2,2,0,0,0,0,0
facility,3,3,0,0,0,0,0
facility_type,9,9,0,0,0,0,0
facility_type_department_rule,1,1,0,0,0,0,0


In [0]:
# ============================================================
# Cell 6: Inspect sample rows for one failed range rule
# ============================================================

table_to_inspect = "package"
rule_to_inspect = "weight_positive_and_reasonable"

selected_rule_row = (
    range_validation_df
    .filter(
        (col("table_name") == table_to_inspect)
        & (col("rule_name") == rule_to_inspect)
    )
    .collect()
)

if not selected_rule_row:
    print(f"No rule found for {table_to_inspect}.{rule_to_inspect}")
else:
    selected_rule = selected_rule_row[0].asDict()

    if selected_rule["status"] == "OK":
        print("Rule passed. No invalid rows to inspect.")
    else:
        df = spark.table(f"`{catalog}`.`{source_schema}`.`{table_to_inspect}`")

        display(
            df
            .filter(expr(selected_rule["invalid_condition"]))
            .limit(100)
        )

package_id,package_status_id,service_type_id,received_date,contents,weight_oz,length_in,width_in,height_in,created_at,updated_at,recipient_customer_id,package_flow_type_id,sender_customer_id,sender_business_id
AA/3kqCJWYSxG3A6mt6V1w==,27,1,2017-07-13T19:40:21.000Z,office_furniture,1345.140000000000000000,19.690000000000000000,20.470000000000000000,39.370000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:26:42.000Z,x34oEghztZCcCNpBoWQG2g==,2,null,fGfhRIsA9ulp02XOprAQqw==
AJJRR9cVWhaCyPg2llZBQQ==,27,1,2018-03-27T17:19:28.000Z,office_furniture,2778.180000000000000000,24.800000000000000000,11.020000000000000000,111.420000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:36:48.000Z,Av867mNqnuHB/vJx2qC8Bw==,2,null,fGfhRIsA9ulp02XOprAQqw==
AtIh3c7VVRGfDmKkqSqUiA==,29,3,2017-11-01T22:25:54.000Z,garden_tools,1830.720000000000000000,10.240000000000000000,10.240000000000000000,30.710000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:28:18.000Z,oUyzRyvyz7AfhIdK/7UEwg==,2,null,oQQ7r9Rx3/U20MRiNSvrSA==
BC4BA0+bUX28pNJvKHzWiQ==,27,1,2017-11-22T09:52:07.000Z,housewares,2254.010000000000000000,16.540000000000000000,16.540000000000000000,181.890000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:29:58.000Z,eLI9X/B+vi8IJHLKJCuSOA==,2,null,yjvXzZ8UnfdZUBUNAQ/kog==
BTcF1SzGW2WRDSceL4VTxQ==,27,1,2017-11-24T23:44:45.000Z,bed_bath_table; home_confort,1955.940000000000000000,17.720000000000000000,13.780000000000000000,17.720000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:29:58.000Z,bGmQUkWhCeUjhI3ik4bg/w==,2,null,SjypMVt0TOn46TdDYUk4hA==
Bd4ztmneEfGwVWBFveO5dg==,7,1,2026-06-13T23:49:08.000Z,furniture_decor 2432,1137.590000000000000000,13.780000000000000000,13.780000000000000000,59.060000000000000000,2026-06-16T23:49:32.000Z,2026-06-17T01:31:42.000Z,OsJlIpcEOd78+7l+d9AjBQ==,2,null,KiYbW2RPoF9PJwDrk1RPLA==
BmqW9hhUU/uPGxKdQrQEEQ==,27,1,2017-10-27T12:38:18.000Z,office_furniture,1140.550000000000000000,25.200000000000000000,9.840000000000000000,48.820000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:28:18.000Z,J4grTt4rsrEfvZqqqXCmZg==,2,null,fGfhRIsA9ulp02XOprAQqw==
Bn8kxL8zU7WjiZVznh779A==,29,3,2017-10-23T08:59:20.000Z,office_furniture,2998.290000000000000000,24.800000000000000000,9.450000000000000000,114.170000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:28:18.000Z,Msi3kpForv8FfcsiVaPe3w==,2,null,fGfhRIsA9ulp02XOprAQqw==
BxssyQzkXh+zUva73Nz5fA==,27,1,2018-06-11T21:17:41.000Z,computers,1671.990000000000000000,19.690000000000000000,9.840000000000000000,78.740000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:38:25.000Z,voJd3TtA2z+RvwW06UNdVg==,2,null,doHvFC/SwZBI2nQwhWtViA==
Byq/IWbjW3+Xl8LuY/krTA==,27,1,2017-05-23T19:24:53.000Z,bed_bath_table,1206.370000000000000000,22.050000000000000000,18.900000000000000000,29.920000000000000000,2026-06-11T01:57:19.000Z,2026-06-15T01:23:23.000Z,voxw9wfG09Bv8yAnFp+o2g==,2,null,QbOeKNsAXZcx2dSFqDtMOA==


In [0]:
# ============================================================
# Cell 7: Optional cross-table date checks
# ============================================================

cross_table_date_results = []

cross_table_date_rules = [
    {
        "rule_name": "package_movement_event_not_before_package_received_date",
        "left_table": "package_movement",
        "right_table": "package",
        "join_condition": "pm.package_id = p.package_id",
        "invalid_condition": "pm.event_timestamp IS NOT NULL AND p.received_date IS NOT NULL AND pm.event_timestamp < p.received_date",
        "description": "Package movement event_timestamp should not be before the package received_date.",
        "severity": "HIGH",
        "sql": f"""
            SELECT COUNT(*) AS invalid_rows
            FROM `{catalog}`.`{source_schema}`.`package_movement` pm
            INNER JOIN `{catalog}`.`{source_schema}`.`package` p
                ON pm.package_id = p.package_id
            WHERE pm.event_timestamp IS NOT NULL
              AND p.received_date IS NOT NULL
              AND pm.event_timestamp < p.received_date
        """,
    },
    {
        "rule_name": "shippingdetails_expected_delivery_not_before_package_received_date",
        "left_table": "shippingdetails",
        "right_table": "package",
        "join_condition": "sd.package_id = p.package_id",
        "invalid_condition": "sd.expected_delivery_date IS NOT NULL AND p.received_date IS NOT NULL AND sd.expected_delivery_date < p.received_date",
        "description": "Expected delivery date should not be before the package received_date.",
        "severity": "HIGH",
        "sql": f"""
            SELECT COUNT(*) AS invalid_rows
            FROM `{catalog}`.`{source_schema}`.`shippingdetails` sd
            INNER JOIN `{catalog}`.`{source_schema}`.`package` p
                ON sd.package_id = p.package_id
            WHERE sd.expected_delivery_date IS NOT NULL
              AND p.received_date IS NOT NULL
              AND sd.expected_delivery_date < p.received_date
        """,
    },
    {
        "rule_name": "shippingdetails_delivered_date_not_before_package_received_date",
        "left_table": "shippingdetails",
        "right_table": "package",
        "join_condition": "sd.package_id = p.package_id",
        "invalid_condition": "sd.delivered_date IS NOT NULL AND p.received_date IS NOT NULL AND sd.delivered_date < p.received_date",
        "description": "Delivered date should not be before the package received_date.",
        "severity": "HIGH",
        "sql": f"""
            SELECT COUNT(*) AS invalid_rows
            FROM `{catalog}`.`{source_schema}`.`shippingdetails` sd
            INNER JOIN `{catalog}`.`{source_schema}`.`package` p
                ON sd.package_id = p.package_id
            WHERE sd.delivered_date IS NOT NULL
              AND p.received_date IS NOT NULL
              AND sd.delivered_date < p.received_date
        """,
    },
    {
        "rule_name": "incident_date_not_before_package_received_date",
        "left_table": "incident",
        "right_table": "package",
        "join_condition": "i.package_id = p.package_id",
        "invalid_condition": "i.incident_date IS NOT NULL AND p.received_date IS NOT NULL AND i.incident_date < p.received_date",
        "description": "Incident date should not be before the package received_date.",
        "severity": "MEDIUM",
        "sql": f"""
            SELECT COUNT(*) AS invalid_rows
            FROM `{catalog}`.`{source_schema}`.`incident` i
            INNER JOIN `{catalog}`.`{source_schema}`.`package` p
                ON i.package_id = p.package_id
            WHERE i.incident_date IS NOT NULL
              AND p.received_date IS NOT NULL
              AND i.incident_date < p.received_date
        """,
    },
    {
        "rule_name": "refund_date_not_before_incident_date",
        "left_table": "refunds",
        "right_table": "incident",
        "join_condition": "r.incident_id = i.incident_id",
        "invalid_condition": "r.refund_date IS NOT NULL AND i.incident_date IS NOT NULL AND r.refund_date < i.incident_date",
        "description": "Refund date should not be before the related incident date.",
        "severity": "HIGH",
        "sql": f"""
            SELECT COUNT(*) AS invalid_rows
            FROM `{catalog}`.`{source_schema}`.`refunds` r
            INNER JOIN `{catalog}`.`{source_schema}`.`incident` i
                ON r.incident_id = i.incident_id
            WHERE r.refund_date IS NOT NULL
              AND i.incident_date IS NOT NULL
              AND r.refund_date < i.incident_date
        """,
    },
]

for rule in cross_table_date_rules:
    left_table_name = quote_full_table(catalog, source_schema, rule["left_table"])
    right_table_name = quote_full_table(catalog, source_schema, rule["right_table"])

    missing_tables = []

    if not table_exists(left_table_name):
        missing_tables.append(left_table_name)

    if not table_exists(right_table_name):
        missing_tables.append(right_table_name)

    if missing_tables:
        cross_table_date_results.append({
            "rule_name": rule["rule_name"],
            "left_table": rule["left_table"],
            "right_table": rule["right_table"],
            "severity": rule["severity"],
            "invalid_rows": None,
            "status": "MISSING_TABLE",
            "description": f"Skipped because required table(s) do not exist: {', '.join(missing_tables)}",
        })
        continue

    try:
        invalid_rows = spark.sql(rule["sql"]).collect()[0]["invalid_rows"]

        cross_table_date_results.append({
            "rule_name": rule["rule_name"],
            "left_table": rule["left_table"],
            "right_table": rule["right_table"],
            "severity": rule["severity"],
            "invalid_rows": invalid_rows,
            "status": "OK" if invalid_rows == 0 else "HAS_CROSS_TABLE_DATE_ISSUE",
            "description": rule["description"],
        })

    except Exception as e:
        cross_table_date_results.append({
            "rule_name": rule["rule_name"],
            "left_table": rule["left_table"],
            "right_table": rule["right_table"],
            "severity": rule["severity"],
            "invalid_rows": None,
            "status": "ERROR",
            "description": str(e),
        })


cross_table_date_schema = StructType([
    StructField("rule_name", StringType(), True),
    StructField("left_table", StringType(), True),
    StructField("right_table", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("invalid_rows", LongType(), True),
    StructField("status", StringType(), True),
    StructField("description", StringType(), True),
])

cross_table_date_df = spark.createDataFrame(
    cross_table_date_results,
    schema=cross_table_date_schema
)

display(
    cross_table_date_df
    .orderBy(
        col("invalid_rows").desc_nulls_last(),
        "severity",
        "rule_name"
    )
)


rule_name,left_table,right_table,severity,invalid_rows,status,description
package_movement_event_not_before_package_received_date,package_movement,package,HIGH,0,OK,Package movement event_timestamp should not be before the package received_date.
refund_date_not_before_incident_date,refunds,incident,HIGH,0,OK,Refund date should not be before the related incident date.
shippingdetails_delivered_date_not_before_package_received_date,shippingdetails,package,HIGH,0,OK,Delivered date should not be before the package received_date.
shippingdetails_expected_delivery_not_before_package_received_date,shippingdetails,package,HIGH,0,OK,Expected delivery date should not be before the package received_date.
incident_date_not_before_package_received_date,incident,package,MEDIUM,0,OK,Incident date should not be before the package received_date.


### Validate lookup/enumeration values

In [0]:
# ============================================================
# Silver Validation Check #7: Lookup / Enumeration Value Validation
# Cell 1: Config
#
# Assumes:
#   - catalog = "postal_databricks"
#   - source_schema = "bronze"
#
# This validates:
#   1. Literal enum/domain columns
#   2. Lookup/reference tables with expected canonical values
# ============================================================

from difflib import get_close_matches

from pyspark.sql.functions import (
    col,
    lit,
    count,
    desc,
    trim,
    lower,
    regexp_replace,
    collect_set,
    concat_ws,
    sum as spark_sum,
    when,
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


def resolve_column_name(df, requested_column_name: str):
    lookup = {c.lower(): c for c in df.columns}
    return lookup.get(requested_column_name.lower())


def normalize_python_string(value):
    if value is None:
        return None

    return " ".join(str(value).strip().lower().split())


def canonical_lookup(values):
    return {
        normalize_python_string(v): v
        for v in values
    }


def alias_lookup(aliases):
    return {
        normalize_python_string(k): v
        for k, v in aliases.items()
    }


def classify_enum_value(raw_value, allowed_values, aliases=None, allow_null=True):
    """
    Classifies one raw value against a controlled enum/domain list.
    """
    aliases = aliases or {}

    if raw_value is None:
        if allow_null:
            return ("NULL_ALLOWED", None)
        return ("NULL_NOT_ALLOWED", None)

    raw_text = str(raw_value)
    normalized_raw = normalize_python_string(raw_text)

    if normalized_raw == "":
        return ("BLANK_STRING", None)

    allowed = canonical_lookup(allowed_values)
    alias_map = alias_lookup(aliases)

    if normalized_raw in allowed:
        canonical_value = allowed[normalized_raw]

        if raw_text == canonical_value:
            return ("OK", canonical_value)

        return ("CASE_OR_SPACING_VARIANT", canonical_value)

    if normalized_raw in alias_map:
        return ("ALIAS_OR_ACRONYM_VARIANT", alias_map[normalized_raw])

    close_matches = get_close_matches(
        normalized_raw,
        list(allowed.keys()),
        n=1,
        cutoff=0.82
    )

    if close_matches:
        return ("POSSIBLE_TYPO", allowed[close_matches[0]])

    return ("OUTSIDE_ALLOWED_DOMAIN", None)


# Keep this bounded because enum/lookup fields should be low-cardinality.
max_distinct_values_per_column = 500


In [0]:
# ============================================================
# Cell 2: Literal enum/domain column rules
# ============================================================

enum_column_rules = {
    # ----------------------------
    # Geography / state fields
    # ----------------------------
    ("business", "state_code"): {
        "allowed": ["TX"],
        "aliases": {
            "texas": "TX",
            "tx.": "TX",
        },
        "allow_null": False,
    },
    ("customer", "state_code"): {
        "allowed": ["TX"],
        "aliases": {
            "texas": "TX",
            "tx.": "TX",
        },
        "allow_null": False,
    },
    ("facility", "state_code"): {
        "allowed": ["TX"],
        "aliases": {
            "texas": "TX",
            "tx.": "TX",
        },
        "allow_null": False,
    },
    ("territory", "state"): {
        "allowed": ["TX"],
        "aliases": {
            "texas": "TX",
            "tx.": "TX",
        },
        "allow_null": False,
    },

    # ----------------------------
    # Customer demographic fields
    # ----------------------------
    ("customer", "marital_status"): {
        "allowed": ["M", "S"],
        "aliases": {
            "married": "M",
            "m": "M",
            "single": "S",
            "s": "S",
        },
        "allow_null": True,
    },
    ("customer", "gender"): {
        "allowed": ["M", "F", "N"],
        "aliases": {
            "male": "M",
            "m": "M",
            "female": "F",
            "f": "F",
            "n/a": "N",
            "na": "N",
            "not applicable": "N",
            "none": "N",
            "unknown": "N",
            "n": "N",
        },
        "allow_null": True,
    },
    ("customer", "education_level"): {
        "allowed": [
            "Partial High School",
            "High School",
            "Partial College",
            "Bachelors",
            "Graduate Degree",
        ],
        "aliases": {
            "bachelor": "Bachelors",
            "bachelor's": "Bachelors",
            "bachelors degree": "Bachelors",
            "graduate": "Graduate Degree",
            "grad degree": "Graduate Degree",
            "partial hs": "Partial High School",
            "hs": "High School",
            "some college": "Partial College",
        },
        "allow_null": True,
    },
    ("customer", "occupation"): {
        "allowed": [
            "Skilled Manual",
            "Professional",
            "Management",
            "Clerical",
            "Manual",
        ],
        "aliases": {
            "manager": "Management",
            "mgmt": "Management",
            "professional worker": "Professional",
            "skilled labor": "Skilled Manual",
            "manual labor": "Manual",
            "office": "Clerical",
        },
        "allow_null": True,
    },
    ("customer", "home_owner"): {
        "allowed": ["Y", "N"],
        "aliases": {
            "yes": "Y",
            "y": "Y",
            "no": "N",
            "n": "N",
            "true": "Y",
            "1": "Y",
            "false": "N",
            "0": "N",
            "homeowner": "Y",
            "renter": "N",
        },
        "allow_null": True,
    },

    # ----------------------------
    # MySQL enum columns
    # ----------------------------
    ("smartlocker", "locker_status"): {
        "allowed": ["Available", "Occupied", "Out of service"],
        "aliases": {
            "open": "Available",
            "free": "Available",
            "vacant": "Available",
            "in use": "Occupied",
            "used": "Occupied",
            "out-of-service": "Out of service",
            "out of service": "Out of service",
            "broken": "Out of service",
            "maintenance": "Out of service",
        },
        "allow_null": False,
    },

    ("refunds", "refund_status"): {
        "allowed": ["Pending", "Approved", "Rejected", "Paid", "Cancelled"],
        "aliases": {
            "approve": "Approved",
            "approved refund": "Approved",
            "denied": "Rejected",
            "deny": "Rejected",
            "rejected refund": "Rejected",
            "complete": "Paid",
            "completed": "Paid",
            "paid out": "Paid",
            "canceled": "Cancelled",
            "cancelled refund": "Cancelled",
        },
        "allow_null": False,
    },

    ("refunds", "refund_reason"): {
        "allowed": [
            "Late Delivery",
            "Damaged Package",
            "Lost Package",
            "Returned Package",
            "Cancelled Package",
            "Service Failure",
            "Goodwill Adjustment",
            "Other",
        ],
        "aliases": {
            "delayed delivery": "Late Delivery",
            "late": "Late Delivery",
            "delay": "Late Delivery",
            "damaged": "Damaged Package",
            "damage": "Damaged Package",
            "lost": "Lost Package",
            "returned": "Returned Package",
            "return": "Returned Package",
            "canceled package": "Cancelled Package",
            "canceled": "Cancelled Package",
            "cancelled": "Cancelled Package",
            "service issue": "Service Failure",
            "service problem": "Service Failure",
            "goodwill": "Goodwill Adjustment",
        },
        "allow_null": False,
    },
}


In [0]:
# ============================================================
# Cell 3: Run literal enum/domain validation
# ============================================================

enum_validation_results = []

for (table_name, column_name), rule in enum_column_rules.items():
    full_table_name = f"`{catalog}`.`{source_schema}`.`{table_name}`"

    if not table_exists(full_table_name):
        enum_validation_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "raw_value": None,
            "row_count": None,
            "status": "MISSING_TABLE",
            "suggested_canonical_value": None,
            "allowed_values": ", ".join(rule["allowed"]),
        })
        continue

    df = spark.table(full_table_name)
    resolved_column = resolve_column_name(df, column_name)

    if resolved_column is None:
        enum_validation_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "raw_value": None,
            "row_count": None,
            "status": "MISSING_COLUMN",
            "suggested_canonical_value": None,
            "allowed_values": ", ".join(rule["allowed"]),
        })
        continue

    value_counts = (
        df
        .groupBy(col(resolved_column).alias("raw_value"))
        .agg(count(lit(1)).alias("row_count"))
        .orderBy(desc("row_count"))
        .limit(max_distinct_values_per_column)
        .collect()
    )

    for row in value_counts:
        raw_value = row["raw_value"]

        status, suggested_value = classify_enum_value(
            raw_value=raw_value,
            allowed_values=rule["allowed"],
            aliases=rule.get("aliases", {}),
            allow_null=rule.get("allow_null", True),
        )

        enum_validation_results.append({
            "table_name": table_name,
            "column_name": resolved_column,
            "raw_value": None if raw_value is None else str(raw_value),
            "row_count": row["row_count"],
            "status": status,
            "suggested_canonical_value": suggested_value,
            "allowed_values": ", ".join(rule["allowed"]),
        })


enum_validation_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("raw_value", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("suggested_canonical_value", StringType(), True),
    StructField("allowed_values", StringType(), True),
])

enum_validation_df = spark.createDataFrame(
    enum_validation_results,
    schema=enum_validation_schema
)

display(
    enum_validation_df
    .orderBy("table_name", "column_name", "status", desc("row_count"))
)

table_name,column_name,raw_value,row_count,status,suggested_canonical_value,allowed_values
business,state_code,TX,3095,OK,TX,TX
customer,education_level,Partial College,26079,OK,Partial College,"Partial High School, High School, Partial College, Bachelors, Graduate Degree"
customer,education_level,High School,22119,OK,High School,"Partial High School, High School, Partial College, Bachelors, Graduate Degree"
customer,education_level,Bachelors,21502,OK,Bachelors,"Partial High School, High School, Partial College, Bachelors, Graduate Degree"
customer,education_level,Partial High School,13752,OK,Partial High School,"Partial High School, High School, Partial College, Bachelors, Graduate Degree"
customer,education_level,Graduate Degree,12644,OK,Graduate Degree,"Partial High School, High School, Partial College, Bachelors, Graduate Degree"
customer,gender,F,48996,OK,F,"M, F, N"
customer,gender,M,47100,OK,M,"M, F, N"
customer,home_owner,Y,58472,OK,Y,"Y, N"
customer,home_owner,N,37624,OK,N,"Y, N"


In [0]:
# ============================================================
# Cell 4: Literal enum/domain problems only
# ============================================================

valid_enum_statuses = ["OK", "NULL_ALLOWED"]

display(
    enum_validation_df
    .filter(~col("status").isin(valid_enum_statuses))
    .orderBy("table_name", "column_name", "status", desc("row_count"))
)

table_name,column_name,raw_value,row_count,status,suggested_canonical_value,allowed_values


In [0]:
# ============================================================
# Cell 5: Required lookup/reference table values
# ============================================================

lookup_table_rules = {
    "facility_type": {
        "value_column": "facility_type_name",
        "required_values": [
            "Post Office",
            "Vehicle Maintenance",
            "Administrative Office",
            "Network Facilities",
            "Mail Processing",
        ],
        "aliases": {
            "po": "Post Office",
            "p.o.": "Post Office",
            "post offices": "Post Office",
            "vehicle maintenance facility": "Vehicle Maintenance",
            "admin office": "Administrative Office",
            "administrative": "Administrative Office",
            "network facility": "Network Facilities",
            "nf": "Network Facilities",
            "mail processing facility": "Mail Processing",
            "mp": "Mail Processing",
        },
        "allow_extra_values": False,
    },

    "department_type": {
        "value_column": "department_type_name",
        "required_values": [
            "Retail Services",
            "Delivery",
            "Logistics",
            "Operations",
            "Maintenance",
            "Administrative",
        ],
        "aliases": {
            "retail": "Retail Services",
            "retail service": "Retail Services",
            "delivery services": "Delivery",
            "logistic": "Logistics",
            "ops": "Operations",
            "operation": "Operations",
            "maint": "Maintenance",
            "admin": "Administrative",
            "administration": "Administrative",
        },
        "allow_extra_values": False,
    },

    "incident_severity": {
        "value_column": "severity_name",
        "required_values": [
            "Low",
            "Medium",
            "High",
            "Critical",
        ],
        "aliases": {
            "med": "Medium",
            "crit": "Critical",
        },
        "allow_extra_values": False,
    },

    "incident_status": {
        "value_column": "status_name",
        "required_values": [
            "Open",
            "Investigating",
            "Resolved",
            "Closed",
        ],
        "aliases": {
            "investigation": "Investigating",
            "in investigation": "Investigating",
            "resolve": "Resolved",
            "complete": "Closed",
            "completed": "Closed",
        },
        "allow_extra_values": False,
    },

    "incident_type": {
        "value_column": "type_name",
        "required_values": [
            "Damaged Package",
            "Lost Package",
            "Delayed Delivery",
            "Customer Complaint",
            "Employee Accident",
            "Tracking Error",
            "Facility Issue",
            "Cancelled Package",
            "Returned Package",
            "Other",
        ],
        "aliases": {
            "damage": "Damaged Package",
            "damaged": "Damaged Package",
            "lost": "Lost Package",
            "delay": "Delayed Delivery",
            "delayed": "Delayed Delivery",
            "late delivery": "Delayed Delivery",
            "complaint": "Customer Complaint",
            "customer issue": "Customer Complaint",
            "accident": "Employee Accident",
            "tracking": "Tracking Error",
            "facility problem": "Facility Issue",
            "canceled package": "Cancelled Package",
            "cancelled": "Cancelled Package",
            "canceled": "Cancelled Package",
            "returned": "Returned Package",
        },
        "allow_extra_values": False,
    },

    "package_flow_type": {
        "value_column": "package_flow_type_name",
        "required_values": [
            "B2C",
            "P2P",
        ],
        "aliases": {
            "business to customer": "B2C",
            "business-to-customer": "B2C",
            "business customer": "B2C",
            "peer to peer": "P2P",
            "peer-to-peer": "P2P",
            "customer to customer": "P2P",
            "customer-to-customer": "P2P",
        },
        "allow_extra_values": False,
    },
}

In [0]:
# ============================================================
# Cell 6: Run lookup/reference table validation
# ============================================================

lookup_validation_results = []

for table_name, rule in lookup_table_rules.items():
    full_table_name = f"`{catalog}`.`{source_schema}`.`{table_name}`"
    value_column = rule["value_column"]

    if not table_exists(full_table_name):
        for required_value in rule["required_values"]:
            lookup_validation_results.append({
                "lookup_table": table_name,
                "value_column": value_column,
                "lookup_value": required_value,
                "row_count": None,
                "status": "MISSING_TABLE",
                "suggested_canonical_value": required_value,
                "detail": f"Lookup table does not exist: {full_table_name}",
            })
        continue

    df = spark.table(full_table_name)
    resolved_value_column = resolve_column_name(df, value_column)

    if resolved_value_column is None:
        for required_value in rule["required_values"]:
            lookup_validation_results.append({
                "lookup_table": table_name,
                "value_column": value_column,
                "lookup_value": required_value,
                "row_count": None,
                "status": "MISSING_COLUMN",
                "suggested_canonical_value": required_value,
                "detail": f"Lookup value column does not exist: {table_name}.{value_column}",
            })
        continue

    actual_values = (
        df
        .groupBy(col(resolved_value_column).alias("lookup_value"))
        .agg(count(lit(1)).alias("row_count"))
        .collect()
    )

    actual_by_normalized = {
        normalize_python_string(row["lookup_value"]): {
            "raw_value": row["lookup_value"],
            "row_count": row["row_count"],
        }
        for row in actual_values
    }

    required_by_normalized = canonical_lookup(rule["required_values"])
    aliases = rule.get("aliases", {})

    # Check required canonical values.
    for required_normalized, required_canonical in required_by_normalized.items():
        if required_normalized in actual_by_normalized:
            actual_raw_value = actual_by_normalized[required_normalized]["raw_value"]
            row_count = actual_by_normalized[required_normalized]["row_count"]

            if actual_raw_value == required_canonical:
                status = "OK"
            else:
                status = "CASE_OR_SPACING_VARIANT"

            lookup_validation_results.append({
                "lookup_table": table_name,
                "value_column": resolved_value_column,
                "lookup_value": None if actual_raw_value is None else str(actual_raw_value),
                "row_count": row_count,
                "status": status,
                "suggested_canonical_value": required_canonical,
                "detail": None if status == "OK" else "Lookup value matches required value after normalization but not exactly.",
            })
        else:
            lookup_validation_results.append({
                "lookup_table": table_name,
                "value_column": resolved_value_column,
                "lookup_value": required_canonical,
                "row_count": 0,
                "status": "MISSING_REQUIRED_LOOKUP_VALUE",
                "suggested_canonical_value": required_canonical,
                "detail": "Required lookup value does not exist.",
            })

    # Check actual values for aliases, typos, and unexpected extras.
    for row in actual_values:
        raw_value = row["lookup_value"]
        row_count = row["row_count"]

        status, suggested_value = classify_enum_value(
            raw_value=raw_value,
            allowed_values=rule["required_values"],
            aliases=aliases,
            allow_null=False,
        )

        if status in ["OK", "CASE_OR_SPACING_VARIANT"]:
            continue

        if status == "OUTSIDE_ALLOWED_DOMAIN" and rule.get("allow_extra_values", False):
            output_status = "EXTRA_LOOKUP_VALUE_ALLOWED"
        elif status == "OUTSIDE_ALLOWED_DOMAIN":
            output_status = "UNEXPECTED_EXTRA_LOOKUP_VALUE"
        else:
            output_status = status

        lookup_validation_results.append({
            "lookup_table": table_name,
            "value_column": resolved_value_column,
            "lookup_value": None if raw_value is None else str(raw_value),
            "row_count": row_count,
            "status": output_status,
            "suggested_canonical_value": suggested_value,
            "detail": "Actual lookup value may need cleanup or review.",
        })


lookup_validation_schema = StructType([
    StructField("lookup_table", StringType(), True),
    StructField("value_column", StringType(), True),
    StructField("lookup_value", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("suggested_canonical_value", StringType(), True),
    StructField("detail", StringType(), True),
])

lookup_validation_df = spark.createDataFrame(
    lookup_validation_results,
    schema=lookup_validation_schema
)

display(
    lookup_validation_df
    .orderBy("lookup_table", "status", "lookup_value")
)

lookup_table,value_column,lookup_value,row_count,status,suggested_canonical_value,detail
department_type,department_type_name,Administrative,1,OK,Administrative,null
department_type,department_type_name,Delivery,1,OK,Delivery,null
department_type,department_type_name,Logistics,1,OK,Logistics,null
department_type,department_type_name,Maintenance,1,OK,Maintenance,null
department_type,department_type_name,Operations,1,OK,Operations,null
department_type,department_type_name,Retail Services,1,OK,Retail Services,null
facility_type,facility_type_name,Administrative Office,1,OK,Administrative Office,null
facility_type,facility_type_name,Mail Processing,1,OK,Mail Processing,null
facility_type,facility_type_name,Network Facilities,1,OK,Network Facilities,null
facility_type,facility_type_name,Post Office,1,OK,Post Office,null


In [0]:
# ============================================================
# Cell 7: Lookup/reference table problems only
# ============================================================

valid_lookup_statuses = ["OK", "EXTRA_LOOKUP_VALUE_ALLOWED"]

display(
    lookup_validation_df
    .filter(~col("status").isin(valid_lookup_statuses))
    .orderBy("lookup_table", "status", "lookup_value")
)

lookup_table,value_column,lookup_value,row_count,status,suggested_canonical_value,detail


In [0]:
# ============================================================
# Cell 8: Combined lookup/enumeration summary
# ============================================================

enum_summary_df = (
    enum_validation_df
    .withColumn("validation_type", lit("literal_enum_column"))
    .withColumn("object_name", col("table_name"))
    .withColumn("field_name", col("column_name"))
    .groupBy("validation_type", "object_name", "field_name", "status")
    .agg(
        spark_sum("row_count").alias("affected_rows")
    )
)

lookup_summary_df = (
    lookup_validation_df
    .withColumn("validation_type", lit("lookup_table"))
    .withColumn("object_name", col("lookup_table"))
    .withColumn("field_name", col("value_column"))
    .groupBy("validation_type", "object_name", "field_name", "status")
    .agg(
        spark_sum("row_count").alias("affected_rows")
    )
)

combined_lookup_enum_summary_df = enum_summary_df.unionByName(lookup_summary_df)

display(
    combined_lookup_enum_summary_df
    .orderBy("validation_type", "object_name", "field_name", "status")
)

validation_type,object_name,field_name,status,affected_rows
literal_enum_column,business,state_code,OK,3095
literal_enum_column,customer,education_level,OK,96096
literal_enum_column,customer,gender,OK,96096
literal_enum_column,customer,home_owner,OK,96096
literal_enum_column,customer,marital_status,OK,96096
literal_enum_column,customer,occupation,OK,96096
literal_enum_column,customer,state_code,OK,96096
literal_enum_column,facility,state_code,OK,541
literal_enum_column,refunds,refund_reason,OK,3417
literal_enum_column,refunds,refund_status,OK,3417


In [0]:
# ============================================================
# Cell 9: Problem-only combined lookup/enumeration summary
# ============================================================

valid_statuses = [
    "OK",
    "NULL_ALLOWED",
    "EXTRA_LOOKUP_VALUE_ALLOWED",
]

display(
    combined_lookup_enum_summary_df
    .filter(~col("status").isin(valid_statuses))
    .orderBy(
        desc("affected_rows"),
        "validation_type",
        "object_name",
        "field_name",
        "status"
    )
)

validation_type,object_name,field_name,status,affected_rows


### Add is_valid_record and dq_error_reason

In [0]:
# ============================================================
# Silver Transform #4: Add Record-Level DQ Fields
# Cell 1: Config
#
# Adds:
#   is_valid_record BOOLEAN
#   dq_error_reason STRING
#
# DQ-invalid rows stay in silver and are also copied to silver_quarantine.
# ============================================================

from datetime import datetime, timezone
import uuid
import re

from functools import reduce

from pyspark.sql import DataFrame, Window
from pyspark.sql.functions import (
    col,
    lit,
    when,
    concat_ws,
    length,
    trim,
    lower,
    upper,
    count as spark_count,
    sum as spark_sum,
    current_timestamp,
    split,
    explode,
    regexp_replace,
    current_date,
    current_timestamp as spark_current_timestamp,
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    BooleanType,
    NumericType,
    DateType,
    TimestampType,
)





# catalog = globals().get("catalog", "workspace")

dq_source_schema = globals().get(
    "dq_source_schema",
    globals().get("target_schema", "silver")
)

dq_target_schema = globals().get(
    "dq_target_schema",
    globals().get("target_schema", "silver")
)

dq_base_path = globals().get(
    "dq_base_path",
    globals().get("silver_base_path", None)
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{dq_target_schema}`")

DQ_FIELD_IS_VALID = "is_valid_record"
DQ_FIELD_ERROR_REASON = "dq_error_reason"
DQ_CAST_ERROR_FIELD = globals().get("DQ_CAST_ERROR_FIELD", "dq_cast_error_reason")

DQ_FIELDS = [
    DQ_FIELD_IS_VALID,
    DQ_FIELD_ERROR_REASON,
]

if "expected_mysql_schema" not in globals():
    raise ValueError("expected_mysql_schema was not found. Run your schema dictionary cell first.")

dq_schema_spec = expected_mysql_schema


In [0]:
# ============================================================
# Silver Transform #4
# Cell 2: Helpers
# ============================================================

def quote_identifier(identifier: str) -> str:
    return "`" + identifier.replace("`", "``") + "`"


def quote_full_table(catalog_name: str, schema_name: str, table_name: str) -> str:
    return (
        f"{quote_identifier(catalog_name)}."
        f"{quote_identifier(schema_name)}."
        f"{quote_identifier(table_name)}"
    )


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


def resolve_column_name(df: DataFrame, requested_column_name: str):
    lookup = {
        existing_column.lower(): existing_column
        for existing_column in df.columns
    }

    return lookup.get(requested_column_name.lower())


def drop_existing_dq_fields(df: DataFrame) -> DataFrame:
    columns_to_keep = [
        column_name
        for column_name in df.columns
        if column_name not in DQ_FIELDS
    ]

    return df.select(*[col(quote_identifier(column_name)) for column_name in columns_to_keep])


def write_delta_table(
    df: DataFrame,
    catalog_name: str,
    schema_name: str,
    table_name: str,
    base_path: str = None,
    mode: str = "overwrite",
):
    target_table = quote_full_table(catalog_name, schema_name, table_name)

    writer = (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
    )

    if base_path:
        table_path = f"{base_path.rstrip('/')}/{table_name}"
        writer = writer.option("path", table_path)

    writer.saveAsTable(target_table)


def prepare_df_for_same_table_overwrite(df: DataFrame) -> DataFrame:
    try:
        return df.localCheckpoint(eager=True)
    except Exception:
        materialized_df = df.cache()
        materialized_df.count()
        return materialized_df


def safe_write_dq_table(
    df: DataFrame,
    catalog_name: str,
    source_schema_name: str,
    target_schema_name: str,
    table_name: str,
    base_path: str = None,
):
    source_table = quote_full_table(catalog_name, source_schema_name, table_name)
    target_table = quote_full_table(catalog_name, target_schema_name, table_name)

    same_table = source_table == target_table
    prepared_df = prepare_df_for_same_table_overwrite(df) if same_table else df

    write_delta_table(
        df=prepared_df,
        catalog_name=catalog_name,
        schema_name=target_schema_name,
        table_name=table_name,
        base_path=base_path,
        mode="overwrite",
    )

    if same_table:
        try:
            prepared_df.unpersist()
        except Exception:
            pass


def or_all(conditions):
    return reduce(lambda left, right: left | right, conditions)


def and_all(conditions):
    return reduce(lambda left, right: left & right, conditions)


In [0]:
# ============================================================
# Silver Transform #4
# Cell 3: Primary key rule loader
# ============================================================

def get_primary_key_map_for_dq():
    """
    Reuses your notebook's existing primary key map.

    Priority:
      1. primary_key_map
      2. pk_map

    Expected final shape:
      {
          "table_name": ["pk_column_1", ...],
          ...
      }
    """
    if "primary_key_map" in globals() and isinstance(primary_key_map, dict):
        return primary_key_map

    if "pk_map" in globals():
        if isinstance(pk_map, dict):
            return pk_map

        return {
            table_name: pk_columns
            for table_name, pk_columns in pk_map
        }

    raise ValueError(
        "No primary key map found. Run your existing pk_map or primary_key_map cell first."
    )


dq_primary_key_map = get_primary_key_map_for_dq()

In [0]:
# ============================================================
# Silver Transform #4
# Cell 4: Foreign key rules
# ============================================================

dq_foreign_key_rules = globals().get("dq_foreign_key_rules", [])


def normalize_fk_rules():
    if "foreign_key_rules" in globals():
        raw_rules = foreign_key_rules
    elif dq_foreign_key_rules:
        raw_rules = dq_foreign_key_rules
    elif "foreign_key_map" in globals():
        raw_rules = [
            {
                "table": fk_rule["child_table"],
                "columns": [fk_rule["child_column"]],
                "ref_table": fk_rule["parent_table"],
                "ref_columns": [fk_rule["parent_column"]],
                "required": False,
            }
            for fk_rule in foreign_key_map
        ]
    else:
        raw_rules = []

    normalized_rules = []

    for rule in raw_rules:
        normalized_rules.append({
            "table": rule["table"],
            "columns": rule["columns"] if isinstance(rule["columns"], list) else [rule["columns"]],
            "ref_table": rule["ref_table"],
            "ref_columns": rule["ref_columns"] if isinstance(rule["ref_columns"], list) else [rule["ref_columns"]],
            "required": bool(rule.get("required", False)),
        })

    return normalized_rules


dq_fk_rules = normalize_fk_rules()


In [0]:
# ============================================================
# Silver Transform #4
# Cell 5: Domain / lookup rules
# ============================================================

APPLY_DEFAULT_DQ_DOMAIN_RULES = True
dq_domain_rules = globals().get("dq_domain_rules", [])

DEFAULT_DQ_DOMAIN_RULES = [
    {
        "table": "*",
        "column": "state_code",
        "allowed_values": ["TX"],
        "required": False,
    },
    {
        "table": "*",
        "column": "state",
        "allowed_values": ["TX"],
        "required": False,
    },
    {
        "table": "customer",
        "column": "gender",
        "allowed_values": ["M", "F", "N"],
        "required": False,
    },
    {
        "table": "customer",
        "column": "home_owner",
        "allowed_values": ["Y", "N"],
        "required": False,
    },
    {
        "table": "customer",
        "column": "marital_status",
        "allowed_values": ["M", "S"],
        "required": False,
    },
    {
        "table": "*",
        "column": "education_level",
        "allowed_values": [
            "Partial High School",
            "High School",
            "Graduate Degree",
            "Partial College",
            "Bachelors",
        ],
        "required": False,
    },
    {
        "table": "*",
        "column": "occupation",
        "allowed_values": [
            "Skilled Manual",
            "Professional",
            "Management",
            "Clerical",
            "Manual",
        ],
        "required": False,
    },
    {
        "table": "*",
        "column": "package_flow_type",
        "allowed_values": ["B2C", "P2P"],
        "required": False,
    },
    {
        "table": "*",
        "column": "severity",
        "allowed_values": ["Low", "Medium", "High", "Critical"],
        "required": False,
    },
    {
        "table": "*",
        "column": "incident_status",
        "allowed_values": ["Open", "Investigating", "Resolved", "Closed"],
        "required": False,
    },
    {
        "table": "*",
        "column": "incident_type",
        "allowed_values": [
            "Damaged Package",
            "Lost Package",
            "Delayed Delivery",
            "Customer Complaint",
            "Employee Accident",
            "Tracking Error",
            "Facility Issue",
            "Cancelled Package",
            "Returned Package",
            "Other",
        ],
        "required": False,
    },
    {
        "table": "*",
        "column": "facility_type_name",
        "allowed_values": [
            "Post Office",
            "Vehicle Maintenance",
            "Administrative Office",
            "Network Facilities",
            "Mail Processing",
        ],
        "required": False,
    },
]


def normalize_domain_rules():
    rules = []

    if APPLY_DEFAULT_DQ_DOMAIN_RULES:
        rules.extend(DEFAULT_DQ_DOMAIN_RULES)

    rules.extend(dq_domain_rules)

    normalized_rules = []

    for rule in rules:
        normalized_rules.append({
            "table": rule.get("table", "*"),
            "column": rule["column"],
            "allowed_values": rule["allowed_values"],
            "required": bool(rule.get("required", False)),
        })

    return normalized_rules


dq_all_domain_rules = normalize_domain_rules()


def get_domain_rules_for_table(table_name: str, df: DataFrame):
    applicable_rules = []

    for rule in dq_all_domain_rules:
        if rule["table"] not in {"*", table_name}:
            continue

        actual_column = resolve_column_name(df, rule["column"])

        if actual_column is None:
            continue

        applicable_rules.append({
            **rule,
            "actual_column": actual_column,
        })

    return applicable_rules


In [0]:
# ============================================================
# Silver Transform #4
# Cell 6: Numeric/date range rules
# ============================================================

APPLY_DEFAULT_DQ_RANGE_RULES = True
dq_range_rules = globals().get("dq_range_rules", [])

DEFAULT_DQ_RANGE_RULES = [
    {
        "table": "customer",
        "column": "total_children",
        "min_value": 0,
        "max_value": 5,
        "required": False,
    },
    {
        "table": "customer",
        "column": "annual_income",
        "min_value": 0,
        "max_value": None,
        "required": False,
    },
    {
        "table": "customer",
        "column": "birth_date",
        "min_value": None,
        "max_value": "CURRENT_DATE",
        "required": False,
    },
    {
        "table": "package",
        "column": "weight_oz",
        "min_value": 0,
        "max_value": None,
        "required": False,
    },
    {
        "table": "package",
        "column": "length_in",
        "min_value": 0,
        "max_value": None,
        "required": False,
    },
    {
        "table": "package",
        "column": "width_in",
        "min_value": 0,
        "max_value": None,
        "required": False,
    },
    {
        "table": "package",
        "column": "height_in",
        "min_value": 0,
        "max_value": None,
        "required": False,
    },
    {
        "table": "*",
        "column": "latitude",
        "min_value": -90,
        "max_value": 90,
        "required": False,
    },
    {
        "table": "*",
        "column": "longitude",
        "min_value": -180,
        "max_value": 180,
        "required": False,
    },
]


def normalize_range_rules():
    rules = []

    if APPLY_DEFAULT_DQ_RANGE_RULES:
        rules.extend(DEFAULT_DQ_RANGE_RULES)

    rules.extend(dq_range_rules)

    normalized_rules = []

    for rule in rules:
        normalized_rules.append({
            "table": rule.get("table", "*"),
            "column": rule["column"],
            "min_value": rule.get("min_value", None),
            "max_value": rule.get("max_value", None),
            "required": bool(rule.get("required", False)),
        })

    return normalized_rules


dq_all_range_rules = normalize_range_rules()


def get_range_rules_for_table(table_name: str, df: DataFrame):
    applicable_rules = []

    for rule in dq_all_range_rules:
        if rule["table"] not in {"*", table_name}:
            continue

        actual_column = resolve_column_name(df, rule["column"])

        if actual_column is None:
            continue

        applicable_rules.append({
            **rule,
            "actual_column": actual_column,
        })

    return applicable_rules


In [0]:
# ============================================================
# Silver Transform #4
# Cell 7: Build record-level DQ error expressions
# ============================================================

def build_missing_expected_column_error_expressions(df: DataFrame, table_name: str):
    error_expressions = []

    expected_columns = dq_schema_spec.get(table_name, {})

    for expected_column_name in expected_columns.keys():
        actual_column = resolve_column_name(df, expected_column_name)

        if actual_column is None:
            error_expressions.append(
                lit(f"MISSING_EXPECTED_COLUMN:{expected_column_name}")
            )

    return error_expressions


def add_pk_dq_work_columns(df: DataFrame, table_name: str):
    """
    Adds temporary helper columns for duplicate PK detection.

    Returns:
      df_with_helpers
      pk_error_expressions
      helper_columns_to_drop
    """
    pk_columns = dq_primary_key_map.get(table_name, [])
    helper_columns_to_drop = []
    error_expressions = []

    if not pk_columns:
        error_expressions.append(lit("MISSING_PK_RULE"))
        return df, error_expressions, helper_columns_to_drop

    resolved_pk_columns = []
    missing_pk_columns = []

    for pk_column in pk_columns:
        actual_column = resolve_column_name(df, pk_column)

        if actual_column is None:
            missing_pk_columns.append(pk_column)
        else:
            resolved_pk_columns.append(actual_column)

    if missing_pk_columns:
        error_expressions.append(
            lit(f"MISSING_PK_COLUMN:{','.join(missing_pk_columns)}")
        )

    if not resolved_pk_columns:
        return df, error_expressions, helper_columns_to_drop

    pk_null_condition = or_all([
        col(quote_identifier(pk_column)).isNull()
        for pk_column in resolved_pk_columns
    ])

    error_expressions.append(
        when(pk_null_condition, lit("NULL_PRIMARY_KEY")).otherwise(lit(None))
    )

    pk_window = Window.partitionBy(*[
        col(quote_identifier(pk_column))
        for pk_column in resolved_pk_columns
    ])

    pk_group_count_col = f"__dq_pk_group_count_{uuid.uuid4().hex[:8]}"

    df = df.withColumn(
        pk_group_count_col,
        spark_count(lit(1)).over(pk_window)
    )

    helper_columns_to_drop.append(pk_group_count_col)

    error_expressions.append(
        when(
            (~pk_null_condition) & (col(pk_group_count_col) > 1),
            lit("DUPLICATE_PRIMARY_KEY")
        ).otherwise(lit(None))
    )

    return df, error_expressions, helper_columns_to_drop


def add_fk_dq_work_columns(df: DataFrame, table_name: str):
    """
    Adds temporary helper columns for FK existence checks.

    Returns:
      df_with_helpers
      fk_error_expressions
      helper_columns_to_drop
    """
    error_expressions = []
    helper_columns_to_drop = []

    applicable_fk_rules = [
        rule
        for rule in dq_fk_rules
        if rule["table"] == table_name
    ]

    for rule_index, rule in enumerate(applicable_fk_rules):
        local_columns = []
        missing_local_columns = []

        for local_column in rule["columns"]:
            actual_column = resolve_column_name(df, local_column)

            if actual_column is None:
                missing_local_columns.append(local_column)
            else:
                local_columns.append(actual_column)

        if missing_local_columns:
            error_expressions.append(
                lit(
                    "MISSING_FK_COLUMN:"
                    + ",".join(missing_local_columns)
                    + f"->"
                    + rule["ref_table"]
                )
            )
            continue

        parent_table_name = rule["ref_table"]
        parent_full_table = quote_full_table(catalog, dq_source_schema, parent_table_name)

        if not table_exists(parent_full_table):
            error_expressions.append(
                lit(f"MISSING_PARENT_TABLE:{parent_table_name}")
            )
            continue

        parent_df = spark.table(parent_full_table)
        parent_df = drop_existing_dq_fields(parent_df)

        parent_columns = []
        missing_parent_columns = []

        for parent_column in rule["ref_columns"]:
            actual_parent_column = resolve_column_name(parent_df, parent_column)

            if actual_parent_column is None:
                missing_parent_columns.append(parent_column)
            else:
                parent_columns.append(actual_parent_column)

        if missing_parent_columns:
            error_expressions.append(
                lit(
                    "MISSING_PARENT_COLUMN:"
                    + parent_table_name
                    + "."
                    + ",".join(missing_parent_columns)
                )
            )
            continue

        local_not_null_condition = and_all([
            col(quote_identifier(local_column)).isNotNull()
            for local_column in local_columns
        ])

        if rule.get("required", False):
            local_any_null_condition = or_all([
                col(quote_identifier(local_column)).isNull()
                for local_column in local_columns
            ])

            error_expressions.append(
                when(
                    local_any_null_condition,
                    lit(
                        "NULL_REQUIRED_FK:"
                        + ",".join(local_columns)
                        + "->"
                        + parent_table_name
                        + "."
                        + ",".join(parent_columns)
                    )
                ).otherwise(lit(None))
            )

        parent_alias_columns = [
            f"__dq_fk_{rule_index}_parent_key_{i}_{uuid.uuid4().hex[:6]}"
            for i in range(len(parent_columns))
        ]

        parent_match_flag_col = f"__dq_fk_{rule_index}_matched_{uuid.uuid4().hex[:8]}"

        parent_key_df = (
            parent_df
            .select(*[
                col(quote_identifier(parent_column)).alias(parent_alias_columns[i])
                for i, parent_column in enumerate(parent_columns)
            ])
            .dropDuplicates()
            .withColumn(parent_match_flag_col, lit(1))
        )

        join_condition = and_all([
            col(quote_identifier(local_columns[i])) == col(parent_alias_columns[i])
            for i in range(len(local_columns))
        ])

        df = df.join(parent_key_df, join_condition, "left")

        helper_columns_to_drop.extend(parent_alias_columns)
        helper_columns_to_drop.append(parent_match_flag_col)

        error_expressions.append(
            when(
                local_not_null_condition & col(parent_match_flag_col).isNull(),
                lit(
                    "INVALID_FK:"
                    + ",".join(local_columns)
                    + "->"
                    + parent_table_name
                    + "."
                    + ",".join(parent_columns)
                )
            ).otherwise(lit(None))
        )

    return df, error_expressions, helper_columns_to_drop


def build_domain_error_expressions(df: DataFrame, table_name: str):
    error_expressions = []

    for rule in get_domain_rules_for_table(table_name, df):
        column_name = rule["actual_column"]
        allowed_values = rule["allowed_values"]

        value_col = col(quote_identifier(column_name))

        if rule.get("required", False):
            error_expressions.append(
                when(
                    value_col.isNull(),
                    lit(f"NULL_REQUIRED_DOMAIN:{column_name}")
                ).otherwise(lit(None))
            )

        error_expressions.append(
            when(
                value_col.isNotNull() & (~value_col.isin(allowed_values)),
                lit(f"INVALID_DOMAIN:{column_name}")
            ).otherwise(lit(None))
        )

    return error_expressions


def build_range_error_expressions(df: DataFrame, table_name: str):
    error_expressions = []

    for rule in get_range_rules_for_table(table_name, df):
        column_name = rule["actual_column"]
        value_col = col(quote_identifier(column_name))

        if rule.get("required", False):
            error_expressions.append(
                when(
                    value_col.isNull(),
                    lit(f"NULL_REQUIRED_RANGE:{column_name}")
                ).otherwise(lit(None))
            )

        range_condition = lit(False)

        if rule["min_value"] is not None:
            min_value = rule["min_value"]
            if min_value == "CURRENT_DATE":
                range_condition = range_condition | (value_col < current_date())
            elif min_value == "CURRENT_TIMESTAMP":
                range_condition = range_condition | (value_col < spark_current_timestamp())
            else:
                range_condition = range_condition | (value_col < lit(min_value))

        if rule["max_value"] is not None:
            max_value = rule["max_value"]
            if max_value == "CURRENT_DATE":
                range_condition = range_condition | (value_col > current_date())
            elif max_value == "CURRENT_TIMESTAMP":
                range_condition = range_condition | (value_col > spark_current_timestamp())
            else:
                range_condition = range_condition | (value_col > lit(max_value))

        error_expressions.append(
            when(
                value_col.isNotNull() & range_condition,
                lit(f"OUT_OF_RANGE:{column_name}")
            ).otherwise(lit(None))
        )

    return error_expressions


In [0]:
# ============================================================
# Silver Transform #4
# Cell 8: Apply DQ fields to one table
# ============================================================

def add_record_level_dq_fields(df: DataFrame, table_name: str):
    """
    Adds:
      is_valid_record
      dq_error_reason

    Canonical silver keeps all rows, including DQ-invalid rows.
    Duplicate-loser and NULL-PK rejects were already handled earlier.
    """
    df = drop_existing_dq_fields(df)

    all_error_expressions = []
    helper_columns_to_drop = []

    cast_error_column = resolve_column_name(df, DQ_CAST_ERROR_FIELD)
    if cast_error_column is not None:
        all_error_expressions.append(col(quote_identifier(cast_error_column)))

    all_error_expressions.extend(
        build_missing_expected_column_error_expressions(df, table_name)
    )

    df, pk_error_expressions, pk_helper_columns = add_pk_dq_work_columns(df, table_name)
    all_error_expressions.extend(pk_error_expressions)
    helper_columns_to_drop.extend(pk_helper_columns)

    df, fk_error_expressions, fk_helper_columns = add_fk_dq_work_columns(df, table_name)
    all_error_expressions.extend(fk_error_expressions)
    helper_columns_to_drop.extend(fk_helper_columns)

    all_error_expressions.extend(
        build_domain_error_expressions(df, table_name)
    )

    all_error_expressions.extend(
        build_range_error_expressions(df, table_name)
    )

    dq_error_expression = concat_ws("; ", *all_error_expressions) if all_error_expressions else lit("")

    dq_tagged_df = (
        df
        .withColumn(
            DQ_FIELD_ERROR_REASON,
            when(
                length(dq_error_expression) > 0,
                dq_error_expression
            ).otherwise(lit(None).cast(StringType()))
        )
        .withColumn(
            DQ_FIELD_IS_VALID,
            col(DQ_FIELD_ERROR_REASON).isNull()
        )
    )

    for helper_column in helper_columns_to_drop:
        if helper_column in dq_tagged_df.columns:
            dq_tagged_df = dq_tagged_df.drop(helper_column)

    original_columns = [
        column_name
        for column_name in dq_tagged_df.columns
        if column_name not in DQ_FIELDS
    ]

    dq_tagged_df = dq_tagged_df.select(
        *[col(quote_identifier(column_name)) for column_name in original_columns],
        col(DQ_FIELD_IS_VALID),
        col(DQ_FIELD_ERROR_REASON),
    )

    return dq_tagged_df


# END OF FILE
The following scripts below have temporarily been removed due to a performance bottleneck and will only be reimplemented after I figure out a more efficient method.

# ================TAKES WAY TOO LONG OVER 20 MINUTES======================
# Silver Transform #4
# Cell 9: Run record-level DQ tagging for every silver table
# ============================================================

dq_run_started_at = datetime.now(timezone.utc).isoformat()

dq_summary_rows = []

for table_name in dq_schema_spec.keys():
    source_table = quote_full_table(catalog, dq_source_schema, table_name)
    target_table = quote_full_table(catalog, dq_target_schema, table_name)

    if not table_exists(source_table):
        dq_summary_rows.append({
            "run_started_at": dq_run_started_at,
            "table_name": table_name,
            "source_table": source_table,
            "target_table": target_table,
            "total_rows": None,
            "valid_rows": None,
            "invalid_rows": None,
            "status": "MISSING_TABLE",
            "message": f"Source table does not exist: {source_table}",
        })
        continue

    try:
        source_df = spark.table(source_table)
        source_df = drop_existing_dq_fields(source_df)

        dq_tagged_df = add_record_level_dq_fields(
            df=source_df,
            table_name=table_name,
        )

        # Compute summary before writing.
        summary_stats = (
            dq_tagged_df
            .agg(
                spark_count(lit(1)).alias("total_rows"),
                spark_sum(
                    when(col(DQ_FIELD_IS_VALID) == True, lit(1)).otherwise(lit(0))
                ).alias("valid_rows"),
                spark_sum(
                    when(col(DQ_FIELD_IS_VALID) == False, lit(1)).otherwise(lit(0))
                ).alias("invalid_rows"),
            )
            .collect()[0]
            .asDict()
        )

        safe_write_dq_table(
            df=dq_tagged_df,
            catalog_name=catalog,
            source_schema_name=dq_source_schema,
            target_schema_name=dq_target_schema,
            table_name=table_name,
            base_path=dq_base_path,
        )

        dq_summary_rows.append({
            "run_started_at": dq_run_started_at,
            "table_name": table_name,
            "source_table": source_table,
            "target_table": target_table,
            "total_rows": summary_stats["total_rows"],
            "valid_rows": summary_stats["valid_rows"],
            "invalid_rows": summary_stats["invalid_rows"],
            "status": "OK",
            "message": "DQ fields added successfully.",
        })

    except Exception as error:
        dq_summary_rows.append({
            "run_started_at": dq_run_started_at,
            "table_name": table_name,
            "source_table": source_table,
            "target_table": target_table,
            "total_rows": None,
            "valid_rows": None,
            "invalid_rows": None,
            "status": "FAILED_DQ_TAGGING",
            "message": str(error),
        })


dq_summary_schema = StructType([
    StructField("run_started_at", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("total_rows", LongType(), True),
    StructField("valid_rows", LongType(), True),
    StructField("invalid_rows", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
])

silver_dq_record_validity_summary_df = spark.createDataFrame(
    dq_summary_rows,
    schema=dq_summary_schema,
)

write_delta_table(
    df=silver_dq_record_validity_summary_df,
    catalog_name=catalog,
    schema_name=dq_target_schema,
    table_name="_silver_dq_record_validity_summary",
    base_path=dq_base_path,
    mode="overwrite",
)

display(
    silver_dq_record_validity_summary_df
    .orderBy(
        col("status").desc(),
        col("invalid_rows").desc_nulls_last(),
        col("table_name"),
    )
)

# ============================================================
# Silver Transform #4
# Cell 10: DQ error reason breakdown
# ============================================================

dq_reason_breakdown_dfs = []

for table_name in dq_schema_spec.keys():
    table_full_name = quote_full_table(catalog, dq_target_schema, table_name)

    if not table_exists(table_full_name):
        continue

    df = spark.table(table_full_name)

    if DQ_FIELD_ERROR_REASON not in df.columns:
        continue

    reason_breakdown_df = (
        df
        .filter(col(DQ_FIELD_ERROR_REASON).isNotNull())
        .select(
            lit(table_name).alias("table_name"),
            explode(split(col(DQ_FIELD_ERROR_REASON), "; ")).alias("dq_error_reason")
        )
        .groupBy("table_name", "dq_error_reason")
        .agg(spark_count(lit(1)).alias("record_count"))
    )

    dq_reason_breakdown_dfs.append(reason_breakdown_df)


if dq_reason_breakdown_dfs:
    silver_dq_error_reason_summary_df = reduce(
        DataFrame.unionByName,
        dq_reason_breakdown_dfs,
    )
else:
    silver_dq_error_reason_summary_df = spark.createDataFrame(
        [],
        StructType([
            StructField("table_name", StringType(), True),
            StructField("dq_error_reason", StringType(), True),
            StructField("record_count", LongType(), True),
        ])
    )


write_delta_table(
    df=silver_dq_error_reason_summary_df,
    catalog_name=catalog,
    schema_name=dq_target_schema,
    table_name="_silver_dq_error_reason_summary",
    base_path=dq_base_path,
    mode="overwrite",
)

display(
    silver_dq_error_reason_summary_df
    .orderBy(
        col("record_count").desc(),
        col("table_name"),
        col("dq_error_reason"),
    )
)

# ============================================================
# Silver Transform #4
# Cell 11: Quick invalid-record preview
# ============================================================

for row in (
    silver_dq_record_validity_summary_df
    .filter(col("invalid_rows") > 0)
    .orderBy(col("invalid_rows").desc())
    .collect()
):
    table_name = row["table_name"]
    table_full_name = quote_full_table(catalog, dq_target_schema, table_name)

    print(f"\nInvalid record preview: {table_full_name}")

    display(
        spark.table(table_full_name)
        .filter(col(DQ_FIELD_IS_VALID) == False)
        .limit(20)
    )

### Keep invalid rows in a separate quarantine table or flag them

# ============================================================
# Silver Transform #5: Quarantine Invalid Records
# Cell 1: Config
#
# Input:
#   Canonical silver tables that already contain:
#     - is_valid_record
#     - dq_error_reason
#
# Output:
#   silver_quarantine.<table_name>__invalid_records
#   silver_quarantine._silver_invalid_record_quarantine_summary
#   silver_quarantine._silver_invalid_record_quarantine_index
#
# Important:
#   This does NOT delete invalid records from silver.
#   silver_quarantine contains both DQ-invalid silver rows and earlier dedupe/NULL-PK rejects.
# ============================================================

from datetime import datetime, timezone
import uuid

from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col,
    lit,
    when,
    current_timestamp,
    count as spark_count,
    sum as spark_sum,
    to_json,
    struct,
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)


#catalog = globals().get("catalog", "workspace")

# Usually this should be the same schema where your DQ-tagged silver tables live.
quarantine_source_schema = globals().get(
    "quarantine_source_schema",
    globals().get("dq_target_schema", globals().get("target_schema", "silver"))
)

# Separate quarantine schema.
quarantine_target_schema = globals().get(
    "quarantine_target_schema",
    globals().get("quarantine_schema", "silver_quarantine")
)

# Optional external location.
# Leave as None to use the schema default location.
quarantine_base_path = globals().get(
    "quarantine_base_path",
    globals().get("quarantine_base_path", None)
)

# If True, create empty quarantine tables even when a source table has 0 invalid rows.
# This is helpful because every source table will have a predictable quarantine table.
WRITE_EMPTY_QUARANTINE_TABLES = True

# Optional valid-only views for gold transformations.
CREATE_VALID_ONLY_VIEWS = True
valid_view_schema = globals().get("valid_view_schema", "silver_valid")

DQ_FIELD_IS_VALID = globals().get("DQ_FIELD_IS_VALID", "is_valid_record")
DQ_FIELD_ERROR_REASON = globals().get("DQ_FIELD_ERROR_REASON", "dq_error_reason")

quarantine_run_id = str(uuid.uuid4())
quarantine_run_started_at = datetime.now(timezone.utc).isoformat()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{quarantine_target_schema}`")

if CREATE_VALID_ONLY_VIEWS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{valid_view_schema}`")


# ============================================================
# Silver Transform #5
# Cell 2: Helpers
# ============================================================

def quote_identifier(identifier: str) -> str:
    return "`" + identifier.replace("`", "``") + "`"


def quote_full_table(catalog_name: str, schema_name: str, table_name: str) -> str:
    return (
        f"{quote_identifier(catalog_name)}."
        f"{quote_identifier(schema_name)}."
        f"{quote_identifier(table_name)}"
    )


def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(0).collect()
        return True
    except Exception:
        return False


def resolve_column_name(df: DataFrame, requested_column_name: str):
    lookup = {
        existing_column.lower(): existing_column
        for existing_column in df.columns
    }

    return lookup.get(requested_column_name.lower())


def write_delta_table(
    df: DataFrame,
    catalog_name: str,
    schema_name: str,
    table_name: str,
    base_path: str = None,
    mode: str = "overwrite",
):
    target_table = quote_full_table(catalog_name, schema_name, table_name)

    writer = (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
    )

    if base_path:
        table_path = f"{base_path.rstrip('/')}/{table_name}"
        writer = writer.option("path", table_path)

    writer.saveAsTable(target_table)


def get_quarantine_source_tables():
    """
    Uses expected_mysql_schema as the table list when available.
    This keeps the quarantine process aligned with the current notebook.

    If expected_mysql_schema is not available, falls back to SHOW TABLES.
    """
    if "expected_mysql_schema" in globals():
        return list(expected_mysql_schema.keys())

    tables_df = spark.sql(
        f"SHOW TABLES IN {quote_identifier(catalog)}.{quote_identifier(quarantine_source_schema)}"
    )

    return [
        row["tableName"]
        for row in tables_df.collect()
        if not row["tableName"].startswith("_silver_")
    ]


def get_primary_key_map_for_quarantine():
    """
    Reuses your notebook's existing PK map if available.
    This is only used for the central quarantine index.
    """
    if "primary_key_map" in globals() and isinstance(primary_key_map, dict):
        return primary_key_map

    if "pk_map" in globals():
        if isinstance(pk_map, dict):
            return pk_map

        return {
            table_name: pk_columns
            for table_name, pk_columns in pk_map
        }

    return {}


quarantine_source_tables = get_quarantine_source_tables()
quarantine_primary_key_map = get_primary_key_map_for_quarantine()

# ============================================================
# Silver Transform #5
# Cell 3: Invalid row filter
# ============================================================

def build_invalid_record_filter(df: DataFrame):
    """
    A row is invalid when:
      - is_valid_record = false, OR
      - dq_error_reason is not null

    This makes the filter resilient if one field is present but the other is not.
    """
    is_valid_column = resolve_column_name(df, DQ_FIELD_IS_VALID)
    error_reason_column = resolve_column_name(df, DQ_FIELD_ERROR_REASON)

    invalid_conditions = []

    if is_valid_column is not None:
        invalid_conditions.append(col(quote_identifier(is_valid_column)) == False)

    if error_reason_column is not None:
        invalid_conditions.append(col(quote_identifier(error_reason_column)).isNotNull())

    if not invalid_conditions:
        return None

    return reduce(lambda left, right: left | right, invalid_conditions)


def add_quarantine_metadata(df: DataFrame, source_table_name: str):
    """
    Adds quarantine metadata while preserving the original row columns.
    """
    source_table = quote_full_table(catalog, quarantine_source_schema, source_table_name)
    quarantine_table = quote_full_table(
        catalog,
        quarantine_target_schema,
        f"{source_table_name}__invalid_records"
    )

    return (
        df
        .withColumn("_quarantine_run_id", lit(quarantine_run_id))
        .withColumn("_quarantine_source_catalog", lit(catalog))
        .withColumn("_quarantine_source_schema", lit(quarantine_source_schema))
        .withColumn("_quarantine_source_table", lit(source_table_name))
        .withColumn("_quarantine_source_full_name", lit(source_table))
        .withColumn("_quarantine_table_full_name", lit(quarantine_table))
        .withColumn("_quarantined_at", current_timestamp())
        .withColumn("_quarantine_reason", col(quote_identifier(DQ_FIELD_ERROR_REASON)) if DQ_FIELD_ERROR_REASON in df.columns else lit(None).cast(StringType()))
    )

# ============================================================
# Silver Transform #5
# Cell 4: Write per-table quarantine tables
# ============================================================

quarantine_summary_rows = []

for table_name in quarantine_source_tables:
    source_table = quote_full_table(catalog, quarantine_source_schema, table_name)
    quarantine_table_name = f"{table_name}__invalid_records"
    quarantine_table = quote_full_table(catalog, quarantine_target_schema, quarantine_table_name)

    if not table_exists(source_table):
        quarantine_summary_rows.append({
            "quarantine_run_id": quarantine_run_id,
            "run_started_at": quarantine_run_started_at,
            "source_table": source_table,
            "quarantine_table": quarantine_table,
            "total_rows": None,
            "valid_rows": None,
            "invalid_rows": None,
            "quarantine_rows_written": None,
            "status": "MISSING_SOURCE_TABLE",
            "message": f"Source table does not exist: {source_table}",
        })
        continue

    try:
        source_df = spark.table(source_table)
        invalid_filter = build_invalid_record_filter(source_df)

        if invalid_filter is None:
            quarantine_summary_rows.append({
                "quarantine_run_id": quarantine_run_id,
                "run_started_at": quarantine_run_started_at,
                "source_table": source_table,
                "quarantine_table": quarantine_table,
                "total_rows": source_df.count(),
                "valid_rows": None,
                "invalid_rows": None,
                "quarantine_rows_written": None,
                "status": "MISSING_DQ_FIELDS",
                "message": f"Expected DQ fields were not found: {DQ_FIELD_IS_VALID}, {DQ_FIELD_ERROR_REASON}",
            })
            continue

        summary_stats = (
            source_df
            .agg(
                spark_count(lit(1)).alias("total_rows"),
                spark_sum(when(invalid_filter, lit(1)).otherwise(lit(0))).alias("invalid_rows"),
                spark_sum(when(~invalid_filter, lit(1)).otherwise(lit(0))).alias("valid_rows"),
            )
            .collect()[0]
            .asDict()
        )

        invalid_rows = summary_stats["invalid_rows"] or 0
        invalid_df = source_df.filter(invalid_filter)
        quarantined_df = add_quarantine_metadata(
            df=invalid_df,
            source_table_name=table_name,
        )

        if invalid_rows > 0 or WRITE_EMPTY_QUARANTINE_TABLES:
            write_delta_table(
                df=quarantined_df,
                catalog_name=catalog,
                schema_name=quarantine_target_schema,
                table_name=quarantine_table_name,
                base_path=quarantine_base_path,
                mode="overwrite",
            )
            quarantine_rows_written = invalid_rows
        else:
            quarantine_rows_written = 0

        quarantine_summary_rows.append({
            "quarantine_run_id": quarantine_run_id,
            "run_started_at": quarantine_run_started_at,
            "source_table": source_table,
            "quarantine_table": quarantine_table,
            "total_rows": summary_stats["total_rows"],
            "valid_rows": summary_stats["valid_rows"],
            "invalid_rows": invalid_rows,
            "quarantine_rows_written": quarantine_rows_written,
            "status": "OK",
            "message": "Invalid records quarantined successfully.",
        })

    except Exception as error:
        quarantine_summary_rows.append({
            "quarantine_run_id": quarantine_run_id,
            "run_started_at": quarantine_run_started_at,
            "source_table": source_table,
            "quarantine_table": quarantine_table,
            "total_rows": None,
            "valid_rows": None,
            "invalid_rows": None,
            "quarantine_rows_written": None,
            "status": "FAILED_QUARANTINE_WRITE",
            "message": str(error),
        })


quarantine_summary_schema = StructType([
    StructField("quarantine_run_id", StringType(), True),
    StructField("run_started_at", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("quarantine_table", StringType(), True),
    StructField("total_rows", LongType(), True),
    StructField("valid_rows", LongType(), True),
    StructField("invalid_rows", LongType(), True),
    StructField("quarantine_rows_written", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
])

silver_invalid_record_quarantine_summary_df = spark.createDataFrame(
    quarantine_summary_rows,
    schema=quarantine_summary_schema,
)

write_delta_table(
    df=silver_invalid_record_quarantine_summary_df,
    catalog_name=catalog,
    schema_name=quarantine_target_schema,
    table_name="_silver_invalid_record_quarantine_summary",
    base_path=quarantine_base_path,
    mode="overwrite",
)

display(
    silver_invalid_record_quarantine_summary_df
    .orderBy(
        col("status").desc(),
        col("invalid_rows").desc_nulls_last(),
        col("source_table"),
    )
)


# ============================================================
# Silver Transform #5
# Cell 5: Central quarantine index table
#
# Purpose:
#   One searchable table for all invalid records across all tables.
#
# Notes:
#   - Per-table quarantine tables preserve original column types.
#   - This index stores row payload as JSON for cross-table searching.
# ============================================================

quarantine_index_dfs = []

for table_name in quarantine_source_tables:
    quarantine_table_name = f"{table_name}__invalid_records"
    quarantine_table = quote_full_table(catalog, quarantine_target_schema, quarantine_table_name)

    if not table_exists(quarantine_table):
        continue

    qdf = spark.table(quarantine_table)

    if qdf.count() == 0:
        continue

    pk_columns = quarantine_primary_key_map.get(table_name, [])

    resolved_pk_columns = []

    for pk_column in pk_columns:
        actual_pk_column = resolve_column_name(qdf, pk_column)

        if actual_pk_column is not None:
            resolved_pk_columns.append(actual_pk_column)

    if resolved_pk_columns:
        primary_key_json_col = to_json(
            struct(*[
                col(quote_identifier(pk_column)).cast("string").alias(pk_column)
                for pk_column in resolved_pk_columns
            ])
        )
    else:
        primary_key_json_col = lit(None).cast(StringType())

    # Store the full row as JSON with string-casted values.
    # The original typed values are still preserved in the per-table quarantine table.
    record_payload_json_col = to_json(
        struct(*[
            col(quote_identifier(column_name)).cast("string").alias(column_name)
            for column_name in qdf.columns
        ])
    )

    index_df = (
        qdf
        .select(
            col("_quarantine_run_id").alias("quarantine_run_id"),
            col("_quarantined_at").alias("quarantined_at"),
            lit(catalog).alias("source_catalog"),
            lit(quarantine_source_schema).alias("source_schema"),
            lit(table_name).alias("source_table"),
            lit(quarantine_table).alias("quarantine_table"),
            primary_key_json_col.alias("primary_key_json"),
            col(quote_identifier(DQ_FIELD_ERROR_REASON)).alias("dq_error_reason"),
            record_payload_json_col.alias("record_payload_json"),
        )
    )

    quarantine_index_dfs.append(index_df)


quarantine_index_schema = StructType([
    StructField("quarantine_run_id", StringType(), True),
    StructField("quarantined_at", StringType(), True),
    StructField("source_catalog", StringType(), True),
    StructField("source_schema", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("quarantine_table", StringType(), True),
    StructField("primary_key_json", StringType(), True),
    StructField("dq_error_reason", StringType(), True),
    StructField("record_payload_json", StringType(), True),
])

if quarantine_index_dfs:
    silver_invalid_record_quarantine_index_df = reduce(
        DataFrame.unionByName,
        quarantine_index_dfs,
    )
else:
    silver_invalid_record_quarantine_index_df = spark.createDataFrame(
        [],
        schema=quarantine_index_schema,
    )


write_delta_table(
    df=silver_invalid_record_quarantine_index_df,
    catalog_name=catalog,
    schema_name=quarantine_target_schema,
    table_name="_silver_invalid_record_quarantine_index",
    base_path=quarantine_base_path,
    mode="overwrite",
)

display(
    silver_invalid_record_quarantine_index_df
    .orderBy(
        col("source_table"),
        col("dq_error_reason"),
    )
)

# ============================================================
# Silver Transform #5
# Cell 6: Quarantine error reason breakdown
# ============================================================

if table_exists(
    quote_full_table(
        catalog,
        quarantine_target_schema,
        "_silver_invalid_record_quarantine_index"
    )
):
    quarantine_index_df = spark.table(
        quote_full_table(
            catalog,
            quarantine_target_schema,
            "_silver_invalid_record_quarantine_index"
        )
    )

    quarantine_error_reason_summary_df = (
        quarantine_index_df
        .groupBy("source_table", "dq_error_reason")
        .agg(spark_count(lit(1)).alias("invalid_record_count"))
        .orderBy(
            col("invalid_record_count").desc(),
            col("source_table"),
            col("dq_error_reason"),
        )
    )

    write_delta_table(
        df=quarantine_error_reason_summary_df,
        catalog_name=catalog,
        schema_name=quarantine_target_schema,
        table_name="_silver_quarantine_error_reason_summary",
        base_path=quarantine_base_path,
        mode="overwrite",
    )

    display(quarantine_error_reason_summary_df)

# ============================================================
# Silver Transform #5
# Cell 7: Optional valid-only views
#
# These are views, not copied tables.
# They do not duplicate data.
# ============================================================

if CREATE_VALID_ONLY_VIEWS:
    valid_view_rows = []

    for table_name in quarantine_source_tables:
        source_table = quote_full_table(catalog, quarantine_source_schema, table_name)
        view_name = quote_full_table(catalog, valid_view_schema, table_name)

        if not table_exists(source_table):
            valid_view_rows.append({
                "source_table": source_table,
                "valid_view": view_name,
                "status": "MISSING_SOURCE_TABLE",
                "message": f"Source table does not exist: {source_table}",
            })
            continue

        df = spark.table(source_table)
        is_valid_column = resolve_column_name(df, DQ_FIELD_IS_VALID)

        if is_valid_column is None:
            valid_view_rows.append({
                "source_table": source_table,
                "valid_view": view_name,
                "status": "MISSING_IS_VALID_RECORD",
                "message": f"Missing {DQ_FIELD_IS_VALID}; view was not created.",
            })
            continue

        spark.sql(f"""
            CREATE OR REPLACE VIEW {view_name} AS
            SELECT *
            FROM {source_table}
            WHERE {quote_identifier(is_valid_column)} = true
        """)

        valid_view_rows.append({
            "source_table": source_table,
            "valid_view": view_name,
            "status": "OK",
            "message": "Valid-only view created successfully.",
        })

    valid_view_summary_schema = StructType([
        StructField("source_table", StringType(), True),
        StructField("valid_view", StringType(), True),
        StructField("status", StringType(), True),
        StructField("message", StringType(), True),
    ])

    valid_view_summary_df = spark.createDataFrame(
        valid_view_rows,
        schema=valid_view_summary_schema,
    )

    write_delta_table(
        df=valid_view_summary_df,
        catalog_name=catalog,
        schema_name=quarantine_target_schema,
        table_name="_silver_valid_view_summary",
        base_path=quarantine_base_path,
        mode="overwrite",
    )

    display(valid_view_summary_df.orderBy("status", "source_table"))


# ============================================================
# Silver Transform #5
# Cell 8: Usage examples
# ============================================================

print("Example 1: Read all invalid records from the quarantine index")
print(
    f"SELECT * FROM {quote_full_table(catalog, quarantine_target_schema, '_silver_invalid_record_quarantine_index')}"
)

print("\nExample 2: Read invalid package rows with original package columns preserved")
print(
    f"SELECT * FROM {quote_full_table(catalog, quarantine_target_schema, 'package__invalid_records')}"
)

print("\nExample 3: Read only valid package rows for gold transformations")
print(
    f"SELECT * FROM {quote_full_table(catalog, valid_view_schema, 'package')}"
)

print("\nExample 4: Filter silver directly without using the valid-only view")
print(
    f"SELECT * FROM {quote_full_table(catalog, quarantine_source_schema, 'package')} WHERE {DQ_FIELD_IS_VALID} = true"
)

### Notebook Change Summary
- Changed: moved the canonical `primary_key_map` ahead of dedupe, fixed column-specific OLTP code standardization, corrected DQ domain/range rules, reused `foreign_key_map` for row-level FK checks, tracked cast failures at row level, removed the duplicate invalid preview cell, and replaced drop/rename same-table swaps with stable overwrite helpers.
- Why: these changes fix ordering bugs, prevent one-letter code corruption, align row-level DQ with the MySQL schema, preserve external table locations more safely, and make quarantine behavior explicit.
- Intentionally not changed: catalog/schema/table names, medallion flow, bronze retention, silver/quarantine/valid naming, and the broader notebook structure.
- Assumptions: canonical silver remains the main output schema, DQ-invalid rows stay in silver and are also quarantined, and duplicate/null-PK rejects belong in quarantine instead of canonical silver.
- Manual Databricks checks: confirm same-table overwrite behavior in your workspace, validate the cast-failure field on a few known-bad records, and spot-check valid-only views plus quarantine table locations.


# ============================================================
# Final sanity-check summary
# ============================================================

display(
    silver_dq_record_validity_summary_df
    .select("table_name", "total_rows", "valid_rows", "invalid_rows", "status")
    .orderBy(col("invalid_rows").desc_nulls_last(), col("table_name"))
)

display(
    silver_dq_error_reason_summary_df
    .orderBy(col("record_count").desc(), col("table_name"), col("dq_error_reason"))
    .limit(100)
)

display(
    silver_invalid_record_quarantine_summary_df
    .select("source_table", "total_rows", "invalid_rows", "quarantine_rows_written", "status")
    .orderBy(col("invalid_rows").desc_nulls_last(), col("source_table"))
)

if "silver_dedupe_summary_df" in globals():
    display(
        silver_dedupe_summary_df
        .select(
            "table_name",
            "source_rows",
            "silver_rows",
            "null_pk_rows_quarantined",
            "duplicate_rows_quarantined",
            "status",
        )
        .orderBy(col("table_name"))
    )
